# Precomputed-embedding hybrid-retrieval entity resolution (local 2-GPU version)

Matches every **Source-1** business record to the **Source-2 / Source-3** records of the same
real-world business. The metric is the competition's **macro F0.5 per Source-1 entity**. Outputs are
`matching_results.tsv` and `candidate_pairs.tsv` for the full test set.

This notebook is **self-contained**. Its inputs are:
* the **raw** `data/` TSV files: `entity_id`, `business_name`, `business_address`, `country`
  (no preprocessed columns);
* **precomputed Qwen3-Embedding-0.6B vectors** for 100% of train and test (`embedding_store/`): a
  256-d **name** vector and a 512-d **address** vector per record, int8. No embedding model runs here.

```text
raw TSVs (4 fields) -> one-time lossless TSV->parquet copy
precomputed name/addr embeddings -> one-time re-order into TSV row order (int8 memmaps)
  -> country invariant check on the FULL train ground truth (hard country blocking is valid only if 0 violations)
  -> per country bucket, 100% of train: dense retrieval on w*name + (1-w)*addr (S1->S2, S1->S3, reverse S2/S3->S1)
     + BM25 + exact keys + char TF-IDF   (dense + BM25 on both GPUs)
  -> diagnostic recall (1% of train queries vs the FULL train pool, depth 500): Recall@K, oracle F0.5 -> choose K
  -> candidate union (cap MAX_CANDIDATES) -> pairwise features
  -> XGBoost reranker (GPU) on a 20% entity subset + decision-rule calibration (macro F0.5)
  -> evaluation on the other 80% of train entities (never trained on), per country -> GATE
  -> 100% test: retrieval, union, features, scoring, decision rule -> strict submission validation -> run report
```

## Local setup
Run cells 0a -> 0b -> 0c once, one at a time:
* **0a** installs/verifies the packages in this kernel's environment (the `venv` kernel);
* **0b** checks the GPUs (expects 2 x 24 GB NVIDIA; RTX PRO 4000 Blackwell = compute capability 12.0,
  which needs a CUDA >= 12.8 PyTorch build);
* **0c** downloads the precomputed embeddings from S3 (`aws s3 sync`, ~19 GB; skipped when complete).

Paths: data in `LOCAL_ROOT/data/{train,test}/*.tsv`, embeddings in `LOCAL_ROOT/embedding_store/`,
everything the run writes in `LOCAL_ROOT/pipeline_run/` (`LOCAL_ROOT` is set in cell 0a; override it with
the env var `ER_ROOT`).

**Raw fields only.** The lexical side (BM25, exact keys, rapidfuzz features) uses the lower-cased raw
name and address under the internal names `name_clean` / `addr_clean`. `extracted_city` /
`extracted_state` do not exist in the raw data and are empty; the existing code treats empty as
missing (0.5 in the soft-equality features, city exact keys off; the state is still looked up in the raw
address by `extract_region_fallback`).

## GPUs
| Work | Device |
|---|---|
| dense exact search (fp16 matmul + top-k, pool streamed in shards) | all GPUs: queries split across GPUs, one thread each |
| BM25 (sparse x sparse product + top-k) | all GPUs: partitions alternate between GPUs |
| char TF-IDF vector search | GPU (same path as dense) |
| XGBoost reranker | cuda:0 |
| normalization, TF-IDF transform, rapidfuzz features | CPU processes (`NUM_WORKERS`) |

## Running
Leave `STAGES_TO_RUN = "all"` and run all cells. Every stage is cached under `pipeline_run/cache`
(fingerprinted by its settings), so a re-run skips finished work and a changed setting recomputes only
what depends on it. `MAX_SESSION_HOURS` stops long stages cleanly; run again to continue.

The test phase (`test_inference`, `submit`) starts only when the train evaluation passes its gate
(configured-union recall >= `GATE1_MIN_UNION_RECALL`). Review the recall tables and the per-country
train F0.5 printed by `train_eval` before trusting the submission. France exists only in test (no labels):
its quality cannot be measured here, and an audit file for hand-labelling is written.

`ER_PROFILE=smoke` runs a tiny end-to-end version of every stage on the same inputs, to check the plumbing.


## 0. Local machine setup
Run 0a -> 0b -> 0c once, one cell at a time. If 0a changes torch, **restart the kernel** and continue with 0b.


In [ ]:
# 0a. Install / verify the packages in THIS kernel's Python environment (the venv kernel)
import os
import subprocess
import sys
from importlib import metadata

LOCAL_ROOT = os.environ.get("ER_ROOT", "/home/parth/Desktop/trial work")   # data, embeddings and all outputs
print("python:", sys.executable)


def _pip(*args):
    print("pip install", " ".join(args))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)


# Blackwell (RTX PRO 4000, compute capability 12.0 = sm_120) needs a PyTorch build for CUDA >= 12.8.
# The installed torch is kept when it already has sm_120 kernels; otherwise the CUDA 12.8 build is installed.
_probe = subprocess.run([sys.executable, "-c", "import torch; print('sm_120' in torch.cuda.get_arch_list())"],
                        capture_output=True, text=True)
if _probe.returncode != 0 or _probe.stdout.strip() != "True":
    _pip("--upgrade", "torch", "--index-url", "https://download.pytorch.org/whl/cu128")
    print("torch (re)installed -> restart the kernel before continuing")

_pip("numpy", "pandas", "pyarrow", "scipy", "scikit-learn", "xgboost>=2.0", "rapidfuzz>=3.0", "joblib", "psutil")
# faiss is optional and not installed: dense search uses exact fp16 torch matmul on the GPUs instead.

for _d in ["torch", "numpy", "pandas", "pyarrow", "scipy", "scikit-learn", "xgboost", "rapidfuzz", "joblib", "psutil"]:
    print(f"  {_d:14s} {metadata.version(_d)}")


In [ ]:
# 0b. GPU check: expects 2 x 24 GB NVIDIA GPUs
import subprocess

import torch

EXPECTED_GPUS = 2
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,compute_cap,driver_version",
                      "--format=csv"], capture_output=True, text=True).stdout)
print(f"torch {torch.__version__}  CUDA build {torch.version.cuda}  compiled archs {torch.cuda.get_arch_list()}")
assert torch.cuda.is_available(), "torch sees no GPU: check the NVIDIA driver and that torch is a CUDA build"
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    arch = f"sm_{p.major}{p.minor}"
    assert arch in torch.cuda.get_arch_list(), f"this torch build has no kernels for {arch} ({p.name}): rerun 0a"
    x = torch.randn(2048, 2048, device=f"cuda:{i}", dtype=torch.float16)
    ok = bool(torch.isfinite((x @ x).float()).all())
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}  {p.name}  cc {p.major}.{p.minor}  {total / 1e9:.1f} GB (free {free / 1e9:.1f})  "
          f"fp16 matmul {'OK' if ok else 'FAILED'}")
    assert ok, f"fp16 matmul failed on cuda:{i}"
    del x
torch.cuda.empty_cache()
if n_gpu >= 2:   # direct GPU->GPU copies: broken peer-to-peer (e.g. IOMMU on) silently returns zeros
    _x = torch.arange(1, 9, dtype=torch.float32, device="cuda:1")
    _p2p_ok = bool((_x.to("cuda:0").cpu() == _x.cpu()).all())
    print(f"direct GPU1->GPU0 copy: {'OK' if _p2p_ok else 'BROKEN (returns wrong data)'} -- the notebook always copies "
          "between GPUs through host memory, so results are correct either way")
    del _x
if n_gpu != EXPECTED_GPUS:
    print(f"WARNING: expected {EXPECTED_GPUS} GPUs but torch sees {n_gpu} (check CUDA_VISIBLE_DEVICES)")
else:
    print(f"OK: {n_gpu} GPUs -- dense search and BM25 are split across all of them")


In [ ]:
# 0c. Precomputed Qwen3-Embedding-0.6B vectors (name 256-d + address 512-d, int8) for 100% of train and test.
# Layout: embedding_store/{train,test}/{s1,s2,s3}/{name,addr}/{meta.json, rows.npy, emb_00000.npy ...}
import json
import math
import os
import subprocess

EMB_S3_URI = "s3://amazon-ml-summer-challenge1/embedding_store/"
EMB_LOCAL = os.path.join(LOCAL_ROOT, "embedding_store")


def embeddings_complete(root):
    """All 12 (split, source, view) folders present with meta.json, rows.npy and ceil(n / chunk_rows) chunks."""
    missing = []
    for split in ("train", "test"):
        for s in ("s1", "s2", "s3"):
            for view in ("name", "addr"):
                d = os.path.join(root, split, s, view)
                mp = os.path.join(d, "meta.json")
                if not (os.path.exists(mp) and os.path.exists(os.path.join(d, "rows.npy"))):
                    missing.append(f"{split}/{s}/{view}: meta/rows")
                    continue
                with open(mp) as f:
                    meta = json.load(f)
                need = math.ceil(meta["n"] / meta["chunk_rows"])
                have = sum(1 for i in range(need) if os.path.exists(os.path.join(d, f"emb_{i:05d}.npy")))
                if have != need:
                    missing.append(f"{split}/{s}/{view}: {have}/{need} chunks")
    return missing


_missing = embeddings_complete(EMB_LOCAL)
if _missing:
    print(f"syncing {EMB_S3_URI} -> {EMB_LOCAL} ({len(_missing)} folder(s) incomplete) ...")
    subprocess.run(["aws", "s3", "sync", EMB_S3_URI, EMB_LOCAL, "--only-show-errors"], check=True)
    _missing = embeddings_complete(EMB_LOCAL)
assert not _missing, f"embeddings still incomplete after sync: {_missing}"
_n_files = sum(len(fs) for _, _, fs in os.walk(EMB_LOCAL))
_gb = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(EMB_LOCAL) for f in fs) / 1e9
print(f"embeddings complete: {EMB_LOCAL}  ({_n_files:,} files, {_gb:.2f} GB)")


## 1. Configuration
Every tunable lives here. Env overrides: `ER_PROFILE`, `ER_STAGES`, `ER_ROOT`, and `ER_SET="KEY=json;KEY2=json"`.


In [ ]:
import os

LOCAL_ROOT = os.environ.get("ER_ROOT", "/home/parth/Desktop/trial work")   # same as cell 0a
PROFILE = os.environ.get("ER_PROFILE", "full")        # "full" = 100% train + 100% test, "smoke" = tiny end-to-end check
STAGES_TO_RUN = os.environ.get("ER_STAGES", "all")    # "all" or comma list of stage names (see ALL_STAGES in §3)

CFG = dict(
    # ---- data volume (deterministic hash sampling; 1.0 = everything) ----
    DATA_FRACTION=1.0,              # share of TRAIN Source-1 entities (all their true matches always kept)
    TEST_FRACTION=1.0,              # 1.0 = full test = a submittable file; <1 = dry run, never named a submission
    POOL_DISTRACTOR_FRACTION=None,  # share of the other train S2/S3 records kept (None = DATA_FRACTION -> 1.0 = all)
    HARD_NEIGHBOR_K=0,              # only useful when DATA_FRACTION < 1 (the full pool already holds every neighbour)
    HARD_NEIGHBOR_POOL_FRACTION=1.0,
    RANDOM_SEED=42,

    # ---- phases and gates ----
    ENABLE_SCALE_PHASE=True,        # test inference + submission (still refused when the train-evaluation gate fails)
    GATE1_MIN_UNION_RECALL=0.99,    # configured-union recall (full train pool) needed before the test phase may start
    COUNTRY_INVARIANT_MAX_VIOLATIONS=0,  # cross-country positive pairs tolerated before the run STOPS
    FORCE_PAST_GATE=False,

    # ---- precomputed embeddings (Qwen3-Embedding-0.6B, computed elsewhere; no model runs here) ----
    EMB_DIR=os.path.join(LOCAL_ROOT, "embedding_store"),   # {split}/{s1,s2,s3}/{name,addr}/ (cell 0c)
    EMB_NAME_WEIGHT="auto",         # dense score = w * cos(name) + (1 - w) * cos(address); "auto" = best w of the sweep
    EMB_WEIGHT_SWEEP=[0.0, 0.25, 0.4, 0.5, 0.6, 0.75, 1.0],   # dense Recall@K per w on the diagnostic queries
    EMBED_CHUNK_ROWS=500_000,       # rows per resumable chunk of the char TF-IDF vector cache

    # ---- hardware ----
    NUM_GPUS="auto",                # "auto" = every visible GPU (2 here)
    NUM_WORKERS=20,                 # CPU processes (normalization, features, tf-idf); 24 cores, 4 left for the parent + OS
    RAM_BUDGET_GB=28,               # feature workers are capped so parent RSS + workers stay under this
    MAX_SESSION_HOURS=4.0,          # long stages stop cleanly after this; run again to continue from the cache
    DISK_MARGIN_GB=1.0,
    BM25_USE_GPU=True,              # sparse x sparse BM25 products + top-k on the GPUs (False = CPU scipy path)

    # ---- retrieval ----
    DENSE_TOP_K=50,                 # per pool source (S2 and S3 each), forward S1->pool
    REV_TOP_K=3,                    # reverse pool->S1 (diag table: 1-3 keep more true matches in the cap than 10)
    REV_DIAG_TOP_K=50,              # reverse depth on the diagnostic context, to check that REV_TOP_K suffices
    ENABLE_GLOBAL_FALLBACK=False,   # cross-country dense pass. OFF: 0 of 7,638,365 train positive pairs cross a country
    GLOBAL_FALLBACK_TOP_K=5,
    BM25_TOP_K=50,                  # per pool source
    BM25_K1=1.2, BM25_B=0.75,
    BM25_FIELD_WEIGHTS={"n": 2.0, "a": 1.0, "c": 0.5, "s": 0.25},  # name / address / city / state
    BM25_MAX_DF_FRAC=0.002,         # tokens in more than this share of a partition's docs are pruned
    BM25_MIN_DF_CAP=50,
    BM25_QUERY_CHUNK=2000,          # queries per sparse product (CPU path); the GPU path sizes its own blocks
    ENABLE_TFIDF_RETRIEVER=False,   # char TF-IDF+SVD retriever: OFF -- full-pool leave-one-out drop was negative (-0.0017)
    TFIDF_TOP_K=20, TFIDF_SVD_DIM=128, TFIDF_MAX_FEATURES=60_000, TFIDF_NGRAM_RANGE=(2, 4),
    TFIDF_FIT_SAMPLE=400_000,
    EXACT_TOP_K=5, EXACT_MAX_BLOCK=200, EXACT_MAX_PAIRS=20_000,
    MAX_CANDIDATES=120,             # final per-S1 cap after the union (diag: 100 -> oracle F0.5 0.989, 200 -> 0.992)
    RECALL_KS=[25, 50, 100, 200, 500],
    DIAG_QUERY_FRACTION=0.01,       # train S1 share (~22k) searched at diagnostic depth against the FULL train pool
    DIAG_TOP_K=500,                 # per-source retrieval depth on the diagnostic queries (recall curves only)
    SEARCH_SHARD_ROWS=1_000_000,    # pool rows per GPU search shard (bounds host RAM of the gathered fp16 shard)
    QUERY_BLOCK_ROWS=50_000,        # Source-1 rows per union/feature/predict block (bounds per-block RAM)
    PAIR_BATCH=200_000,

    # ---- model ----
    GBDT_TRAIN_ENTITY_FRACTION=0.20,  # train S1 entities the reranker is fit + calibrated on; the rest is evaluation
    GBDT_GRID_SELECT_FRACTION=0.25,   # share of the training entities used to pick GBDT_PARAM_GRID (faster)
    NESTED_HOLDOUT_FRAC=0.12, N_FOLDS=5, GBDT_TRAIN_NEG_FRAC=1.0,
    GBDT_PARAM_GRID=[dict(max_depth=6, learning_rate=0.10, n_estimators=300),
                     dict(max_depth=8, learning_rate=0.07, n_estimators=400)],
    RULE_SIMPLICITY_TOL=0.001,      # a more complex decision rule must beat the simplest one by this much (CV macro F0.5)

    # ---- manual audit of test countries unseen in training (France) ----
    AUDIT_COUNTRIES=None,           # None = test countries absent from train; [] = off
    AUDIT_N_ENTITIES=200, AUDIT_TOP_N=5,
    AUDIT_LABELS_PATH=None,         # path/glob of audit TSVs with the is_match column filled in (scored in the report)

    # ---- paths (None = derived from WORK_DIR) ----
    DATA_DIR=os.path.join(LOCAL_ROOT, "data"),   # raw TSVs: {train,test}/{split}_source{1,2,3}.tsv
    WORK_DIR=os.path.join(LOCAL_ROOT, "pipeline_run"),
    INPUT_CACHE_DIR=os.path.join(LOCAL_ROOT, "pipeline_run", "aligned_inputs"),   # parquet copies + aligned embeddings (all profiles)
    CACHE_DIR=None, SCRATCH_DIR=None, OUTPUT_DIR=None,
    RUN_SELF_TESTS=True,
)

PROFILES = {
    "full": {},
    "smoke": dict(
        DATA_FRACTION=0.002, TEST_FRACTION=0.002, DIAG_QUERY_FRACTION=0.5, GBDT_TRAIN_ENTITY_FRACTION=0.5,
        TFIDF_FIT_SAMPLE=30_000, TFIDF_SVD_DIM=48, TFIDF_MAX_FEATURES=20_000, EMBED_CHUNK_ROWS=3000,
        DIAG_TOP_K=200, REV_DIAG_TOP_K=20, SEARCH_SHARD_ROWS=2500, QUERY_BLOCK_ROWS=600, PAIR_BATCH=5000,
        N_FOLDS=3, GBDT_PARAM_GRID=[dict(max_depth=4, learning_rate=0.2, n_estimators=60)],
        GATE1_MIN_UNION_RECALL=0.0, AUDIT_N_ENTITIES=20, WORK_DIR=os.path.join(LOCAL_ROOT, "pipeline_run_smoke"),
    ),
}
CFG.update(PROFILES[PROFILE])
for _kv in filter(None, os.environ.get("ER_SET", "").split(";")):
    _k, _v = _kv.split("=", 1)
    assert _k in CFG, f"ER_SET: unknown config key {_k}"
    import json as _json
    CFG[_k] = _json.loads(_v)
if CFG["POOL_DISTRACTOR_FRACTION"] is None:
    CFG["POOL_DISTRACTOR_FRACTION"] = CFG["DATA_FRACTION"]


## 2. Setup: imports, dependency check, hardware, paths
Packages are installed by cell 0a; this cell only verifies them and stops with a clear message if one is
missing or too old.


In [ ]:
import csv
import gc
import glob
import functools
import hashlib
import importlib
import itertools
import json
import math
import platform
import queue
import re
import shutil
import subprocess
import sys
import tempfile
import threading
import time
import unicodedata
import warnings
from collections import defaultdict
from importlib import metadata as _md
from types import SimpleNamespace

warnings.filterwarnings("ignore")
cfg = SimpleNamespace(**CFG)
T0 = time.time()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")


def _vt(v):
    return tuple(int(x) for x in re.findall(r"\d+", str(v))[:3])


def _dist_version(dist):
    try:
        return _md.version(dist)
    except _md.PackageNotFoundError:
        return None


_REQUIRED = [("numpy", None), ("pandas", None), ("pyarrow", None), ("scipy", None), ("scikit-learn", None),
             ("xgboost", "2.0.0"), ("rapidfuzz", "3.0.0"), ("joblib", None), ("torch", None)]
_missing = [(d, m, _dist_version(d)) for d, m in _REQUIRED
            if _dist_version(d) is None or (m and _vt(_dist_version(d)) < _vt(m))]
if _missing:
    raise RuntimeError("Missing/outdated packages: "
                       + ", ".join(f"{d} (need >={m}, have {v})" for d, m, v in _missing)
                       + "\nFix: run cell 0a (it installs them into this kernel's environment), then restart the kernel.")

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import scipy.sparse as sp
import joblib
import torch
from rapidfuzz import fuzz as rf_fuzz

try:  # pandas>=3 infers Arrow-backed "str" columns; the copied feature code expects object arrays
    pd.set_option("future.infer_string", False)
except Exception:
    pass

TORCH_AVAILABLE = True
N_GPU_AVAIL = torch.cuda.device_count()
N_GPUS = N_GPU_AVAIL if cfg.NUM_GPUS == "auto" else min(int(cfg.NUM_GPUS), N_GPU_AVAIL)
DEVICE = "cuda" if N_GPUS > 0 else "cpu"
DEVICES = [f"cuda:{i}" for i in range(N_GPUS)] or ["cpu"]   # dense search / BM25 are split across these
GPU_INFO = []
for _i in range(N_GPUS):
    _p = torch.cuda.get_device_properties(_i)
    GPU_INFO.append(dict(idx=_i, name=_p.name, cc=f"{_p.major}.{_p.minor}", mem_gb=round(_p.total_memory / 1e9, 1)))

try:   # optional; only used for CPU-only runs. On GPUs, exact fp16 torch search runs on every device.
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False

DATA_DIR = cfg.DATA_DIR
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"DATA_DIR {DATA_DIR} not found (set DATA_DIR / LOCAL_ROOT in the config cell)")
WORK_DIR = cfg.WORK_DIR
CACHE_DIR = cfg.CACHE_DIR or os.path.join(WORK_DIR, "cache")
SCRATCH_DIR = cfg.SCRATCH_DIR or os.path.join(WORK_DIR, "scratch")
OUTPUT_DIR = cfg.OUTPUT_DIR or os.path.join(WORK_DIR, "output")
CACHE_SEARCH_PATHS = []   # extra read-only cache roots (none locally: everything is in CACHE_DIR)
for _d in (CACHE_DIR, SCRATCH_DIR, OUTPUT_DIR):
    os.makedirs(_d, exist_ok=True)


def _ram_gb():
    try:
        import psutil
        vm = psutil.virtual_memory()
        return round(vm.total / 1e9, 1), round(vm.available / 1e9, 1)
    except ImportError:
        pass
    try:
        with open("/proc/meminfo") as f:
            info = {l.split(":")[0]: int(l.split()[1]) for l in f}
        return round(info["MemTotal"] / 1e6, 1), round(info.get("MemAvailable", 0) / 1e6, 1)
    except OSError:
        return None, None


def free_gb(path):
    return round(shutil.disk_usage(path).free / 1e9, 2)


print(f"profile={PROFILE}  stages={STAGES_TO_RUN}  python={platform.python_version()}  platform={platform.platform()}")
print(f"torch={torch.__version__} (CUDA {torch.version.cuda})  xgboost={_dist_version('xgboost')}  "
      f"rapidfuzz={_dist_version('rapidfuzz')}  faiss={FAISS_AVAILABLE}")
print(f"GPUs used={N_GPUS} {GPU_INFO}  devices={DEVICES}")
print(f"RAM (total, available) GB={_ram_gb()}  CPUs={os.cpu_count()}  NUM_WORKERS={cfg.NUM_WORKERS}")
print(f"DATA_DIR={DATA_DIR}\nEMB_DIR={cfg.EMB_DIR}\nCACHE_DIR={CACHE_DIR} (free {free_gb(CACHE_DIR)} GB)\n"
      f"SCRATCH_DIR={SCRATCH_DIR}\nOUTPUT_DIR={OUTPUT_DIR}")


## 3. Checkpointing, logging and timing helpers
* Every expensive artifact is written atomically (tmp file, then `os.replace`) with a `.done` marker
  holding its row count and config fingerprint, so a killed run never leaves a half-written file
  that looks complete.
* Artifacts live in `CACHE_DIR`; a re-run finds them there and skips the work.
* Each stage appends wall time, peak RAM and peak GPU memory to `run_log.jsonl`.


In [ ]:
CODE_VERSION = "pe1"   # bump when a stage's logic changes so stale caches are not reused (invalidates samples + frames)
RETR_CODE_VERSION = "r3"   # retrieval / union logic only (retrieval caches, training sets, models)
MODEL_CODE_VERSION = "m3"  # reranker / decision-rule logic only
REPORT_SUFFIX = ""


def fp_of(obj):
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:10]


def cfg_fp(keys, extra=None):
    return fp_of({"v": CODE_VERSION, "cfg": {k: getattr(cfg, k) for k in keys}, "extra": extra})


def cache_path(rel, base=None):
    p = os.path.join(base or CACHE_DIR, rel)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    return p


def locate(rel):
    """First existing copy of a cache artifact: this session's CACHE_DIR, then earlier sessions' outputs."""
    for base in [CACHE_DIR] + CACHE_SEARCH_PATHS:
        p = os.path.join(base, rel)
        if os.path.exists(p):
            return p
    return None


def is_done(rel):
    return locate(rel + ".done") is not None


def read_done(rel):
    p = locate(rel + ".done")
    if p is None:
        return None
    with open(p) as f:
        return json.load(f)


def _atomic_write_bytes(path, write_fn):
    tmp = path + ".tmp"
    write_fn(tmp)
    os.replace(tmp, path)


def save_json(rel, obj, base=None):
    p = cache_path(rel, base)
    def _w(t):
        with open(t, "w") as f:
            json.dump(obj, f, default=str, indent=1)
    _atomic_write_bytes(p, _w)
    return p


def mark_done(rel, **meta):
    save_json(rel + ".done", dict(meta, t=time.time()))


def save_npy(rel, arr, base=None):
    p = cache_path(rel, base)
    def _w(t):
        with open(t, "wb") as f:
            np.save(f, arr)
    _atomic_write_bytes(p, _w)
    return p


PARQUET_STREAM_ROWS = 5_000_000   # numeric frames above this are converted to Arrow and written slice by slice


def save_parquet(rel, df, base=None, row_group_size=None):
    p = cache_path(rel, base)
    numeric = all(t.kind in "biuf" for t in df.dtypes)
    if len(df) <= PARQUET_STREAM_ROWS or not numeric:
        _atomic_write_bytes(p, lambda t: pq.write_table(pa.Table.from_pandas(df, preserve_index=False), t,
                                                         compression="zstd", row_group_size=row_group_size))
        return p
    # same file content (schema, row groups), without an Arrow copy of the whole frame
    rg = row_group_size or PARQUET_STREAM_ROWS
    step = max(rg, PARQUET_STREAM_ROWS // rg * rg)

    def _w(t):
        with pq.ParquetWriter(t, pa.Schema.from_pandas(df.iloc[:0], preserve_index=False), compression="zstd") as w:
            for a in range(0, len(df), step):
                w.write_table(pa.Table.from_pandas(df.iloc[a:a + step], preserve_index=False), row_group_size=rg)
    _atomic_write_bytes(p, _w)
    return p


def arrow_to_frame(table):
    """pyarrow Table -> DataFrame with string columns as plain object arrays (pandas-version independent)."""
    cols = {}
    for name in table.column_names:
        col = table.column(name)
        if pa.types.is_string(col.type) or pa.types.is_large_string(col.type):
            arr = col.to_numpy(zero_copy_only=False)
            cols[name] = np.where(pd.isna(arr), "", arr).astype(object) if col.null_count else arr.astype(object)
        else:
            cols[name] = col.to_numpy(zero_copy_only=False)
    return pd.DataFrame(cols)


def load_parquet(path, columns=None, filters=None):
    return arrow_to_frame(pq.read_table(path, columns=columns, filters=filters))


def dir_size_gb(path):
    total = 0
    for root, _, files in os.walk(path):
        for fn in files:
            try:
                total += os.path.getsize(os.path.join(root, fn))
            except OSError:
                pass
    return round(total / 1e9, 3)


def _fmt_duration(seconds):   # copied from er_crossencoder_v2 §2c
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h{m:02d}m{s:02d}s"
    if m:
        return f"{m}m{s:02d}s"
    return f"{s}s"


def _proc_status_gb(*keys):
    try:
        with open("/proc/self/status") as f:
            st_ = {l.split(":")[0]: int(l.split()[1]) for l in f if l.split(":")[0] in keys}
        return [round(st_[k] / 1e6, 2) for k in keys]
    except (OSError, KeyError, ValueError):
        return [None] * len(keys)


def rss_now_gb():
    """(RSS, anonymous, file-backed) GB. File-backed = memmapped embeddings: page cache, reclaimable."""
    return tuple(_proc_status_gb("VmRSS", "RssAnon", "RssFile"))


def _reset_peak_rss():
    try:
        with open("/proc/self/clear_refs", "w") as f:   # Linux >= 4.0: resets VmHWM to the current RSS
            f.write("5")
    except OSError:
        pass


def peak_rss_gb():
    """RSS high-water mark since the last reset (VmHWM; per stage, see `stage`)."""
    return _proc_status_gb("VmHWM")[0]


def gpu_peaks_gb():
    """Per-GPU peak allocated GB since the last reset."""
    if not (TORCH_AVAILABLE and N_GPUS):
        return None
    return [round(torch.cuda.max_memory_allocated(i) / 1e9, 2) for i in range(N_GPUS)]


def gpu_peak_gb():
    p = gpu_peaks_gb()
    return max(p) if p else None


def _obj_gb(obj):
    if obj is None:
        return None
    if isinstance(obj, pd.DataFrame):   # object columns count their 8-byte pointers only (strings may be shared)
        return obj.memory_usage(deep=False, index=False).sum() / 1e9
    if isinstance(obj, dict):
        return sum(_obj_gb(v) or 0 for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return sum(_obj_gb(v) or 0 for v in obj)
    return getattr(obj, "nbytes", None) and obj.nbytes / 1e9


def mem_mark(label, obj=None, rows=None):
    """Log current RSS (anonymous / file-backed) and the size of `obj` right after a large object is built."""
    rss, anon, fil = rss_now_gb()
    size = _obj_gb(obj)
    if rows is None and obj is not None:
        rows = sum(len(v) for v in obj.values()) if isinstance(obj, dict) else (len(obj) if hasattr(obj, "__len__") else None)
    log(f"   [mem] {label}: " + (f"{rows:,} rows, " if rows is not None else "")
        + (f"{size:.2f} GB, " if size is not None else "") + f"RSS {rss} GB (anon {anon}, file {fil})")


REPORT = {}
RUN_LOG_PATH = os.path.join(OUTPUT_DIR, "run_log.jsonl")
PROGRESS_LOG_PATH = os.path.join(OUTPUT_DIR, "progress.log")
SESSION_ID = f"{int(T0)}-{os.getpid()}"


def log(msg):
    """Print and append a timestamped line to output/progress.log (survives a crash / kernel restart)."""
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')} [{_fmt_duration(time.time() - T0)}] {msg}"
    print(line)
    with open(PROGRESS_LOG_PATH, "a") as f:
        f.write(line + "\n")


F05_PROGRESS_REL = "reports/f05_progress.json"


def track_f05(step, macro_f05, **extra):
    """Record a macro F0.5 checkpoint of the pipeline (retrieval ceiling, CV, holdout, running eval, final)
    in reports/f05_progress.json and the progress log, and print the table so far."""
    p = locate(F05_PROGRESS_REL)
    rows = []
    if p:
        with open(p) as f:
            rows = json.load(f)
    rows = [r for r in rows if r["step"] != step] + [dict(step=step, macro_f05=round(float(macro_f05), 5),
                                                          t=time.strftime("%Y-%m-%d %H:%M:%S"), **extra)]
    save_json(F05_PROGRESS_REL, rows)
    REPORT["f05_progress"] = rows
    log(f"F0.5 checkpoint  {step}: {float(macro_f05):.5f}  {extra if extra else ''}")
    return rows


def show_f05_progress():
    p = locate(F05_PROGRESS_REL)
    if p:
        with open(p) as f:
            rows = json.load(f)
        print("\n==== macro F0.5 at each pipeline step ====")
        print(pd.DataFrame(rows).to_string(index=False))


class stage:
    """Context manager: timing + peak RAM / GPU memory per stage, appended to run_log.jsonl."""
    def __init__(self, name, **info):
        self.name, self.info, self.extra = name, info, {}

    def add(self, **kv):
        self.extra.update(kv)

    _open = []   # enclosing stages: the peak counters are reset per stage, so parents fold in their children's peaks

    @staticmethod
    def _fold():
        rss, gpu = peak_rss_gb(), gpu_peaks_gb()
        for s in stage._open:
            s.rss_peak = max(s.rss_peak or 0, rss or 0)
            s.gpu_peaks = [max(a, b) for a, b in zip(s.gpu_peaks, gpu)] if gpu else s.gpu_peaks

    def __enter__(self):
        stage._fold()
        _reset_peak_rss()
        if TORCH_AVAILABLE and N_GPUS:
            for i in range(N_GPUS):
                torch.cuda.reset_peak_memory_stats(i)
        self.rss_peak, self.gpu_peaks = peak_rss_gb(), [0.0] * N_GPUS
        stage._open.append(self)
        self.t0 = time.time()
        print()
        log(f">> {self.name} {self.info or ''}  RSS={rss_now_gb()[0]}GB")
        return self

    def __exit__(self, exc_type, exc, tb):
        stage._fold()
        stage._open.remove(self)
        rec = dict(stage=self.name, seconds=round(time.time() - self.t0, 2), ok=exc_type is None,
                   peak_rss_gb=self.rss_peak, gpu_peak_gb=max(self.gpu_peaks) if self.gpu_peaks else None,
                   gpu_peaks_gb=self.gpu_peaks, session=SESSION_ID,
                   t_session=round(time.time() - T0, 1), **self.info, **self.extra)
        REPORT.setdefault("stages", []).append(rec)
        with open(RUN_LOG_PATH, "a") as f:
            f.write(json.dumps(rec, default=str) + "\n")
        log(f"<< {self.name} {'done' if exc_type is None else 'FAILED: ' + repr(exc)} "
            f"in {_fmt_duration(rec['seconds'])}  peak_rss={rec['peak_rss_gb']}GB (this stage)  "
            f"gpu_peaks={rec['gpu_peaks_gb']}GB  RSS now={rss_now_gb()[0]}GB")
        return False


def hours_left():
    return cfg.MAX_SESSION_HOURS - (time.time() - T0) / 3600


ALL_STAGES = ["selftest", "sample", "embed_check", "weight_sweep", "diag_recall", "train_model", "train_eval",
              "test_inference", "submit", "report"]
SCALE_STAGES = ("test_inference", "submit")   # gated by ENABLE_SCALE_PHASE and the train-evaluation gate
SELECTED = set(ALL_STAGES) if STAGES_TO_RUN.strip() == "all" else {s.strip() for s in STAGES_TO_RUN.split(",") if s.strip()}
assert SELECTED <= set(ALL_STAGES), f"unknown stages: {SELECTED - set(ALL_STAGES)}"
if not cfg.RUN_SELF_TESTS:
    SELECTED.discard("selftest")


def want(name):
    return name in SELECTED

## 2b. Inputs: raw TSV -> parquet, precomputed embeddings -> TSV row order
**Raw data.** The pipeline reads its sources with pyarrow and filters rows **before** building pandas
frames, which needs parquet. Each raw `{split}/{split}_source{s}.tsv` is copied once, field for field
(`entity_id`, `business_name`, `business_address`, `country`, every value a string), to
`INPUT_CACHE_DIR/data_parquet/{split}_source{s}_final.parquet`. Nothing is derived or cleaned. The row count is
checked against the TSV line count, and the copy is rebuilt when a TSV changes (size/mtime fingerprint).

**Embeddings.** In `embedding_store/{split}/{s}/{view}/`, the int8 chunks `emb_*.npy` concatenate to an
`(n, dim)` matrix whose row `i` is the TSV row `rows.npy[i]` (a permutation). `align_embeddings` writes each
(split, source, view) once into `INPUT_CACHE_DIR/emb_aligned/{split}_s{s}_{view}.npy` in **TSV row order**, so a
lookup is a plain memmap index. Checks, each of which stops the run:
* `meta.json` agrees with the files: `n` = TSV rows, `dim`, int8 dtype, chunk shapes;
* `rows.npy` is a permutation of `0..n-1`;
* alignment: records with the same `business_name` must have (near-)identical name vectors
  (cos > 0.999 for >= 99% of sampled duplicate pairs); a wrong mapping gives ~0%.


In [ ]:
import pyarrow.csv as pacsv

RAW_FIELDS = ["entity_id", "business_name", "business_address", "country"]
SRC_DIR = os.path.join(cfg.INPUT_CACHE_DIR, "data_parquet")
EMB_ALIGNED_DIR = os.path.join(cfg.INPUT_CACHE_DIR, "emb_aligned")
EMB_VIEWS = ("name", "addr")


def _raw_tsv(split, s):
    for p in (os.path.join(DATA_DIR, split, f"{split}_source{s}.tsv"), os.path.join(DATA_DIR, f"{split}_source{s}.tsv")):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{split}_source{s}.tsv not found under {DATA_DIR}")


def _gt_tsv():
    for p in (os.path.join(DATA_DIR, "train", "train_ground_truth.tsv"), os.path.join(DATA_DIR, "train_ground_truth.tsv")):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"train_ground_truth.tsv not found under {DATA_DIR}")


def _count_lines(path):
    n = 0
    with open(path, "rb") as f:
        for buf in iter(lambda: f.read(1 << 24), b""):
            n += buf.count(b"\n")
    return n


def _tsv_to_parquet(tsv, out):
    """Every field read as a plain string, no quoting (names contain quote characters), '' kept as ''."""
    t = pacsv.read_csv(
        tsv,
        read_options=pacsv.ReadOptions(block_size=64 << 20),
        parse_options=pacsv.ParseOptions(delimiter="\t", quote_char=False, newlines_in_values=False),
        convert_options=pacsv.ConvertOptions(column_types={c: pa.string() for c in RAW_FIELDS},
                                             strings_can_be_null=False, include_columns=RAW_FIELDS))
    n_lines = _count_lines(tsv)
    if t.num_rows != n_lines - 1:
        raise RuntimeError(f"{tsv}: parsed {t.num_rows:,} rows but the file has {n_lines - 1:,} data lines")
    tmp = out + ".tmp"
    pq.write_table(t, tmp, compression="zstd", row_group_size=500_000)
    os.replace(tmp, out)
    return t.num_rows


def _file_fp(path):
    st = os.stat(path)
    return f"{os.path.abspath(path)}|{st.st_size}|{int(st.st_mtime)}"


def prepare_raw_inputs():
    os.makedirs(SRC_DIR, exist_ok=True)
    for sp_ in ("train", "test"):
        for s in (1, 2, 3):
            tsv = _raw_tsv(sp_, s)
            out = os.path.join(SRC_DIR, f"{sp_}_source{s}_final.parquet")
            marker = out + ".done"
            fp = _file_fp(tsv)
            if os.path.exists(out) and os.path.exists(marker) and open(marker).read() == fp:
                print(f"  {os.path.basename(out)}: up to date ({pq.ParquetFile(out).metadata.num_rows:,} rows)")
                continue
            t0 = time.time()
            n = _tsv_to_parquet(tsv, out)
            with open(marker, "w") as f:
                f.write(fp)
            print(f"  {os.path.basename(tsv)} -> {os.path.basename(out)}: {n:,} rows in {time.time() - t0:.1f}s")


def _emb_src_dir(split, s, view):
    return os.path.join(cfg.EMB_DIR, split, f"s{s}", view)


def aligned_path(split, s, view):
    return os.path.join(EMB_ALIGNED_DIR, f"{split}_s{s}_{view}.npy")


def _align_one(split, s, view, n_rows):
    """Write the (split, source, view) vectors as one int8 (n, dim) memmap in TSV row order."""
    src = _emb_src_dir(split, s, view)
    with open(os.path.join(src, "meta.json")) as f:
        meta = json.load(f)
    out = aligned_path(split, s, view)
    fp = dict(meta=meta, rows_npy=_file_fp(os.path.join(src, "rows.npy")), n_rows=n_rows)
    marker = out + ".done"
    if os.path.exists(out) and os.path.exists(marker):
        with open(marker) as f:
            if json.load(f) == json.loads(json.dumps(fp)):
                return meta, False
    n, dim, R = int(meta["n"]), int(meta["dim"]), int(meta["chunk_rows"])
    if meta.get("store_dtype") != "int8":
        raise RuntimeError(f"{src}: store_dtype {meta.get('store_dtype')} (expected int8)")
    if meta.get("file_split") not in (None, split) or meta.get("source") not in (None, f"s{s}") or meta.get("view") not in (None, view):
        raise RuntimeError(f"{src}: meta.json describes {meta.get('file_split')}/{meta.get('source')}/{meta.get('view')}")
    if n != n_rows:
        raise RuntimeError(f"{src}: meta n={n:,} but {split}_source{s} has {n_rows:,} rows")
    rows = np.load(os.path.join(src, "rows.npy"))
    if rows.shape != (n,) or rows.min() != 0 or rows.max() != n - 1 or np.bincount(rows, minlength=n).max() != 1:
        raise RuntimeError(f"{src}/rows.npy is not a permutation of 0..{n - 1}")
    tmp = out + ".tmp.npy"
    mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.int8, shape=(n, dim))
    n_chunks = math.ceil(n / R)
    for c in range(n_chunks):
        E = np.load(os.path.join(src, f"emb_{c:05d}.npy"))
        lo, hi = c * R, min(n, (c + 1) * R)
        if E.shape != (hi - lo, dim) or E.dtype != np.int8:
            raise RuntimeError(f"{src}/emb_{c:05d}.npy has shape {E.shape} {E.dtype}, expected {(hi - lo, dim)} int8")
        mm[rows[lo:hi]] = E        # embedding row i belongs to TSV row rows[i]
    mm.flush()
    del mm
    os.replace(tmp, out)
    with open(marker, "w") as f:
        json.dump(fp, f)
    return meta, True


def _check_alignment(split, s, sample=4000):
    """Records sharing a business_name must share the name vector (int8 rounding aside)."""
    names = pq.read_table(SRC_FILE[(split, s)], columns=["business_name"]).column(0).to_numpy(zero_copy_only=False)
    codes, uniq = pd.factorize(names)
    cnt = np.bincount(codes)
    dup_codes = np.flatnonzero(cnt > 1)
    if not len(dup_codes):
        return None
    rng = np.random.default_rng(0)
    pick = rng.choice(dup_codes, size=min(sample, len(dup_codes)), replace=False)
    order = np.argsort(codes, kind="stable")
    starts = np.r_[0, np.cumsum(cnt)[:-1]]
    a, b = order[starts[pick]], order[starts[pick] + 1]
    E = np.load(aligned_path(split, s, "name"), mmap_mode="r")
    va, vb = np.asarray(E[np.sort(a)], np.float32), np.asarray(E[np.sort(b)], np.float32)
    va, vb = va[np.argsort(np.argsort(a))], vb[np.argsort(np.argsort(b))]
    cos = (va * vb).sum(1) / np.maximum(np.linalg.norm(va, axis=1) * np.linalg.norm(vb, axis=1), 1e-9)
    frac = float((cos > 0.999).mean())
    if frac < 0.99:
        raise RuntimeError(f"embedding alignment check FAILED for {split}/S{s}: only {frac:.3f} of duplicate-name pairs "
                           "have identical name vectors -- rows.npy mapping or files are wrong")
    return round(frac, 4)


def align_embeddings():
    os.makedirs(EMB_ALIGNED_DIR, exist_ok=True)
    info = {}
    for sp_ in ("train", "test"):
        for s in (1, 2, 3):
            n_rows = pq.ParquetFile(SRC_FILE[(sp_, s)]).metadata.num_rows
            for view in EMB_VIEWS:
                t0 = time.time()
                meta, built = _align_one(sp_, s, view, n_rows)
                info[f"{sp_}_s{s}_{view}"] = dict(n=int(meta["n"]), dim=int(meta["dim"]), model=os.path.basename(meta.get("model", "")))
                if built:
                    print(f"  aligned {sp_}/S{s}/{view}: {meta['n']:,} x {meta['dim']} int8 in {time.time() - t0:.1f}s")
            if not os.path.exists(aligned_path(sp_, s, "name") + ".checked"):
                frac = _check_alignment(sp_, s)
                with open(aligned_path(sp_, s, "name") + ".checked", "w") as f:
                    f.write(str(frac))
                print(f"  alignment check {sp_}/S{s}: duplicate-name pairs with identical name vectors = {frac}")
    dims = {v: {d["dim"] for k, d in info.items() if k.endswith(v)} for v in EMB_VIEWS}
    assert all(len(x) == 1 for x in dims.values()), f"inconsistent embedding dims across files: {dims}"
    return {v: next(iter(x)) for v, x in dims.items()}, info


with stage("inputs"):
    prepare_raw_inputs()
    GT_PATH = _gt_tsv()
    SRC_FILE = {(split, s): os.path.join(SRC_DIR, f"{split}_source{s}_final.parquet")
                for split in ("train", "test") for s in (1, 2, 3)}
    EMB_DIMS, EMB_INFO = align_embeddings()
print(f"SRC_DIR={SRC_DIR}\nGT_PATH={GT_PATH}\nEMB_ALIGNED_DIR={EMB_ALIGNED_DIR}  dims={EMB_DIMS}  "
      f"model={sorted({d['model'] for d in EMB_INFO.values()})}")


## 4. Normalization (copied from `er_crossencoder_v2.ipynb` §4) plus address keys
The raw data has no preprocessed columns. `name_clean` / `addr_clean` are the **lower-cased raw**
`business_name` / `business_address` (computed at read time by `read_source`, §5), and
`extracted_city` / `extracted_state` are empty. The functions below derive the lexical signal from them
(legal-suffix-stripped name core, sorted/acronym/phonetic keys, state found in the address, postcode,
house number). They are copied unchanged, with one exception: the two `frozenset` columns of
`normalize_all` are built per feature chunk (`add_set_columns`) instead of for a whole 10M-row frame, to
bound RAM. Large frames are normalized in `NUM_WORKERS` processes.

New in this notebook: `extract_postcode` (country-aware ZIP/PIN), `primary_house_number`, and
`is_non_latin` (Indic-script names).


In [ ]:
# ------------------------- copied from er_crossencoder_v2 §4 (cell 14) -------------------------
_LEGAL_SUFFIX_MAP = {
    "incorporated": "inc", "inc": "inc", "corporation": "corp", "corp": "corp",
    "company": "co", "co": "co", "limited": "ltd", "ltd": "ltd",
    "llc": "llc", "l.l.c": "llc", "llp": "llp", "l.l.p": "llp", "lp": "lp",
    "private": "pvt", "pvt": "pvt", "plc": "plc", "gmbh": "gmbh",
    "sa": "sa", "sarl": "sarl", "sas": "sas", "bv": "bv", "nv": "nv",
    "pty": "pty", "group": "group", "holdings": "holdings", "trust": "trust",
    "foundation": "fdn", "enterprises": "ent", "industries": "ind",
    "international": "intl", "solutions": "sol", "services": "svc",
}
_LEGAL_SUFFIX_TOKENS = set(_LEGAL_SUFFIX_MAP)


def strip_legal_suffix(name_clean):
    """Return (core_name, canonical_suffix_or_None). Ampersand normalized to 'and'."""
    if not name_clean:
        return "", None
    text = re.sub(r"[^a-z0-9&\s]", " ", name_clean)
    text = text.replace("&", " and ")
    tokens = text.split()
    suffix = None
    # only the single trailing token -- stripping repeatedly would eat real words ("Trust Bank Co")
    if len(tokens) > 1 and tokens[-1] in _LEGAL_SUFFIX_TOKENS:
        suffix = _LEGAL_SUFFIX_MAP[tokens.pop()]
    return " ".join(tokens), suffix


def name_sorted_signature(core_tokens_str):
    """Order-invariant signature: sorted tokens joined. Catches word transpositions."""
    return " ".join(sorted(core_tokens_str.split()))


def name_acronym(core_tokens_str):
    toks = core_tokens_str.split()
    if len(toks) < 2:
        return ""
    return "".join(t[0] for t in toks if t)


_SOUNDEX_CODES = {
    **{c: "1" for c in "bfpv"}, **{c: "2" for c in "cgjkqsxz"},
    **{c: "3" for c in "dt"}, "l": "4", **{c: "5" for c in "mn"}, "r": "6",
}

def soundex(token):
    """Classic Soundex phonetic code (4 chars). Pure-stdlib, no extra dependency."""
    token = re.sub(r"[^a-z]", "", token.lower())
    if not token:
        return ""
    first = token[0]
    out = []
    prev = _SOUNDEX_CODES.get(first, "")
    for c in token[1:]:
        code_c = _SOUNDEX_CODES.get(c, "")
        if code_c and code_c != prev:
            out.append(code_c)
        if c not in "hw":  # h/w don't separate duplicate codes; vowels do
            prev = code_c
    return (first.upper() + "".join(out) + "000")[:4]


def name_phonetic_key(core_tokens_str):
    toks = core_tokens_str.split()
    if not toks:
        return ""
    return soundex(toks[0]) + (soundex(toks[1]) if len(toks) > 1 else "")


def normalize_names(df):
    core_suffix = df["name_clean"].map(strip_legal_suffix)
    df["name_core"] = [c for c, _ in core_suffix]
    df["name_suffix"] = [s or "" for _, s in core_suffix]
    df["name_sorted"] = df["name_core"].map(name_sorted_signature)
    df["name_acronym"] = df["name_core"].map(name_acronym)
    df["name_phonetic"] = df["name_core"].map(name_phonetic_key)
    return df


_COUNTRY_ALIASES = {
    "us": "US", "usa": "US", "u.s.": "US", "u.s.a.": "US", "united states": "US",
    "united states of america": "US",
    "india": "India", "in": "India", "bharat": "India",
    "france": "France", "fr": "France",
}

def canonicalize_country(raw):
    """Soft canonicalization only -- unrecognized strings pass through as-is (title-cased),
    never filtered or mapped to null. Keeps `country` an open set end to end."""
    if raw is None:
        return ""
    key = str(raw).strip().lower()
    if key in _COUNTRY_ALIASES:
        return _COUNTRY_ALIASES[key]
    return str(raw).strip().title() if raw else ""

# ------------------------- copied from er_crossencoder_v2 §4 (cell 15) -------------------------
# Region registry keyed by COUNTRY -> {lowercase name/code/alias: canonical name}. Keyed by country
# because short codes collide across countries ("tn" = Tennessee or Tamil Nadu, "ga" = Georgia or
# Goa). It is an open dict: a country with no entry simply gets no region canonicalization.
_REGION_REGISTRY = {}

def _strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def _add_regions(country, spec):
    """spec: 'Name:code1/code2, Name2:code, ...' -- codes/aliases optional."""
    d = _REGION_REGISTRY.setdefault(country, {})
    for item in spec.split(","):
        name, _, alts = item.strip().partition(":")
        for k in [name] + [a for a in alts.split("/") if a]:
            d[_strip_accents(k).lower().strip()] = name

_add_regions("US", "Alabama:al, Alaska:ak, Arizona:az, Arkansas:ar, California:ca/calif, Colorado:co, "
    "Connecticut:ct, Delaware:de, District of Columbia:dc/washington dc, Florida:fl, Georgia:ga, Hawaii:hi, "
    "Idaho:id, Illinois:il, Indiana:in, Iowa:ia, Kansas:ks, Kentucky:ky, Louisiana:la, Maine:me, "
    "Maryland:md, Massachusetts:ma, Michigan:mi, Minnesota:mn, Mississippi:ms, Missouri:mo, Montana:mt, "
    "Nebraska:ne, Nevada:nv, New Hampshire:nh, New Jersey:nj, New Mexico:nm, New York:ny, "
    "North Carolina:nc, North Dakota:nd, Ohio:oh, Oklahoma:ok, Oregon:or, Pennsylvania:pa, "
    "Rhode Island:ri, South Carolina:sc, South Dakota:sd, Tennessee:tn, Texas:tx, Utah:ut, Vermont:vt, "
    "Virginia:va, Washington:wa, West Virginia:wv, Wisconsin:wi, Wyoming:wy, Puerto Rico:pr")
_add_regions("India", "Andhra Pradesh:ap, Arunachal Pradesh:ar, Assam:as, Bihar:br, Chhattisgarh:cg/ct/chattisgarh, "
    "Goa:ga, Gujarat:gj/gujrat, Haryana:hr, Himachal Pradesh:hp, Jharkhand:jh, Karnataka:ka/karnatka, "
    "Kerala:kl, Madhya Pradesh:mp, Maharashtra:mh/maharastra, Manipur:mn, Meghalaya:ml, Mizoram:mz, "
    "Nagaland:nl, Odisha:od/or/orissa, Punjab:pb, Rajasthan:rj, Sikkim:sk, Tamil Nadu:tn/tamilnadu, "
    "Telangana:tg/ts, Tripura:tr, Uttar Pradesh:up, Uttarakhand:uk/uttaranchal, West Bengal:wb, "
    "Delhi:dl/new delhi, Jammu and Kashmir:jk, Ladakh:la, Chandigarh:ch, Puducherry:py/pondicherry, "
    "Andaman and Nicobar Islands:an, Dadra and Nagar Haveli and Daman and Diu:dd, Lakshadweep:ld")
_add_regions("France", "Ile-de-France:idf/ile de france, Auvergne-Rhone-Alpes:ara/rhone alpes, "
    "Bourgogne-Franche-Comte:bfc/franche comte/bourgogne, Bretagne:bzh/brittany, Centre-Val de Loire:cvl, "
    "Corse:corsica, Grand Est:alsace/lorraine/champagne ardenne, Hauts-de-France:hdf/nord pas de calais/picardie, "
    "Normandie:normandy, Nouvelle-Aquitaine:aquitaine, Occitanie:languedoc roussillon/midi pyrenees, "
    "Pays de la Loire:pdl, Provence-Alpes-Cote d'Azur:paca/provence alpes cote d azur, "
    "Guadeloupe, Martinique, Guyane:french guiana, La Reunion:reunion, Mayotte")

# unambiguous long names/aliases usable for ANY country (e.g. a mislabeled record)
_REGION_LONG_ANY = {k: v for d in _REGION_REGISTRY.values() for k, v in d.items() if len(k) > 3}


def canonical_region(value, country):
    v = _strip_accents(str(value)).strip().lower().rstrip(".")
    if not v:
        return ""
    d = _REGION_REGISTRY.get(country, {})
    if v in d:
        return d[v].lower()
    return _REGION_LONG_ANY.get(v, v).lower()


def extract_region_fallback(addr_clean, country):
    """Country-aware region lookup from the address; used only when extracted_state is blank.
    Short codes (<=3 chars) only count when they are an entire comma-separated segment --
    otherwise they'd false-match ordinary address tokens."""
    if not addr_clean:
        return ""
    d = _REGION_REGISTRY.get(country, {})
    text = re.sub(r"[^a-z\s,]", " ", _strip_accents(addr_clean).lower())
    segments = [seg.split() for seg in text.split(",")]
    for seg in segments:
        if len(seg) == 1 and len(seg[0]) <= 3 and seg[0] in d:
            return d[seg[0]]
    tokens = [t for seg in segments for t in seg]
    n = len(tokens)
    for width in (5, 4, 3, 2, 1):  # longest first, so "west bengal" beats "bengal"
        for start in range(n - width + 1):
            cand = " ".join(tokens[start:start + width])
            if len(cand) > 3 and (cand in d or cand in _REGION_LONG_ANY):
                return d.get(cand) or _REGION_LONG_ANY[cand]
    return ""


_DIGIT_TOKEN_RE = re.compile(r"\d+")

def digit_tokens(addr_clean):
    return frozenset(_DIGIT_TOKEN_RE.findall(addr_clean or ""))


def normalize_all(df):
    """copied; the addr_digits / name_tokens frozenset columns moved to add_set_columns()."""
    df = normalize_names(df)
    df["country_norm"] = df["country"].map(canonicalize_country)
    blank = df["extracted_state"].astype(str).str.strip() == ""
    if blank.any():
        df.loc[blank, "extracted_state"] = [
            extract_region_fallback(a, c) for a, c in zip(df.loc[blank, "addr_clean"], df.loc[blank, "country_norm"])]
    df["state_norm"] = [canonical_region(s, c) for s, c in zip(df["extracted_state"], df["country_norm"])]
    df["city_norm"] = [_strip_accents(str(c)).strip().lower() for c in df["extracted_city"]]
    return df

# ------------------------- new in this notebook -------------------------
_ADDR_WORD_RE = re.compile(r"[a-z]{3,}")


def add_set_columns(df):
    """The frozenset columns the copied _features_chunk expects, built only for the rows a chunk needs."""
    df["addr_digits"] = df["addr_clean"].map(digit_tokens)
    df["name_tokens"] = df["name_clean"].astype(str).map(lambda s: frozenset(s.split()))
    df["addr_words"] = df["addr_clean"].map(lambda s: frozenset(_ADDR_WORD_RE.findall(s or "")))
    return df


_POSTCODE_RE = {
    "India": re.compile(r"(?<!\d)([1-9]\d{2})\s?(\d{3})(?!\d)"),
    "US": re.compile(r"(?<!\d)(\d{5})(?:-\d{4})?(?!\d)"),
    "France": re.compile(r"(?<!\d)(\d{5})(?!\d)"),
}


def extract_postcode(addr_clean, country_norm):
    """Country-aware ZIP/PIN. '' when absent -- a missing postcode never counts as agreement."""
    rx = _POSTCODE_RE.get(country_norm)
    if rx is None or not addr_clean:
        return ""
    ms = list(rx.finditer(addr_clean))
    if not ms:
        return ""
    m = ms[-1]
    if country_norm == "US" and len(ms) == 1 and _DIGIT_TOKEN_RE.search(addr_clean).start() == m.start():
        return ""   # a lone leading 5-digit number is a house number ("12345 main st"), not a ZIP
    return "".join(g for g in m.groups() if g)


def primary_house_number(addr_clean, postcode):
    """First number in the address that is not (part of) the postcode; '' when there is none."""
    for tok in _DIGIT_TOKEN_RE.findall(addr_clean or ""):
        if postcode and (tok == postcode or tok in (postcode[:3], postcode[3:])):
            continue
        return tok
    return ""


def is_non_latin(text):
    return any(ch.isalpha() and ord(ch) > 0x024F for ch in (text or ""))


NORM_KEEP_COLS = ["entity_id", "name_clean", "addr_clean", "name_core", "name_suffix", "name_sorted",
                  "name_acronym", "name_phonetic", "country_norm", "state_norm", "city_norm",
                  "postcode", "house_no", "non_latin"]


def normalize_frame(df, min_parallel_rows=200_000):
    """Raw columns -> the lean normalized frame used by retrieval and features. Large frames are cut into
    row chunks normalized in NUM_WORKERS processes (the functions are pure per row, so the result is identical)."""
    if len(df) >= min_parallel_rows and cfg.NUM_WORKERS > 1:
        step = math.ceil(len(df) / (4 * cfg.NUM_WORKERS))
        parts = joblib.Parallel(n_jobs=cfg.NUM_WORKERS, backend="loky")(
            joblib.delayed(_normalize_frame_serial)(df.iloc[a:a + step].reset_index(drop=True))
            for a in range(0, len(df), step))
        return pd.concat(parts, ignore_index=True)
    return _normalize_frame_serial(df)


def _normalize_frame_serial(df):
    df = normalize_all(df)
    df["postcode"] = [extract_postcode(a, c) for a, c in zip(df["addr_clean"], df["country_norm"])]
    df["house_no"] = [primary_house_number(a, p) for a, p in zip(df["addr_clean"], df["postcode"])]
    df["non_latin"] = np.fromiter((is_non_latin(x) for x in df["business_name"]), dtype=np.int8, count=len(df))
    keep = NORM_KEEP_COLS + [c for c in ("pool_source", "samp_match", "samp_dist", "samp_hard") if c in df.columns]
    return df[keep].reset_index(drop=True)

## 5. Loading and the train frame
* The parquet copies of the raw TSVs (§2b) are read with pyarrow and **filtered before** conversion to
  pandas. Original row order is kept; entity ids are never changed. `read_source` serves the views
  `name_clean` / `addr_clean` (lower-cased raw name / address) and the empty `extracted_city` / `extracted_state`.
* With the default `DATA_FRACTION=1.0` the "train sample" is **all of train**: every S1 entity and every
  S2/S3 record (2.2M S1, 10.3M pool). The sampling machinery is kept for the smoke profile: a keyed hash of
  each entity id (`splitmix64(id ^ seed) < fraction`, reproducible and nested) keeps the sampled S1, **all**
  their true matches, and the same share of the other pool records.
* **Test** defaults to 100% (a valid submission). A smaller `TEST_FRACTION` is a dry run only.
* **Country invariant** (`check_country_invariant`, run with the `sample` stage). Every labelled positive
  pair of the **full** train ground truth is checked for `country(S1) == country(pool)`. Retrieval is
  partitioned by country, which is only valid if this holds. Measured: 0 of 7,638,365 pairs cross a
  country. A violation **stops** the run instead of silently losing matches.


In [ ]:
# SRC_FILE / GT_PATH are set by §2b (parquet copies of the raw TSVs)
RAW_COLS = ["entity_id", "business_name", "business_address", "country", "name_clean", "addr_clean",
            "extracted_city", "extracted_state"]
# Only RAW_FIELDS exist in the data. The other names are views computed at read time from the raw fields:
# name_clean / addr_clean = lower-cased raw name / address; extracted_city / extracted_state = "" (not in the data).
_VIEW_OF = {"name_clean": "business_name", "addr_clean": "business_address",
            "extracted_city": None, "extracted_state": None}
ID_BASE = 10 ** 10
SALT = dict(s1=1, pool=2, hard=3, test_s1=4, test_pool=5, fold=6, fit=7, audit=8, diag=9, gbdt=10, grid=11)


def ids_to_int(ids):
    """'S2-681193310' -> 2*10^10 + 681193310 (int64). Raises on any unexpected id format."""
    a = ids if isinstance(ids, (pa.Array, pa.ChunkedArray)) else pa.array(np.asarray(ids, dtype=object), type=pa.string())
    if len(a) == 0:
        return np.zeros(0, np.int64)
    if not pc.all(pc.match_substring_regex(a, r"^S[123]-[0-9]{1,10}$")).as_py():
        raise ValueError("unexpected entity_id format (expected S<1|2|3>-<digits>)")
    src = pc.cast(pc.utf8_slice_codeunits(a, 1, 2), pa.int64())
    num = pc.cast(pc.utf8_slice_codeunits(a, 3, 20), pa.int64())
    out = pc.add(pc.multiply(src, ID_BASE), num)
    return np.asarray(out.to_numpy() if isinstance(out, pa.Array) else out.to_numpy(), dtype=np.int64)


def read_ids(split, s):
    col = pq.read_table(SRC_FILE[(split, s)], columns=["entity_id"]).column(0)
    return col, ids_to_int(col)


def read_raw_table(path, columns):
    """Read `columns` from a source parquet; the derived views (_VIEW_OF) are built from the raw fields."""
    need = list(dict.fromkeys(_VIEW_OF.get(c) or c for c in columns if _VIEW_OF.get(c, c) is not None))
    t = pq.read_table(path, columns=need or ["entity_id"])
    out = {}
    for c in columns:
        if c not in _VIEW_OF:
            out[c] = t.column(c)
        elif _VIEW_OF[c] is None:
            out[c] = pa.array(np.full(t.num_rows, "", dtype=object), type=pa.string())
        else:
            out[c] = pc.utf8_lower(pc.fill_null(t.column(_VIEW_OF[c]), ""))
    return pa.table(out)


def read_source(split, s, columns=RAW_COLS, mask=None):
    t = read_raw_table(SRC_FILE[(split, s)], columns)
    if mask is not None:
        t = t.filter(pa.array(np.asarray(mask, dtype=bool)))
    return arrow_to_frame(t)


def splitmix64(x):
    with np.errstate(over="ignore"):
        z = x + np.uint64(0x9E3779B97F4A7C15)
        z = (z ^ (z >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)
        z = (z ^ (z >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)
        return z ^ (z >> np.uint64(31))


def hash_u01(ids_int, salt):
    """Deterministic uniform [0,1) per id (independent of machine / Python hash seed)."""
    key = np.uint64((cfg.RANDOM_SEED * 0x100000001B3 + salt * 0x9E3779B1 + 12345) & ((1 << 64) - 1))
    u = splitmix64(np.asarray(ids_int, dtype=np.int64).astype(np.uint64) ^ key)
    return (u >> np.uint64(11)).astype(np.float64) * (1.0 / (1 << 53))


def load_ground_truth():
    gt = pd.read_csv(GT_PATH, sep="\t", dtype=str, keep_default_na=False)
    assert list(gt.columns) == ["source1_entity_id", "matched_entity_ids"], gt.columns
    return gt


def explode_gt(gt):
    rows = gt[gt["matched_entity_ids"] != ""]
    counts = rows["matched_entity_ids"].str.count(",").values + 1
    s1 = np.repeat(rows["source1_entity_id"].values, counts)
    pool = np.array([x.strip() for s in rows["matched_entity_ids"].values for x in s.split(",")], dtype=object)
    return s1, pool


def sample_fps():
    """(FP_SAMPLE, FP_TEST) for the current cfg.RANDOM_SEED (recomputed by the seed sweep)."""
    return (cfg_fp(["DATA_FRACTION", "POOL_DISTRACTOR_FRACTION", "HARD_NEIGHBOR_K", "HARD_NEIGHBOR_POOL_FRACTION",
                    "RANDOM_SEED", "BM25_K1", "BM25_B", "BM25_MAX_DF_FRAC", "BM25_MIN_DF_CAP"]),
            cfg_fp(["TEST_FRACTION", "RANDOM_SEED"]))


FP_SAMPLE, FP_TEST = sample_fps()
SAMPLE_REL = f"samples/{FP_SAMPLE}"


def _canonical_country_column(split, s):
    """(int64 ids, canonical country per row) for one source file; only the unique values are canonicalized."""
    t = pq.read_table(SRC_FILE[(split, s)], columns=["entity_id", "country"])
    codes, uniq = pd.factorize(t.column(1).to_pandas(), use_na_sentinel=True)
    names = np.array([canonicalize_country(x) for x in uniq] + [""], dtype=object)   # code -1 (null) -> ""
    return ids_to_int(t.column(0)), names[codes]


def _lookup_rows(sorted_ids, order, query):
    j = order[np.clip(np.searchsorted(sorted_ids, query), 0, len(sorted_ids) - 1)]
    return j


def check_country_invariant():
    """Hard country blocking is valid only if NO labelled positive pair crosses countries. Checked on the FULL
    train ground truth (every pair, not the sample), with the same canonical country the partitions use.
    Raises (STOP) when the violation count exceeds COUNTRY_INVARIANT_MAX_VIOLATIONS."""
    rel = "reports/country_invariant.json"
    p = locate(rel)
    if p:
        with open(p) as f:
            res = json.load(f)
        print(f"country invariant (cached): {res['positive_pairs_checked']:,} positive pairs checked, "
              f"{res['cross_country_pairs']:,} cross-country")
    else:
        with stage("country_invariant") as st:
            cols, per_country = {}, {}
            for split in ("train", "test"):
                for s in (1, 2, 3):
                    ids, cn = _canonical_country_column(split, s)
                    per_country[f"{split}_S{s}"] = pd.Series(cn).value_counts().to_dict()
                    if split == "train":
                        cols[s] = (ids, cn)
            s1_ids, s1_cn = cols[1]
            pool_ids = np.concatenate([cols[2][0], cols[3][0]])
            pool_cn = np.concatenate([cols[2][1], cols[3][1]])
            a, b = explode_gt(load_ground_truth())
            ai, bi = ids_to_int(a), ids_to_int(b)
            o1, o2 = np.argsort(s1_ids, kind="stable"), np.argsort(pool_ids, kind="stable")
            j1 = _lookup_rows(s1_ids[o1], o1, ai)
            j2 = _lookup_rows(pool_ids[o2], o2, bi)
            if not ((s1_ids[j1] == ai).all() and (pool_ids[j2] == bi).all()):
                raise RuntimeError("ground-truth ids missing from the source files")
            c1, c2 = s1_cn[j1], pool_cn[j2]
            viol = c1 != c2
            combos = pd.DataFrame({"s1": c1, "pool": c2}).value_counts()
            te_s1 = set(per_country["test_S1"])
            te_pool = set(per_country["test_S2"]) | set(per_country["test_S3"])
            tr_all = set(per_country["train_S1"]) | set(per_country["train_S2"]) | set(per_country["train_S3"])
            res = dict(positive_pairs_checked=int(len(a)), cross_country_pairs=int(viol.sum()),
                       cross_country_pct=round(100 * float(viol.mean()) if len(a) else 0.0, 6),
                       pairs_with_empty_country=int(((c1 == "") | (c2 == "")).sum()),
                       pair_country_combos={f"{k[0]}->{k[1]}": int(v) for k, v in combos.items()},
                       violation_examples=[dict(s1=str(x), pool=str(y), s1_country=str(u), pool_country=str(v))
                                           for x, y, u, v in list(zip(a[viol], b[viol], c1[viol], c2[viol]))[:10]],
                       records_per_country=per_country,
                       test_s1_countries_without_test_pool=sorted(te_s1 - te_pool),
                       test_countries_unseen_in_train=sorted(te_s1 - tr_all))
            save_json(rel, res)
            st.add(pairs=res["positive_pairs_checked"], violations=res["cross_country_pairs"])
    REPORT["country_invariant"] = res
    print(f"  positive pairs checked: {res['positive_pairs_checked']:,}   cross-country: {res['cross_country_pairs']:,} "
          f"({res['cross_country_pct']}%)   empty country: {res['pairs_with_empty_country']:,}\n"
          f"  pair country combinations: {res['pair_country_combos']}\n"
          f"  test countries unseen in train: {res['test_countries_unseen_in_train']}   "
          f"test S1 countries with no test-pool record: {res['test_s1_countries_without_test_pool']}")
    if res["cross_country_pairs"] > cfg.COUNTRY_INVARIANT_MAX_VIOLATIONS:
        raise RuntimeError(
            f"STOP: the country invariant is violated ({res['cross_country_pairs']:,} labelled positive pairs cross "
            f"countries, e.g. {res['violation_examples'][:3]}). Hard country blocking would lose these matches. Review "
            "the data before changing the architecture (ENABLE_GLOBAL_FALLBACK is the optional cross-country pass).")
    if res["test_s1_countries_without_test_pool"] and not cfg.ENABLE_GLOBAL_FALLBACK:
        print(f"  NOTE: test S1 countries {res['test_s1_countries_without_test_pool']} have no pool records, so they get "
              "no candidates (correct under the invariant: they cannot have matches)")
    print("  -> country hard partitioning is valid and is kept" if res["cross_country_pairs"] == 0 else
          "  -> violations within the configured tolerance")
    return res


def _light_norm(df):
    df["name_core"] = [strip_legal_suffix(x)[0] for x in df["name_clean"]]
    df["city_norm"] = [_strip_accents(str(c)).strip().lower() for c in df["extracted_city"]]
    df["country_norm"] = df["country"].map(canonicalize_country)
    df["addr_clean"] = ""
    df["state_norm"] = ""
    return df


def mine_hard_neighbours(q, own_matches, s, ids_int, in_m, k):
    """Top-k BM25 (name+city) non-matches per sampled S1 from the (sub-sampled) FULL train pool of source s."""
    mine_mask = hash_u01(ids_int, SALT["hard"]) < cfg.HARD_NEIGHBOR_POOL_FRACTION
    mine_rows = np.flatnonzero(mine_mask)
    d = _light_norm(read_source("train", s, ["entity_id", "name_clean", "extracted_city", "country"], mine_mask))
    hard = np.zeros(len(ids_int), dtype=bool)
    q_country, d_country = q["country_norm"].values, d["country_norm"].values
    for c in sorted(set(q_country)):
        qr = np.flatnonzero(q_country == c)
        dr = np.flatnonzero(d_country == c)
        if len(qr) == 0 or len(dr) == 0:
            continue
        qi, di, _, rk = bm25_search(d, dr, q, qr, k + 12, fields=("n", "c"))
        q_ids = q["entity_id"].values[qr[qi]]
        d_ids = d["entity_id"].values[dr[di]]
        not_own = np.fromiter((did not in own_matches.get(qid, ()) for qid, did in zip(q_ids, d_ids)),
                              dtype=bool, count=len(qi))
        qi, di, rk = qi[not_own], di[not_own], rk[not_own]
        pos = within_group_rank(qi.astype(np.int64), -rk.astype(np.float32))
        sel = pos < k
        hard[mine_rows[dr[di[sel]]]] = True
    return hard & ~in_m


def build_train_sample():
    if is_done(f"{SAMPLE_REL}/sample"):
        print(f"train sample {SAMPLE_REL} already built -- reusing")
        return
    with stage("sample", fraction=cfg.DATA_FRACTION) as st:
        s1_col, s1_int = read_ids("train", 1)
        keep_s1 = hash_u01(s1_int, SALT["s1"]) < cfg.DATA_FRACTION
        gt = load_ground_truth()
        assert len(gt) == len(s1_int) and gt["source1_entity_id"].is_unique, "GT must have one row per train S1"
        all_s1, all_pool = explode_gt(gt)
        assert pd.Index(all_pool).is_unique, "a pool record is matched to more than one S1 -- exclusivity assumption broken"
        kept_ids = set(s1_col.filter(pa.array(keep_s1)).to_pylist())
        gt_s = gt[gt["source1_entity_id"].isin(kept_ids)].reset_index(drop=True)
        gs1, gpool = explode_gt(gt_s)
        own = defaultdict(set)
        for a, b in zip(gs1, gpool):
            own[a].add(b)
        m_int = np.unique(ids_to_int(gpool))
        masks = {}
        for s in (2, 3):
            _, ids_int = read_ids("train", s)
            in_m = np.isin(ids_int, m_int)
            dist = ~in_m & (hash_u01(ids_int, SALT["pool"]) < cfg.POOL_DISTRACTOR_FRACTION)
            masks[s] = dict(ids_int=ids_int, in_m=in_m, dist=dist, hard=np.zeros(len(ids_int), bool))
        assert sum(int(m["in_m"].sum()) for m in masks.values()) == len(m_int), "some GT matches missing from pool files"
        if cfg.HARD_NEIGHBOR_K > 0:
            q = _light_norm(read_source("train", 1, ["entity_id", "name_clean", "extracted_city", "country"], keep_s1))
            for s in (2, 3):
                with stage(f"sample.hard_neighbours.S{s}"):
                    m = masks[s]
                    m["hard"] = mine_hard_neighbours(q, own, s, m["ids_int"], m["in_m"], cfg.HARD_NEIGHBOR_K) & ~m["dist"]
        s1_raw = read_source("train", 1, RAW_COLS, keep_s1)
        parts = []
        for s in (2, 3):
            m = masks[s]
            keep = m["in_m"] | m["dist"] | m["hard"]
            df = read_source("train", s, RAW_COLS, keep)
            df["pool_source"] = f"S{s}"
            df["samp_match"] = m["in_m"][keep].astype(np.int8)
            df["samp_dist"] = m["dist"][keep].astype(np.int8)
            df["samp_hard"] = m["hard"][keep].astype(np.int8)
            parts.append(df)
        pool_raw = pd.concat(parts, ignore_index=True)
        # ---- integrity checks (fail loudly) ----
        assert s1_raw["entity_id"].is_unique and set(s1_raw["entity_id"]) == kept_ids
        assert pool_raw["entity_id"].is_unique
        assert set(gpool) <= set(pool_raw["entity_id"]), "GT closure broken"
        comp = pool_raw.groupby("pool_source")[["samp_match", "samp_dist", "samp_hard"]].sum()
        print(f"sampled S1 entities: {len(s1_raw):,} of {len(s1_int):,} ({len(s1_raw) / len(s1_int):.4%})  "
              f"per country: {s1_raw['country'].value_counts().to_dict()}")
        print(f"pool records: {len(pool_raw):,}  (true matches={len(gpool):,})\n{comp.to_string()}")
        print(f"GT pairs: {len(gpool):,}  singletons among sampled S1: {int((gt_s['matched_entity_ids'] == '').sum()):,}")
        save_parquet(f"{SAMPLE_REL}/train_s1_raw.parquet", s1_raw)
        save_parquet(f"{SAMPLE_REL}/train_pool_raw.parquet", pool_raw)
        save_parquet(f"{SAMPLE_REL}/gt.parquet", gt_s)
        info = dict(n_s1=len(s1_raw), n_pool=len(pool_raw), n_true_pairs=len(gpool),
                    composition=comp.to_dict(), s1_per_country=s1_raw["country"].value_counts().to_dict())
        st.add(**{k: v for k, v in info.items() if k != "composition"})
        save_json(f"reports/sample_{FP_SAMPLE}.json", info)
        mark_done(f"{SAMPLE_REL}/sample", **{k: v for k, v in info.items() if isinstance(v, int)})


def test_masks(s):
    _, ids_int = read_ids("test", s)
    if cfg.TEST_FRACTION >= 1.0:
        return np.ones(len(ids_int), dtype=bool)
    return hash_u01(ids_int, SALT["test_s1" if s == 1 else "test_pool"]) < cfg.TEST_FRACTION


def raw_frame(split, role, columns):
    """Raw columns for a frame, in the canonical order (S1 file order; pool = S2 rows then S3 rows)."""
    if split == "train":
        rel = f"{SAMPLE_REL}/train_{role}_raw.parquet"
        p = locate(rel)
        if p is None:
            raise FileNotFoundError(f"{rel} missing -- run the 'sample' stage first")
        cols = columns + [c for c in ("pool_source", "samp_match", "samp_dist", "samp_hard") if role == "pool"]
        return load_parquet(p, columns=cols)
    if role == "s1":
        return read_source("test", 1, columns, test_masks(1))
    parts = []
    for s in (2, 3):
        df = read_source("test", s, columns, test_masks(s))
        df["pool_source"] = f"S{s}"
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


_IDS_CACHE = {}


def frame_ids(split, role):
    key = (split, role)
    if key not in _IDS_CACHE:
        _IDS_CACHE[key] = ids_to_int(raw_frame(split, role, ["entity_id"])["entity_id"].values)
    return _IDS_CACHE[key]


def normalized_frame(split, role):
    """Cached normalized frame: train sample in CACHE_DIR (small), test in SCRATCH_DIR (rebuilt per session)."""
    base = CACHE_DIR if split == "train" else SCRATCH_DIR
    rel = f"frames/{FP_SAMPLE if split == 'train' else FP_TEST}/{split}_{role}_norm.parquet"
    p = os.path.join(base, rel)
    if split == "train" and is_done(rel):   # also found in earlier sessions' outputs
        return load_parquet(locate(rel))
    if os.path.exists(p + ".done"):
        return load_parquet(p)
    with stage(f"normalize.{split}.{role}"):
        if split == "train" or role == "s1":
            df = normalize_frame(raw_frame(split, role, RAW_COLS))
        else:   # test pool: one source file at a time to bound peak RAM
            parts = []
            for s in (2, 3):
                raw = read_source("test", s, RAW_COLS, test_masks(s))
                raw["pool_source"] = f"S{s}"
                parts.append(normalize_frame(raw))
                del raw
                gc.collect()
            df = pd.concat(parts, ignore_index=True)
        save_parquet(rel, df, base=base)
        save_json(rel + ".done", dict(rows=len(df)), base=base)
    return df

## 6. Embeddings: precomputed name / address vectors, and the char TF-IDF cache
**Precomputed (no model runs here).** Qwen3-Embedding-0.6B vectors were computed elsewhere for 100% of
train and test: one **name** vector (256-d) and one **address** vector (512-d) per record, int8. §2b put
them in TSV row order. `PrecomputedStore(split, role, view)` maps each row of a frame (by entity id) to its
source file and TSV row, and `gather` returns L2-normalized float vectors.

**Dense retrieval vector.** `CombinedStore` concatenates `[sqrt(w)·name, sqrt(1-w)·addr]` (768-d, unit
norm), so one inner product equals `w·cos(name) + (1-w)·cos(addr)`, and the existing exact dense search,
reverse search and union code work unchanged. `w = EMB_NAME_WEIGHT` is chosen by the `weight_sweep` stage
on the diagnostic queries. The reranker also gets the name-only and address-only cosines as features.

**Char TF-IDF store.** The `tfidf` retriever's vectors (char 2-4-gram TF-IDF + SVD, fitted on a train +
test sample) are cached in row-aligned chunks of `EMBED_CHUNK_ROWS` with ids and a `.done` marker per
chunk; a restart skips completed chunks. The transform runs in `NUM_WORKERS` processes.

**Sanity check** (`embed_check` stage): on 20k labelled train pairs, true pairs must be closer than random
pairs for the name, address and combined vectors. A wrong row mapping would fail this.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD


def combined_text(df):   # copied from er_crossencoder_v2 §5 (char TF-IDF text)
    return (df["name_clean"].astype(str) + " | " + df["addr_clean"].astype(str)).tolist()


# ------------------------- precomputed Qwen3-Embedding-0.6B vectors (name / address views) -------------------------
_SRC_LOOKUP = {}


def _src_lookup(split, s):
    """(sorted int ids, their TSV rows) for one source file -- maps entity ids to rows of the aligned memmaps."""
    key = (split, s)
    if key not in _SRC_LOOKUP:
        _, ids_int = read_ids(split, s)
        order = np.argsort(ids_int, kind="stable")
        _SRC_LOOKUP[key] = (ids_int[order], order.astype(np.int64))
    return _SRC_LOOKUP[key]


class PrecomputedStore:
    """One embedding view ('name' 256-d or 'addr' 512-d) for the rows of one frame (split x role), read from the
    aligned int8 memmaps of §2b. Same read interface as EmbStore: n, dim, gather(rows, use_dim, out_dtype)."""

    def __init__(self, split, role, view):
        self.split, self.role, self.view = split, role, view
        self.dim, self.dtype = int(EMB_DIMS[view]), "int8"
        ids = frame_ids(split, role)
        self.n = len(ids)
        self.tag = f"pre-{view}/{frame_fp(split)}_{split}_{role}"
        self.src = (ids // ID_BASE).astype(np.int8)          # 1, 2 or 3 (the id prefix is the source)
        self.tsv_row = np.empty(self.n, np.int64)
        self._mm = {}
        for s in np.unique(self.src):
            s = int(s)
            m = self.src == s
            sid, srow = _src_lookup(split, s)
            j = np.clip(np.searchsorted(sid, ids[m]), 0, len(sid) - 1)
            if not np.array_equal(sid[j], ids[m]):
                raise RuntimeError(f"{self.tag}: frame ids missing from {split}_source{s}")
            self.tsv_row[m] = srow[j]
            self._mm[s] = np.load(aligned_path(split, s, view), mmap_mode="r")
            if self._mm[s].shape[1] != self.dim:
                raise RuntimeError(f"{aligned_path(split, s, view)}: dim {self._mm[s].shape[1]} != {self.dim}")

    def complete(self):
        return True

    def missing(self):
        return []

    def bytes_per_row(self):
        return self.dim

    def gather(self, rows, use_dim=None, out_dtype=np.float16):
        """(len(rows), use_dim) L2-normalized float vectors; memmap reads in ascending TSV-row order per source."""
        use_dim = self.dim if use_dim is None else int(use_dim)
        assert use_dim <= self.dim
        rows = np.asarray(rows, dtype=np.int64)
        res = np.empty((len(rows), use_dim), dtype=out_dtype)
        if not len(rows):
            return res
        src, tr = self.src[rows], self.tsv_row[rows]
        for s, mm in self._mm.items():
            m = np.flatnonzero(src == s)
            if not len(m):
                continue
            o = np.argsort(tr[m], kind="stable")
            block = np.asarray(mm[tr[m][o], :use_dim], dtype=np.float32)
            nrm = np.linalg.norm(block, axis=1, keepdims=True)
            nrm[nrm == 0] = 1.0
            res[m[o]] = block / nrm
        return res


class CombinedStore:
    """The dense retrieval vector: [sqrt(w) * name, sqrt(1-w) * addr] (unit norm), so that
    <u, v> = w * cos(name_u, name_v) + (1 - w) * cos(addr_u, addr_v)."""

    def __init__(self, split, role, w):
        self.name, self.addr = get_store("name", split, role), get_store("addr", split, role)
        self.w = float(w)
        self.dim, self.dtype, self.n = self.name.dim + self.addr.dim, "float16", self.name.n
        self.tag = f"combined-w{self.w:g}/{self.name.tag}"
        self._a, self._b = math.sqrt(self.w), math.sqrt(1.0 - self.w)

    def complete(self):
        return True

    def missing(self):
        return []

    def bytes_per_row(self):
        return self.name.dim + self.addr.dim

    def gather(self, rows, use_dim=None, out_dtype=np.float16):
        assert use_dim in (None, self.dim), f"the combined vector is used at its full dim {self.dim}"
        res = np.empty((len(rows), self.dim), dtype=out_dtype)
        res[:, :self.name.dim] = self.name.gather(rows, None, np.float32) * self._a
        res[:, self.name.dim:] = self.addr.gather(rows, None, np.float32) * self._b
        return res


# ------------------------- chunked vector cache (char TF-IDF retriever) -------------------------
class EmbStore:
    """Row-aligned chunked embedding store for one frame. Chunk c = frame rows [c*R, (c+1)*R)."""

    def __init__(self, tag, ids_int, dim, dtype, chunk_rows):
        assert dtype in ("float16", "int8"), dtype
        self.tag, self.dim, self.dtype, self.R = tag, int(dim), dtype, int(chunk_rows)
        self.ids = np.asarray(ids_int, dtype=np.int64)
        self.n = len(self.ids)
        self.rel = f"embeddings/{tag}"
        self.n_chunks = math.ceil(self.n / self.R) if self.n else 0
        self.meta = dict(tag=tag, n=self.n, dim=self.dim, dtype=dtype, chunk_rows=self.R,
                         ids_sha=hashlib.sha1(self.ids.tobytes()).hexdigest()[:16])
        man = locate(self.rel + "/manifest.json")
        if man:
            with open(man) as f:
                old = json.load(f)
            diff = {k: (old.get(k), v) for k, v in self.meta.items() if old.get(k) != v}
            if diff:
                raise RuntimeError(f"embedding cache {self.rel} exists with different settings {diff}")
        else:
            save_json(self.rel + "/manifest.json", self.meta)
        self._mm = {}

    def _crel(self, c, kind):
        return f"{self.rel}/chunk_{c:05d}.{kind}.npy"

    def rng(self, c):
        lo = c * self.R
        return lo, min(self.n, lo + self.R)

    def chunk_done(self, c):
        d = read_done(self._crel(c, "vec"))
        lo, hi = self.rng(c)
        return d is not None and d.get("rows") == hi - lo

    def missing(self):
        return [c for c in range(self.n_chunks) if not self.chunk_done(c)]

    def complete(self):
        return not self.missing()

    def bytes_per_row(self):
        return self.dim * (1 if self.dtype == "int8" else 2) + 8

    def write(self, c, vecs):
        lo, hi = self.rng(c)
        assert vecs.shape == (hi - lo, self.dim), (vecs.shape, (hi - lo, self.dim))
        if self.dtype == "int8":   # row-scaled; the per-row scale cancels after re-normalization
            m = np.abs(vecs).max(axis=1, keepdims=True)
            m[m == 0] = 1.0
            arr = np.rint(vecs / m * 127.0).astype(np.int8)
        else:
            arr = vecs.astype(np.float16)
        save_npy(self._crel(c, "ids"), self.ids[lo:hi])
        save_npy(self._crel(c, "vec"), arr)
        mark_done(self._crel(c, "vec"), rows=hi - lo, dim=self.dim, dtype=self.dtype)

    def _chunk(self, c):
        if c not in self._mm:
            p, ip = locate(self._crel(c, "vec")), locate(self._crel(c, "ids"))
            if p is None or ip is None:
                raise FileNotFoundError(f"embedding chunk {self._crel(c, 'vec')} not found in {CACHE_DIR}")
            lo, hi = self.rng(c)
            if not np.array_equal(np.load(ip), self.ids[lo:hi]):
                raise RuntimeError(f"id mismatch in {self._crel(c, 'ids')} -- cache does not belong to this frame")
            self._mm[c] = np.load(p, mmap_mode="r")
        return self._mm[c]

    def gather(self, rows, use_dim, out_dtype=np.float16):
        """(len(rows), use_dim) L2-normalized vectors, read chunk by chunk."""
        rows = np.asarray(rows, dtype=np.int64)
        assert use_dim <= self.dim
        res = np.empty((len(rows), use_dim), dtype=out_dtype)
        if len(rows) == 0:
            return res
        order = np.argsort(rows, kind="stable")
        rs = rows[order]
        ch = rs // self.R
        cut = np.flatnonzero(np.diff(ch)) + 1
        for a, b in zip(np.r_[0, cut], np.r_[cut, len(rs)]):
            c = int(ch[a])
            block = np.asarray(self._chunk(c)[rs[a:b] - c * self.R, :use_dim], dtype=np.float32)
            nrm = np.linalg.norm(block, axis=1, keepdims=True)
            nrm[nrm == 0] = 1.0
            res[order[a:b]] = block / nrm
        return res


class EmbView:
    """Concatenation of stores (e.g. train + test) addressed by combined row position."""

    def __init__(self, stores, use_dim):
        self.stores = list(stores)
        self.dim = int(use_dim)
        self.offsets = np.cumsum([0] + [s.n for s in self.stores])
        self.n = int(self.offsets[-1])
        for s in self.stores:
            assert use_dim <= s.dim, f"use_dim={use_dim} > stored dim {s.dim} of {s.tag}"

    def gather(self, rows, out_dtype=np.float16):
        rows = np.asarray(rows, dtype=np.int64)
        res = np.empty((len(rows), self.dim), dtype=out_dtype)
        which = np.searchsorted(self.offsets, rows, side="right") - 1
        for i, s in enumerate(self.stores):
            m = which == i
            if m.any():
                res[m] = s.gather(rows[m] - self.offsets[i], self.dim, out_dtype)
        return res


# ------------------------- char TF-IDF + SVD (copied from er_crossencoder_v2 §5) -------------------------
def _char_embed_batch(texts, vec, svd):   # copied from er_crossencoder_v2 §5
    Z = svd.transform(vec.transform(texts)).astype(np.float32)
    return _l2_normalize_rows(Z).astype(np.float16)


def _l2_normalize_rows(mat):   # copied from er_crossencoder_v2 §5
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return mat / norms


def fit_char_tfidf_svd(text_sample, n_components, max_features, ngram_range, seed):
    """copied from er_crossencoder_v2 §5 (config globals -> arguments)"""
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=tuple(ngram_range),
                          max_features=max_features, sublinear_tf=True, dtype=np.float32)
    X = vec.fit_transform(text_sample)
    n_comp = min(n_components, X.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, algorithm="randomized", n_iter=4, random_state=seed)
    svd.fit(X)
    print(f"  char TF-IDF vocab={len(vec.vocabulary_):,}  SVD dim={n_comp}  "
          f"explained_var={svd.explained_variance_ratio_.sum():.3f}")
    return vec, svd


class TfidfSvdEmbedder:
    """Char TF-IDF + SVD vectors for the `tfidf` retriever, transformed in NUM_WORKERS processes."""

    def __init__(self, vec, svd):
        self.vec, self.svd = vec, svd
        self.native_dim = int(svd.n_components)

    def embed(self, texts, out_dim):
        texts = list(texts)
        step = max(5000, math.ceil(len(texts) / max(cfg.NUM_WORKERS, 1)))
        batches = [texts[i:i + step] for i in range(0, len(texts), step)]
        if len(batches) > 1 and cfg.NUM_WORKERS > 1:
            parts = joblib.Parallel(n_jobs=cfg.NUM_WORKERS, backend="loky")(
                joblib.delayed(_char_embed_batch)(b, self.vec, self.svd) for b in batches)
        else:
            parts = [_char_embed_batch(b, self.vec, self.svd) for b in batches]
        Z = np.vstack(parts).astype(np.float32)[:, :out_dim] if parts else np.zeros((0, out_dim), np.float32)
        return _l2_normalize_rows(Z), 0


def _fit_sample_texts(text_fn, columns, n_total):
    """Unsupervised fit text: a hash sample of train + of the three TEST files (so test-only
    n-grams, e.g. France, are in the vocabulary -- same idea as er_crossencoder_v2 §5)."""
    rng = np.random.default_rng(cfg.RANDOM_SEED)
    texts = []
    for split, share in (("train", n_total // 2), ("test", n_total // 2)):
        for s in (1, 2, 3):
            n = pq.ParquetFile(SRC_FILE[(split, s)]).metadata.num_rows
            idx = np.sort(rng.choice(n, size=min(n, share // 3), replace=False))
            texts += text_fn(arrow_to_frame(read_raw_table(SRC_FILE[(split, s)], columns).take(pa.array(idx))))
    return texts


_TFIDF_MODELS = {}


def get_tfidf_model(kind="ret"):
    """The char TF-IDF retriever model (name_clean | addr_clean)."""
    if kind in _TFIDF_MODELS:
        return _TFIDF_MODELS[kind]
    assert kind == "ret", kind
    dim, fn, cols = cfg.TFIDF_SVD_DIM, combined_text, ["name_clean", "addr_clean"]
    fpk = fp_of([kind, dim, cfg.TFIDF_MAX_FEATURES, cfg.TFIDF_NGRAM_RANGE, cfg.TFIDF_FIT_SAMPLE, cfg.RANDOM_SEED, CODE_VERSION])
    rel = f"models/tfidf_{kind}_{fpk}.joblib"
    p = locate(rel)
    if p:
        vec, svd = joblib.load(p)
    else:
        with stage(f"fit_tfidf_{kind}"):
            vec, svd = fit_char_tfidf_svd(_fit_sample_texts(fn, cols, cfg.TFIDF_FIT_SAMPLE), dim,
                                          cfg.TFIDF_MAX_FEATURES, cfg.TFIDF_NGRAM_RANGE, cfg.RANDOM_SEED)
            tmp = cache_path(rel) + ".tmp"
            joblib.dump((vec, svd), tmp)
            os.replace(tmp, cache_path(rel))
    _TFIDF_MODELS[kind] = (vec, svd, fpk)
    return _TFIDF_MODELS[kind]


_EMBEDDERS = {}


def get_embedder(kind="tfidf_ret"):
    assert kind == "tfidf_ret", kind
    if kind not in _EMBEDDERS:
        vec, svd, _ = get_tfidf_model("ret")
        _EMBEDDERS[kind] = TfidfSvdEmbedder(vec, svd)
    return _EMBEDDERS[kind]


def release_embedders():
    _EMBEDDERS.clear()
    gc.collect()
    if N_GPUS:
        torch.cuda.empty_cache()


def embed_model_tag():
    """Identifies the dense vectors: the precomputed model files + the name/address weight."""
    models = sorted({d["model"] for d in EMB_INFO.values()})
    return f"pre-{fp_of([models, EMB_DIMS])[:6]}-w{float(resolve_emb_weight()):g}"


def frame_fp(split):
    return FP_SAMPLE if split == "train" else FP_TEST


_STORES = {}


def get_store(kind, split, role):
    """kind 'dense' = combined name+addr vector; 'name' / 'addr' = one precomputed view; 'tfidf' = char TF-IDF."""
    key = (kind, split, role, float(resolve_emb_weight()) if kind == "dense" else None)
    if key not in _STORES:
        if kind in EMB_VIEWS:
            _STORES[key] = PrecomputedStore(split, role, kind)
        elif kind == "dense":
            _STORES[key] = CombinedStore(split, role, cfg.EMB_NAME_WEIGHT)
        elif kind == "tfidf":
            dim, dtype = int(get_tfidf_model("ret")[1].n_components), "float16"
            tag = f"tfidf-{get_tfidf_model('ret')[2]}/{frame_fp(split)}_{split}_{role}_d{dim}_{dtype}"
            _STORES[key] = EmbStore(tag, frame_ids(split, role), dim, dtype, cfg.EMBED_CHUNK_ROWS)
        else:
            raise ValueError(kind)
    return _STORES[key]


def store_texts(kind, split, role):
    assert kind == "tfidf", kind
    return combined_text(raw_frame(split, role, ["name_clean", "addr_clean"]))


def run_embedding(store, texts, kind, label):
    """Fill the TF-IDF store's missing chunks (CPU processes inside the embedder). Returns True when complete."""
    todo = store.missing()
    if not todo:
        print(f"[embed {label}] all {store.n_chunks} chunks already done")
        return True
    assert len(texts) == store.n
    emb = get_embedder(kind)
    t_start, rows_done = time.time(), 0
    remaining_rows = sum(store.rng(c)[1] - store.rng(c)[0] for c in todo)
    print(f"[embed {label}] {len(todo)}/{store.n_chunks} chunks to do ({remaining_rows:,} rows, d={store.dim} {store.dtype})")
    for c in todo:
        if hours_left() <= 0:
            print(f"  [embed {label}] MAX_SESSION_HOURS reached -- stopping cleanly")
            return False
        lo, hi = store.rng(c)
        codes, uniques = pd.factorize(pd.Series(texts[lo:hi], dtype=object))
        vu, _ = emb.embed(list(uniques), store.dim)
        store.write(c, vu[codes])
        rows_done += hi - lo
    secs = time.time() - t_start
    print(f"  [embed {label}] {rows_done:,} rows in {secs:.1f}s ({rows_done / max(secs, 1e-9):,.0f} rows/s)")
    return store.complete()


# ------------------------- sanity check of the precomputed vectors -------------------------
def embedding_sanity_train():
    """True pairs must be closer than random pairs, per view and combined (catches a wrong row mapping)."""
    rel = f"reports/embedding_sanity_{FP_SAMPLE}_{fp_of(sorted(EMB_INFO.items()))}.json"
    if locate(rel):
        with open(locate(rel)) as f:
            REPORT["embedding_sanity"] = json.load(f)
        print(f"embedding sanity (cached): {REPORT['embedding_sanity']}")
        return REPORT["embedding_sanity"]
    with stage("embed_check"):
        s1_ids = raw_frame("train", "s1", ["entity_id"])["entity_id"].values
        pool_ids = raw_frame("train", "pool", ["entity_id"])["entity_id"].values
        gt = load_parquet(locate(f"{SAMPLE_REL}/gt.parquet"))
        a, b = explode_gt(gt)
        i1 = pd.Index(s1_ids).get_indexer(a)
        i2 = pd.Index(pool_ids).get_indexer(b)
        rng = np.random.default_rng(cfg.RANDOM_SEED)
        sel = rng.choice(len(i1), size=min(len(i1), 20000), replace=False)
        rnd = rng.integers(0, len(pool_ids), len(sel))
        res = {}
        for kind in ("name", "addr", "combined_w0.5"):
            if kind in EMB_VIEWS:
                v1, v2 = get_store(kind, "train", "s1"), get_store(kind, "train", "pool")
            else:
                v1, v2 = CombinedStore("train", "s1", 0.5), CombinedStore("train", "pool", 0.5)
            q = v1.gather(i1[sel], None, np.float32)
            pos = (q * v2.gather(i2[sel], None, np.float32)).sum(1)
            neg = (q * v2.gather(rnd, None, np.float32)).sum(1)
            res[kind] = dict(true_pair_cos_mean=round(float(pos.mean()), 4), random_pair_cos_mean=round(float(neg.mean()), 4),
                             p_true_gt_random=round(float((pos > neg).mean()), 4))
            log(f"  {kind:13s}: mean cos true pairs={res[kind]['true_pair_cos_mean']:.4f}  random pairs="
                  f"{res[kind]['random_pair_cos_mean']:.4f}  P(true > random)={res[kind]['p_true_gt_random']:.4f}")
            if pos.mean() <= neg.mean():
                raise RuntimeError(f"{kind} embeddings do not separate true from random pairs -- check the row mapping")
        REPORT["embedding_sanity"] = res
        save_json(rel, res)
    return res


## 7. Retrieval: dense (precomputed embeddings and char TF-IDF), BM25, exact keys
**Dense.** Exact inner-product search in fp16 on the GPUs (no ANN index, so recall numbers measure the
embeddings, not an index approximation). The pool side is cut into shards of at most `SEARCH_SHARD_ROWS`
that are spread **round-robin over all GPUs** (one thread per GPU with its own running top-k; the per-GPU
lists are merged at the end), so a 10M-row pool never has to fit in GPU or host RAM at once. Pools smaller
than 50k rows are searched on one GPU.

**Country buckets.** Every retriever searches only inside the query's country bucket (dense, reverse,
BM25 and TF-IDF per `(country, source)` partition; every exact key starts with the country). This is
valid because the country invariant holds on the full train ground truth (0 cross-country positives).
Test-only France has its own bucket. The recall report and test inference **assert** that no candidate
crosses a country.

**Directions.** Forward S1→S2 and S1→S3 run as *separate* per-source top-K lists, so one source can
never crowd out the other. Reverse S2→S1 and S3→S1 top-`REV_TOP_K` adds candidates and `rev_rank`
features. On the diagnostic context the reverse pass searches `REV_DIAG_TOP_K` deep, so the recall report
can show whether `REV_TOP_K` is enough. The cross-country "global" pass is an optional experiment
(`ENABLE_GLOBAL_FALLBACK`, default off).

**BM25.** A hashed sparse matrix product. Tokens are field-prefixed (`n:` name core, `a:` address,
`c:` city, `s:` state), with the name weighted highest on the query side. Tokens in more than
`BM25_MAX_DF_FRAC` of a partition's documents are pruned. Token hashing runs in `NUM_WORKERS` processes;
the query x document products run on **all GPUs** as CUDA sparse x sparse products (`torch.sparse.mm`,
query chunks from a shared queue, one thread per GPU, chunk halved on OOM) with the per-query top-k taken on
the GPU. `BM25_USE_GPU=False` uses the CPU scipy path; a self-test checks both give the same scores.

**Exact keys.** Built only from non-empty values, since missing is never a match:
`name_core|country`, `name_sorted|city`, `postcode|first-name-token`, `house_no|city` (the two city keys are
inactive: the raw data has no city). Over-general blocks are skipped (logged). Survivors are ranked in-block
by the other field's token-set similarity.

**Checkpointing.** Every (retriever, country, source) partition is written to its own parquet file with a
`.done` marker as soon as it finishes, so a crash or a `MAX_SESSION_HOURS` stop loses at most one partition.


In [ ]:
# ------------------------- exact inner-product search (copied from er_crossencoder_v2 §5, multi-GPU) -------------------------
SCORE_BUDGET_BYTES = 4e9   # score matrix per search step on one GPU (24 GB cards)


def build_ann_index(vecs, device=None):
    """vecs: (n, d) L2-normalized. Exact search: fp16 torch tensor on a GPU, else faiss/numpy on the CPU."""
    device = device or DEVICES[0]
    if device.startswith("cuda"):
        return ("torch", torch.as_tensor(np.asarray(vecs), dtype=torch.float16, device=device))
    if FAISS_AVAILABLE:
        index = faiss.IndexFlatIP(vecs.shape[1])
        for start in range(0, len(vecs), 500_000):
            index.add(np.ascontiguousarray(vecs[start:start + 500_000], dtype=np.float32))
        return ("faiss", index)
    return ("numpy", np.asarray(vecs, dtype=np.float32))


def ann_search(index_handle, queries, k, chunk=8192):
    """Returns (scores, idx), each (n_queries, k). idx == -1 where fewer than k results exist.
    (copied; changes: score-matrix budget is SCORE_BUDGET_BYTES, top-k runs on the fp16 scores directly,
    and the queries go to the index's own device)"""
    kind, index = index_handle
    nq = queries.shape[0]
    out_s = np.full((nq, k), -1.0, dtype=np.float32)
    out_i = np.full((nq, k), -1, dtype=np.int64)
    if nq == 0 or k == 0:
        return out_s, out_i
    if kind == "torch":
        chunk = int(max(64, min(chunk, SCORE_BUDGET_BYTES // max(1, index.shape[0] * index.element_size()))))
    for start in range(0, nq, chunk):
        q = np.ascontiguousarray(queries[start:start + chunk], dtype=np.float32)
        if kind == "faiss":
            s, i = index.search(q, k)
            out_s[start:start + len(q)] = s
            out_i[start:start + len(q)] = i
        elif kind == "torch":
            qt = torch.as_tensor(q, dtype=index.dtype, device=index.device)
            sims = qt @ index.T
            kk = min(k, sims.shape[1])
            ts, ti = torch.topk(sims, kk, dim=1)
            out_s[start:start + len(q), :kk] = ts.float().cpu().numpy()
            out_i[start:start + len(q), :kk] = ti.cpu().numpy()
            del qt, sims, ts, ti
        else:
            sims = q @ index.T
            kk = min(k, sims.shape[1])
            ti = np.argpartition(-sims, kk - 1, axis=1)[:, :kk]
            ts = np.take_along_axis(sims, ti, axis=1)
            order = np.argsort(-ts, axis=1)
            out_s[start:start + len(q), :kk] = np.take_along_axis(ts, order, axis=1)
            out_i[start:start + len(q), :kk] = np.take_along_axis(ti, order, axis=1)
    out_i[out_s < -0.5] = -1  # FAISS pads missing results with very negative scores
    return out_s, out_i


def free_index(index_handle):
    kind, index = index_handle
    dev = index.device if kind == "torch" else None
    del index
    gc.collect()
    if dev is not None:
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()

# ------------------------- sharded exact search over all GPUs (new) -------------------------
MULTI_GPU_MIN_POOL = 50_000   # smaller pools are searched on one GPU (a second copy would cost more than it saves)


def _merge_topk(best_s, best_i, a, s, i, step=200_000):
    k = best_s.shape[1]
    for r0 in range(0, len(s), step):
        sl = slice(a + r0, a + r0 + min(step, len(s) - r0))
        cs = np.concatenate([best_s[sl], s[r0:r0 + step]], axis=1)
        ci = np.concatenate([best_i[sl], i[r0:r0 + step]], axis=1)
        if cs.shape[1] > k:
            top = np.argpartition(-cs, k - 1, axis=1)[:, :k]
            cs, ci = np.take_along_axis(cs, top, 1), np.take_along_axis(ci, top, 1)
        o = np.argsort(-cs, axis=1, kind="stable")
        best_s[sl], best_i[sl] = np.take_along_axis(cs, o, 1), np.take_along_axis(ci, o, 1)


def _dense_search_cpu(q_view, q_rows, p_view, p_rows, k):
    """CPU fallback (no GPU): shards + numpy running top-k."""
    nq = len(q_rows)
    best_s = np.full((nq, k), -np.inf, np.float32)
    best_i = np.full((nq, k), -1, np.int64)
    Q = q_view.gather(q_rows)
    for s0 in range(0, len(p_rows), cfg.SEARCH_SHARD_ROWS):
        pr = p_rows[s0:s0 + cfg.SEARCH_SHARD_ROWS]
        handle = build_ann_index(p_view.gather(pr), "cpu")
        s, i = ann_search(handle, Q, min(k, len(pr)))
        ok = i >= 0
        _merge_topk(best_s, best_i, 0, np.where(ok, s, -np.inf).astype(np.float32), np.where(ok, i + s0, -1))
    return best_s, best_i


def dense_search(q_view, q_rows, p_view, p_rows, k):
    """Exact top-k inner product of q_rows vs p_rows. Returns (scores, idx) with idx = position into p_rows
    (-1 = none). The pool is cut into shards spread round-robin over all GPUs (one thread per GPU). Each GPU keeps
    the queries and its running top-k ON the GPU (fp16 matmul, torch.topk merge), so the CPU only gathers vectors;
    the per-GPU lists are merged on cuda:0 at the end."""
    q_rows, p_rows = np.asarray(q_rows, np.int64), np.asarray(p_rows, np.int64)
    nq, k = len(q_rows), int(min(k, len(p_rows)))
    if nq == 0 or k <= 0:
        return np.full((nq, max(k, 0)), -np.inf, np.float32), np.full((nq, max(k, 0)), -1, np.int64)
    if DEVICE != "cuda":
        s, i = _dense_search_cpu(q_view, q_rows, p_view, p_rows, k)
        i[~np.isfinite(s)] = -1
        return s, i
    devices = DEVICES if len(p_rows) >= MULTI_GPU_MIN_POOL else DEVICES[:1]
    shard = int(min(cfg.SEARCH_SHARD_ROWS, math.ceil(len(p_rows) / len(devices))))
    shards = [(s0, p_rows[s0:s0 + shard]) for s0 in range(0, len(p_rows), shard)]
    used = devices[:len(shards)]
    Qh = q_view.gather(q_rows)                       # fp16 host copy, uploaded once per GPU
    results, errors = {}, []

    def worker(di, dev):
        try:
            with torch.cuda.device(dev):
                Q = torch.as_tensor(Qh, dtype=torch.float16, device=dev)
                bs = torch.full((nq, k), -float("inf"), dtype=torch.float32, device=dev)
                bi = torch.full((nq, k), -1, dtype=torch.int64, device=dev)
                for s0, pr in shards[di::len(used)]:
                    P = torch.as_tensor(p_view.gather(pr), dtype=torch.float16, device=dev)
                    kk = min(k, len(pr))
                    step = int(max(64, min(65536, SCORE_BUDGET_BYTES // max(1, len(pr) * 2))))
                    for a in range(0, nq, step):
                        ts, ti = torch.topk(Q[a:a + step] @ P.T, kk, dim=1)
                        cs = torch.cat([bs[a:a + step], ts.float()], 1)
                        ci = torch.cat([bi[a:a + step], ti + s0], 1)
                        ms, mi = torch.topk(cs, k, dim=1)
                        bs[a:a + step], bi[a:a + step] = ms, torch.gather(ci, 1, mi)
                        del ts, ti, cs, ci, ms, mi
                    del P
                results[di] = (bs, bi)
                del Q
        except Exception as e:   # surfaced after join
            errors.append(e)

    if len(used) == 1:
        worker(0, used[0])
    else:
        threads = [threading.Thread(target=worker, args=(di, dev), daemon=True) for di, dev in enumerate(used)]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
    if errors:
        raise errors[0]
    d0 = used[0]
    bs, bi = results.pop(0)
    for di in sorted(results):                       # merge the other GPUs' lists on the first GPU, in row blocks
        s2, i2 = results.pop(di)
        for a in range(0, nq, 1_000_000):
            # via host memory: direct GPU->GPU (peer) copies silently return zeros on this machine (IOMMU / PHB)
            cs = torch.cat([bs[a:a + 1_000_000], s2[a:a + 1_000_000].cpu().to(d0)], 1)
            ci = torch.cat([bi[a:a + 1_000_000], i2[a:a + 1_000_000].cpu().to(d0)], 1)
            ms, mi = torch.topk(cs, k, dim=1)
            bs[a:a + 1_000_000], bi[a:a + 1_000_000] = ms, torch.gather(ci, 1, mi)
        del s2, i2
    best_s, best_i = bs.cpu().numpy(), bi.cpu().numpy()
    del bs, bi
    for dev in used:
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
    best_i[~np.isfinite(best_s)] = -1
    return best_s, best_i


def _topk_frame(q_rows, p_rows, s, i, reverse=False):
    """Long format. Forward: q_rows are S1 and i indexes p_rows (pool). Reverse: swapped."""
    valid = i >= 0
    rank = np.broadcast_to(np.arange(i.shape[1], dtype=np.int32), i.shape)[valid]
    a = np.broadcast_to(q_rows[:, None], i.shape)[valid]
    b = p_rows[i[valid]]
    s1, pool = (b, a) if reverse else (a, b)
    return pd.DataFrame({"s1_pos": s1.astype(np.int64), "pool_pos": pool.astype(np.int64),
                         "score": s[valid].astype(np.float32), "rank": rank})

# ------------------------- BM25 (hashed sparse product; GPU sparse x sparse on all GPUs) -------------------------
from sklearn.feature_extraction.text import HashingVectorizer

BM25_N_FEATURES = 2 ** 22
_TOK_RE = re.compile(r"[a-z0-9]+")
_BM25_COLS = {"n": "name_core", "a": "addr_clean", "c": "city_norm", "s": "state_norm"}
BM25_GPU_QUERY_CHUNK = 8192   # queries per GPU sparse product (halved automatically on OOM)


def _identity_analyzer(tokens):
    return tokens


_HV = HashingVectorizer(analyzer=_identity_analyzer, n_features=BM25_N_FEATURES, alternate_sign=False,
                        norm=None, dtype=np.float32)


def _bm25_tokens(frame, rows, fields):
    cols = [(f + ":", frame[_BM25_COLS[f]].values) for f in fields]
    out = []
    for r in rows:
        toks = []
        for pre, col in cols:
            toks.extend(pre + t for t in _TOK_RE.findall(col[r]) if len(t) >= 2)
        out.append(toks)
    return out


def _hash_token_block(col_values, prefixes):
    """Hashed term-count matrix for one block of rows (col_values: one object array per field)."""
    docs = []
    for r in range(len(col_values[0]) if col_values else 0):
        toks = []
        for pre, col in zip(prefixes, col_values):
            toks.extend(pre + t for t in _TOK_RE.findall(col[r]) if len(t) >= 2)
        docs.append(toks)
    return _HV.transform(docs).tocsr()


def _bm25_matrix(frame, rows, fields, block=200_000):
    """Hashed term counts for frame[rows] over `fields`, blocks hashed in NUM_WORKERS processes."""
    rows = np.asarray(rows, np.int64)
    cols = [frame[_BM25_COLS[f]].values for f in fields]
    prefixes = [f + ":" for f in fields]
    blocks = [rows[i:i + block] for i in range(0, len(rows), block)]
    if len(blocks) > 1 and cfg.NUM_WORKERS > 1:
        mats = joblib.Parallel(n_jobs=min(cfg.NUM_WORKERS, len(blocks)), backend="loky")(
            joblib.delayed(_hash_token_block)([c[b] for c in cols], prefixes) for b in blocks)
    else:
        mats = [_hash_token_block([c[b] for c in cols], prefixes) for b in blocks]
    return sp.vstack(mats).tocsr() if mats else sp.csr_matrix((0, BM25_N_FEATURES), dtype=np.float32)


def bm25_index(frame, rows, fields):
    X = _bm25_matrix(frame, rows, fields)
    N = X.shape[0]
    df = np.bincount(X.indices, minlength=BM25_N_FEATURES)
    cap = max(cfg.BM25_MIN_DF_CAP, cfg.BM25_MAX_DF_FRAC * N)
    keep = (df > 0) & (df <= cap)
    idf = np.log1p((N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
    avgdl = max(float(dl.mean()) if N else 1.0, 1e-6)
    row_of = np.repeat(np.arange(N, dtype=np.int32), np.diff(X.indptr))
    tf = X.data
    k1, b = cfg.BM25_K1, cfg.BM25_B
    w = idf[X.indices] * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[row_of] / avgdl))
    w[~keep[X.indices]] = 0.0
    W = sp.csr_matrix((w.astype(np.float32), X.indices, X.indptr), shape=X.shape)
    W.eliminate_zeros()
    return dict(WT=W.T.tocsr(), keep=keep, n_docs=N, pruned=int((df > cap).sum()), vocab=int((df > 0).sum()))


def bm25_query_matrix(frame, rows, fields, keep):
    Q = None
    for f in fields:
        M = _bm25_matrix(frame, rows, (f,))
        M.data[:] = cfg.BM25_FIELD_WEIGHTS.get(f, 1.0)   # binary presence x field weight
        Q = M if Q is None else Q + M
    Q = Q.tocsr()
    Q.data *= keep[Q.indices]
    Q.eliminate_zeros()
    return Q


def _bm25_topk_cpu(Q, WT, k, out):
    for start in range(0, Q.shape[0], cfg.BM25_QUERY_CHUNK):
        C = (Q[start:start + cfg.BM25_QUERY_CHUNK] @ WT).tocsr()
        for r in range(C.shape[0]):   # loop copied from er_crossencoder_v2 name_token_block
            lo, hi = C.indptr[r], C.indptr[r + 1]
            if hi == lo:
                continue
            data, cols = C.data[lo:hi], C.indices[lo:hi]
            if hi - lo > k:
                top = np.argpartition(-data, k - 1)[:k]
                data, cols = data[top], cols[top]
            out.append((np.full(len(cols), start + r, dtype=np.int64), cols.astype(np.int64), data.astype(np.float32)))


def _csr_to_torch(M, device):
    M = M.tocsr()
    M.sort_indices()
    return torch.sparse_csr_tensor(torch.from_numpy(M.indptr.astype(np.int64)), torch.from_numpy(M.indices.astype(np.int64)),
                                   torch.from_numpy(M.data.astype(np.float32)), size=M.shape, device=device)


def _gpu_rowwise_topk(C, k):
    """Top-k columns per row of a CUDA CSR matrix -> (row, col, value) tensors, rows ascending, values descending."""
    crow, col, val = C.crow_indices(), C.col_indices(), C.values()
    if val.numel() == 0:
        z = torch.zeros(0, dtype=torch.int64, device=val.device)
        return z, z, val
    row = torch.repeat_interleave(torch.arange(C.shape[0], device=val.device), crow[1:] - crow[:-1])
    o = torch.argsort(val, descending=True, stable=True)
    o = o[torch.argsort(row[o], stable=True)]
    row, col, val = row[o], col[o], val[o]
    pos = torch.arange(val.numel(), device=val.device) - crow[row]
    keep = pos < k
    return row[keep], col[keep], val[keep]


def _bm25_topk_gpu(Q, WT, k, out):
    """Q @ WT on every GPU (query chunks from a shared queue, one thread per GPU), top-k per query on the GPU."""
    chunks = queue.Queue()
    for start in range(0, Q.shape[0], BM25_GPU_QUERY_CHUNK):
        chunks.put((start, min(Q.shape[0], start + BM25_GPU_QUERY_CHUNK)))
    lock, errors = threading.Lock(), []

    def run(dev, WTg, lo, hi):
        try:
            A = _csr_to_torch(Q[lo:hi], dev)
            r, c, v = _gpu_rowwise_topk(torch.sparse.mm(A, WTg), k)
            res = (r.cpu().numpy() + lo, c.cpu().numpy(), v.cpu().numpy())
            del A, r, c, v
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if hi - lo <= 64:
                raise
            mid = (lo + hi) // 2
            run(dev, WTg, lo, mid)
            run(dev, WTg, mid, hi)
            return
        with lock:
            out.append(res)

    def worker(dev):
        try:
            WTg = _csr_to_torch(WT, dev)
            while True:
                try:
                    lo, hi = chunks.get_nowait()
                except queue.Empty:
                    break
                run(dev, WTg, lo, hi)
            del WTg
            with torch.cuda.device(dev):
                torch.cuda.empty_cache()
        except Exception as e:   # surfaced after join
            errors.append(e)

    threads = [threading.Thread(target=worker, args=(d,), daemon=True) for d in DEVICES]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    if errors:
        raise errors[0]


def bm25_search(doc_frame, doc_rows, q_frame, q_rows, k, fields=("n", "a", "c", "s"), use_gpu=None):
    """Returns (qi, di, score, rank): positions into q_rows / doc_rows, rank within query by score."""
    use_gpu = (cfg.BM25_USE_GPU and DEVICE == "cuda") if use_gpu is None else use_gpu
    idx = bm25_index(doc_frame, doc_rows, fields)
    Q = bm25_query_matrix(q_frame, q_rows, fields, idx["keep"])
    out = []
    (_bm25_topk_gpu if use_gpu else _bm25_topk_cpu)(Q, idx["WT"], k, out)
    if not out:
        z = np.zeros(0, np.int64)
        return z, z, np.zeros(0, np.float32), np.zeros(0, np.int32)
    qi = np.concatenate([o[0] for o in out]).astype(np.int64)
    di = np.concatenate([o[1] for o in out]).astype(np.int64)
    s = np.concatenate([o[2] for o in out]).astype(np.float32)
    o = np.lexsort((di, -s, qi))   # deterministic order regardless of which GPU finished first
    qi, di, s = qi[o], di[o], s[o]
    return qi, di, s, within_group_rank(qi, s).astype(np.int32)

# ------------------------- exact / structured keys (new) -------------------------
EXACT_KEYS = ["exact_name", "exact_sorted_city", "exact_postcode", "exact_house_city"]


def exact_key_values(frame, rows, kind):
    c = frame["country_norm"].values[rows].astype(object)
    if kind == "exact_name":
        v = frame["name_core"].values[rows]
        ok = np.fromiter((len(x) >= 3 for x in v), dtype=bool, count=len(v))
        key = c + "|" + v
    elif kind == "exact_sorted_city":
        v, city = frame["name_sorted"].values[rows], frame["city_norm"].values[rows]
        ok = (v != "") & (city != "")
        key = c + "|" + city + "|" + v
    elif kind == "exact_postcode":
        pcode = frame["postcode"].values[rows]
        first = np.array([x.split(" ", 1)[0] for x in frame["name_core"].values[rows]], dtype=object)
        ok = (pcode != "") & (first != "")
        key = c + "|" + pcode + "|" + first
    elif kind == "exact_house_city":
        h, city = frame["house_no"].values[rows], frame["city_norm"].values[rows]
        ok = (h != "") & (city != "")
        key = c + "|" + city + "|" + h
    else:
        raise ValueError(kind)
    return np.where(ok, key, "").astype(object)


def exact_block(s1_frame, q_rows, pool_frame, pool_rows, kind, stats=None):
    qk = exact_key_values(s1_frame, q_rows, kind)
    pk = exact_key_values(pool_frame, pool_rows, kind)
    qm, pm = qk != "", pk != ""
    empty = pd.DataFrame({"s1_pos": np.zeros(0, np.int64), "pool_pos": np.zeros(0, np.int64),
                          "score": np.zeros(0, np.float32), "rank": np.zeros(0, np.int32)})
    if not qm.any() or not pm.any():
        return empty
    codes, uniq = pd.factorize(np.concatenate([qk[qm], pk[pm]]))
    nqm = int(qm.sum())
    qc, pcd = codes[:nqm], codes[nqm:]
    nq = np.bincount(qc, minlength=len(uniq))
    npl = np.bincount(pcd, minlength=len(uniq))
    both = (nq > 0) & (npl > 0)
    allowed = both & (npl <= cfg.EXACT_MAX_BLOCK) & (nq * npl <= cfg.EXACT_MAX_PAIRS)
    if stats is not None:
        stats.update(shared_keys=int(both.sum()), skipped_blocks=int((both & ~allowed).sum()),
                     queries_with_key=nqm, queries_in_skipped=int((both & ~allowed)[qc].sum()))
    dq = pd.DataFrame({"code": qc, "s1_pos": q_rows[qm]})
    dp = pd.DataFrame({"code": pcd, "pool_pos": pool_rows[pm]})
    m = dq[allowed[dq["code"].values]].merge(dp[allowed[dp["code"].values]], on="code")
    if not len(m):
        return empty
    s1p, pp = m["s1_pos"].values.astype(np.int64), m["pool_pos"].values.astype(np.int64)
    col = "addr_clean" if kind in ("exact_name", "exact_sorted_city") else "name_clean"
    score = np.empty(len(m), np.float32)
    for a in range(0, len(m), cfg.PAIR_BATCH):
        sl = slice(a, a + cfg.PAIR_BATCH)
        score[sl] = _pairwise(rf_fuzz.token_set_ratio, s1_frame[col].values[s1p[sl]],
                              pool_frame[col].values[pp[sl]], _rf_default_process) / 100
    grp = s1p * 2 + (pool_frame["pool_source"].values[pp] == "S3")
    rank = within_group_rank(grp, score).astype(np.int32)
    keep = rank < cfg.EXACT_TOP_K
    if stats is not None:
        stats["pairs_before_topk"] = int(len(m))
    return pd.DataFrame({"s1_pos": s1p[keep], "pool_pos": pp[keep], "score": score[keep], "rank": rank[keep]})

## 8. Frames, retrieval contexts, and the candidate union
One split is held in memory at a time (`FR`, train or test; ~12.5M normalized rows each), never both.
A *context* selects the query rows and the retrieval depth:

| context | Source-1 queries | pool searched | reverse-search S1 index | depth |
|---|---|---|---|---|
| `diag` | `DIAG_QUERY_FRACTION` (1%) hash sample of train S1 | **full** train pool | all train S1 | diagnostic (`DIAG_TOP_K`, `REV_DIAG_TOP_K`) |
| `train` | **all** train S1 | full train pool | all train S1 | production |
| `test` | all test S1 | full test pool | all test S1 | production |

`diag` measures recall curves and the oracle F0.5 at realistic distractor density (the full pool) for a
cheap query sample; it is how the top-Ks and `MAX_CANDIDATES` are chosen. `train` produces the reranker's
training set (on `GBDT_TRAIN_ENTITY_FRACTION` of the entities) and the full train evaluation (all other
entities). `test` produces the submission.

**Union.** Candidates from every retriever are merged on the `(s1, pool)` pair (copied
`generate_candidates` logic). Each retriever keeps `{r}_score`, `{r}_rank` (sentinel when absent)
and `by_{r}`. The per-S1 cap uses the **best per-retriever rank** (round-robin), so no single
retriever crowds out the others. Large contexts are unioned in S1 blocks of `QUERY_BLOCK_ROWS`.


In [ ]:
RETRIEVERS = ["dense", "dense_global", "dense_rev", "bm25", "tfidf"] + EXACT_KEYS
RANK_SENTINEL = 9999
# FR holds the normalized frames of ONE split at a time (train or test), never both: 12.5M rows each.
# n_tr_s1 / n_tr_pool = number of TRAIN rows at the front of the frames (all rows for train, 0 for test).
FR = SimpleNamespace(split=None, s1=None, pool=None, n_tr_s1=0, n_tr_pool=0, s1_cc=None, pool_cc=None, pool_is_s3=None)


def country_codes(s1_country, pool_country):
    """Integer country codes shared by both sides (fast bucket checks on millions of pairs)."""
    cats = pd.Index(sorted(set(s1_country) | set(pool_country)))
    return cats.get_indexer(s1_country).astype(np.int16), cats.get_indexer(pool_country).astype(np.int16)


def cross_country_count(s1_pos, pool_pos):
    """Number of (S1, pool) pairs whose countries differ -- must be 0 while retrieval is country-partitioned."""
    return int((FR.s1_cc[np.asarray(s1_pos, np.int64)] != FR.pool_cc[np.asarray(pool_pos, np.int64)]).sum())


SHARED_STR_COLS = ["country_norm", "state_norm", "city_norm", "name_suffix", "postcode", "house_no", "pool_source"]
SHARED_STR_MAX_UNIQUE = 0.05   # share only where values repeat a lot; high-cardinality columns would not shrink


def to_object_strings(df):
    """String columns as plain object arrays ("" for missing), as arrow_to_frame returns them. A frame that was
    just normalized (not read back from its parquet cache) has pandas>=3 Arrow-backed "str" columns instead:
    different types reaching the feature code than at training time, and slow to slice/pickle for the workers."""
    for c in df.columns:
        if df[c].dtype != object and pd.api.types.is_string_dtype(df[c].dtype):
            df[c] = df[c].to_numpy(dtype=object, na_value="")
    return df


def share_low_cardinality_strings(df, label):
    """Point every row of a low-cardinality object column at ONE str object per distinct value (pd.factorize).
    Same dtype (object) and identical values, so no downstream code changes; saves ~50 bytes per cell."""
    shared = []
    for c in SHARED_STR_COLS:
        if c not in df.columns or df[c].dtype != object or not len(df):
            continue
        codes, uniques = pd.factorize(df[c].values)
        if (codes < 0).any() or len(uniques) > SHARED_STR_MAX_UNIQUE * len(df):
            continue
        df[c] = np.asarray(uniques, dtype=object)[codes]
        shared.append(f"{c}({len(uniques):,})")
    gc.collect()
    mem_mark(f"frames {label}: shared string objects for {shared}", df)


def ensure_frames(split):
    assert split in ("train", "test"), split
    if FR.split == split and FR.s1 is not None:
        return FR
    with stage("frames", split=split):
        FR.s1 = FR.pool = FR.s1_cc = FR.pool_cc = FR.pool_is_s3 = None
        FR.split = None
        gc.collect()
        mem_mark("frames released")
        FR.s1 = to_object_strings(normalized_frame(split, "s1"))
        mem_mark(f"frames {split} S1 loaded", FR.s1)
        share_low_cardinality_strings(FR.s1, f"{split} S1")
        FR.pool = to_object_strings(normalized_frame(split, "pool"))
        mem_mark(f"frames {split} pool loaded", FR.pool)
        share_low_cardinality_strings(FR.pool, f"{split} pool")
        FR.n_tr_s1, FR.n_tr_pool = (len(FR.s1), len(FR.pool)) if split == "train" else (0, 0)
        for c in ("samp_match", "samp_dist", "samp_hard"):
            FR.pool[c] = (FR.pool[c].fillna(0) if c in FR.pool.columns else 0)
            FR.pool[c] = FR.pool[c].astype(np.int8)
        FR.pool_is_s3 = FR.pool["pool_source"].values == "S3"
        FR.s1_cc, FR.pool_cc = country_codes(FR.s1["country_norm"].values, FR.pool["country_norm"].values)
        FR.split = split
        assert FR.s1["entity_id"].is_unique and FR.pool["entity_id"].is_unique, "entity ids must be unique"
        log(f"frames[{split}]: S1 rows={len(FR.s1):,}  pool rows={len(FR.pool):,}  RAM={_ram_gb()}")
    return FR


def get_views(kind, split):
    stores = {r: get_store(kind, split, r) for r in ("s1", "pool")}
    for r, st_ in stores.items():
        if st_.complete():
            continue
        if kind == "tfidf":
            with stage(f"embed_tfidf.{split}.{r}"):
                run_embedding(st_, store_texts("tfidf", split, r), "tfidf_ret", f"tfidf {split}/{r}")
        if not st_.complete():
            raise RuntimeError(f"{kind} vectors for {split}/{r} incomplete ({len(st_.missing())} chunks) -- run again "
                               "to continue (MAX_SESSION_HOURS reached?)")
    return EmbView([stores["s1"]], stores["s1"].dim), EmbView([stores["pool"]], stores["pool"].dim)


RETR_KEYS = ["DENSE_TOP_K", "REV_TOP_K", "REV_DIAG_TOP_K", "ENABLE_GLOBAL_FALLBACK", "GLOBAL_FALLBACK_TOP_K",
             "BM25_TOP_K", "BM25_K1", "BM25_B", "BM25_FIELD_WEIGHTS", "BM25_MAX_DF_FRAC", "BM25_MIN_DF_CAP",
             "ENABLE_TFIDF_RETRIEVER", "TFIDF_TOP_K", "EXACT_TOP_K", "EXACT_MAX_BLOCK", "EXACT_MAX_PAIRS", "DIAG_TOP_K",
             "DIAG_QUERY_FRACTION", "EMB_NAME_WEIGHT"]


def retr_cfg_fp():
    """Fingerprint of everything a recall / gate number depends on (checked before a gate is trusted)."""
    return cfg_fp(RETR_KEYS + ["MAX_CANDIDATES"], extra=[RETR_CODE_VERSION, FP_SAMPLE, embed_model_tag()])


def diag_query_mask():
    """Train S1 rows searched at diagnostic depth (hash sample; the full train pool is always searched)."""
    return hash_u01(ids_to_int(FR.s1["entity_id"].values[:FR.n_tr_s1]), SALT["diag"]) < cfg.DIAG_QUERY_FRACTION


def make_ctx(name):
    """'diag'  = DIAG_QUERY_FRACTION of train S1 vs the FULL train pool at diagnostic depth (recall curves, choose K);
       'train' = all train S1 vs the full train pool at production depth (training set + train evaluation);
       'test'  = all test S1 vs the full test pool at production depth (submission)."""
    split = "test" if name == "test" else "train"
    ensure_frames(split)
    ctx = SimpleNamespace(name=name, split=split, depth="diag" if name == "diag" else "prod")
    ctx.dense = get_views("dense", split)
    ctx.emb_name, ctx.emb_addr = get_views("name", split), get_views("addr", split)   # reranker features
    ctx.tfidf = get_views("tfidf", split) if cfg.ENABLE_TFIDF_RETRIEVER else None
    n1, npool = len(FR.s1), len(FR.pool)
    ctx.pool, ctx.rev = np.arange(npool), np.arange(n1)   # the reverse index always holds every S1 of the split
    if name == "diag":
        ctx.q = np.flatnonzero(diag_query_mask())
    elif name in ("train", "test"):
        ctx.q = np.arange(n1)
    else:
        raise ValueError(name)
    tfp = get_tfidf_model("ret")[2] if cfg.ENABLE_TFIDF_RETRIEVER else None
    ctx.fp = cfg_fp(RETR_KEYS, extra=[name, FP_SAMPLE if split == "train" else FP_TEST, embed_model_tag(), tfp, n1, npool,
                                      len(ctx.q), RETR_CODE_VERSION])
    ctx.rrel = f"retrieval/{name}_{ctx.fp}"   # persisted in CACHE_DIR; a re-run finds finished partitions there
    log(f"context {name}: {len(ctx.q):,} queries vs {npool:,} pool records ({ctx.depth} depth)  fp={ctx.fp}")
    return ctx




def ctx_rel(ctx, *parts):
    return "/".join([ctx.rrel] + [str(p) for p in parts])


def ctx_save_parquet(ctx, df, *parts, row_group_size=None):
    rel = ctx_rel(ctx, *parts)
    save_parquet(rel, df, row_group_size=row_group_size)
    mark_done(rel, rows=len(df))
    return rel


def ctx_read_parquet(ctx, *parts, filters=None):
    p = locate(ctx_rel(ctx, *parts))
    if p is None:
        raise FileNotFoundError(ctx_rel(ctx, *parts))
    return pq.read_table(p, filters=filters).to_pandas()


def ctx_list_done(ctx, *parts):
    """Names of completed parquet parts under ctx/<parts>/ across this session and earlier sessions."""
    names = set()
    for base in [CACHE_DIR] + CACHE_SEARCH_PATHS:
        names |= {os.path.basename(p)[:-5] for p in glob.glob(os.path.join(base, ctx_rel(ctx, *parts), "*.parquet.done"))}
    return sorted(names)


def _slug(s):
    return re.sub(r"[^A-Za-z0-9]+", "-", str(s)) or "none"


def depth_k(ctx, prod_k):
    return max(prod_k, cfg.DIAG_TOP_K) if ctx.depth == "diag" else prod_k


def partitions(ctx):
    qc = FR.s1["country_norm"].values[ctx.q]
    pcn = FR.pool["country_norm"].values[ctx.pool]
    psrc = FR.pool["pool_source"].values[ctx.pool]
    rc = FR.s1["country_norm"].values[ctx.rev]
    out = []
    for c in sorted(set(qc)):
        for s in ("S2", "S3"):
            out.append((c, s, ctx.q[qc == c], ctx.pool[(pcn == c) & (psrc == s)], ctx.rev[rc == c]))
    return out


def _dense_fwd(views, qr, pr, k):
    s, i = dense_search(views[0], qr, views[1], pr, k)
    return _topk_frame(qr, pr, s, i)


def _dense_rev(ctx, pr, rev_rows):
    k = max(cfg.REV_TOP_K, cfg.REV_DIAG_TOP_K) if ctx.depth == "diag" else cfg.REV_TOP_K
    s, i = dense_search(ctx.dense[1], pr, ctx.dense[0], rev_rows, k)
    # keep hits on this context's queries only (the scale context's reverse index also holds test S1). Masking the
    # index array first keeps memory bounded -- building the frame first would materialize len(pr) * k rows.
    in_q = np.zeros(len(FR.s1), bool)
    in_q[ctx.q] = True
    i = np.where((i >= 0) & in_q[rev_rows[np.maximum(i, 0)]], i, -1)
    return _topk_frame(pr, rev_rows, s, i, reverse=True)


def _dense_global(ctx, src):
    pr = ctx.pool[FR.pool["pool_source"].values[ctx.pool] == src]
    pool_countries = list(set(FR.pool["country_norm"].values[pr]))
    orphan = ~np.isin(FR.s1["country_norm"].values[ctx.q].astype(str), np.array(pool_countries, dtype=str))
    parts = []
    for mask, k in ((~orphan, cfg.GLOBAL_FALLBACK_TOP_K), (orphan, depth_k(ctx, cfg.DENSE_TOP_K))):
        qr = ctx.q[mask]
        if len(qr) and len(pr):
            parts.append(_dense_fwd(ctx.dense, qr, pr, k))
    REPORT.setdefault("orphan_queries", {})[f"{ctx.name}_{src}"] = int(orphan.sum())
    return pd.concat(parts, ignore_index=True) if parts else _topk_frame(np.zeros(0, np.int64), pr, np.zeros((0, 1), np.float32), np.zeros((0, 1), np.int64))


def _bm25(qr, pr, k):
    qi, di, s, rank = bm25_search(FR.pool, pr, FR.s1, qr, k)
    return pd.DataFrame({"s1_pos": qr[qi], "pool_pos": pr[di], "score": s, "rank": rank})


def _sorted_s1_rank(df):
    """df ordered by (s1_pos, rank). Forward retrievers already come out in that order: the stable sort would
    return the same rows, so it is skipped there (it copies the whole partition, up to 66M rows)."""
    s1, rk = df["s1_pos"].values, df["rank"].values
    d = np.diff(s1)
    if len(df) < 2 or ((d >= 0).all() and (np.diff(rk)[d == 0] >= 0).all()):
        return df
    return df.sort_values(["s1_pos", "rank"], kind="stable")


def run_retrievers(ctx):
    jobs = []
    for c, s, qr, pr, rr in partitions(ctx):
        tag = f"{_slug(c)}_{s}"
        if not len(qr) or not len(pr):
            continue
        jobs.append(("dense", tag, lambda qr=qr, pr=pr: _dense_fwd(ctx.dense, qr, pr, depth_k(ctx, cfg.DENSE_TOP_K))))
        if len(rr):
            jobs.append(("dense_rev", tag, lambda pr=pr, rr=rr: _dense_rev(ctx, pr, rr)))
        jobs.append(("bm25", tag, lambda qr=qr, pr=pr: _bm25(qr, pr, depth_k(ctx, cfg.BM25_TOP_K))))
        if ctx.tfidf is not None:
            jobs.append(("tfidf", tag, lambda qr=qr, pr=pr: _dense_fwd(ctx.tfidf, qr, pr, depth_k(ctx, cfg.TFIDF_TOP_K))))
    if cfg.ENABLE_GLOBAL_FALLBACK:
        for s in ("S2", "S3"):
            jobs.append(("dense_global", s, lambda s=s: _dense_global(ctx, s)))
    for key in EXACT_KEYS:
        jobs.append((key, "all", lambda key=key: exact_block(FR.s1, ctx.q, FR.pool, ctx.pool, key,
                                                           REPORT.setdefault("exact_stats", {}).setdefault(f"{ctx.name}_{key}", {}))))
    times = defaultdict(float)
    todo = [j for j in jobs if not is_done(ctx_rel(ctx, j[0], f"{j[1]}.parquet"))]
    log(f"[{ctx.name}] retrieval: {len(jobs) - len(todo)}/{len(jobs)} partition jobs already checkpointed, {len(todo)} to run")
    for n_done, (r, part, fn) in enumerate(todo, 1):
        if hours_left() <= 0:
            raise RuntimeError("MAX_SESSION_HOURS reached during retrieval -- finished partitions are saved; run again")
        t0 = time.time()
        df = _sorted_s1_rank(fn())
        ctx_save_parquet(ctx, df, r, f"{part}.parquet", row_group_size=500_000)   # checkpoint per partition
        times[r] += time.time() - t0
        log(f"  [{ctx.name}] {n_done}/{len(todo)} {r:17s} {part:14s} {len(df):>11,} pairs  {time.time() - t0:7.1f}s  "
            f"RSS {rss_now_gb()[0]} GB")
        del df
    rel = f"reports/retrieval_seconds_{ctx.name}_{ctx.fp}.json"
    prev_p = locate(rel)
    prev = {}
    if prev_p:
        with open(prev_p) as f:
            prev = {k: v for k, v in json.load(f).items() if k != "fp_sample"}
    merged = {k: round(float(prev.get(k, 0)) + times.get(k, 0.0), 2) for k in set(prev) | set(times)}
    if times:   # a fully cached (resumed) run must not overwrite earlier sessions' timings with zeros
        save_json(rel, dict(merged, fp_sample=FP_SAMPLE))
    REPORT.setdefault("retrieval_seconds", {})[ctx.name + REPORT_SUFFIX] = merged


def load_retr(ctx, lo=None, hi=None):
    out = {}
    filters = [("s1_pos", ">=", int(lo)), ("s1_pos", "<", int(hi))] if lo is not None else None
    for r in RETRIEVERS:
        dfs = [ctx_read_parquet(ctx, r, name, filters=filters) for name in ctx_list_done(ctx, r)]
        dfs = [d for d in dfs if len(d)]
        if not dfs:
            continue
        d = pd.concat(dfs, ignore_index=True)
        if r == "dense_rev":   # union ordering: rank among this S1's reverse hits (rev_rank is the pool's view)
            d["order_rank"] = within_group_rank(d["s1_pos"].values, d["score"].values).astype(np.int32)
        out[r] = d
    return out


PROD_TOPK = lambda: {"dense": cfg.DENSE_TOP_K, "dense_global": cfg.DENSE_TOP_K, "dense_rev": cfg.REV_TOP_K,
                     "bm25": cfg.BM25_TOP_K, "tfidf": cfg.TFIDF_TOP_K, **{k: cfg.EXACT_TOP_K for k in EXACT_KEYS}}


def truncate_retr(retr, override=None):
    """Production view of diagnostic-depth lists. The reverse list's per-S1 order_rank is recomputed on the
    truncated hits, exactly as production computes it (otherwise train and test see different best_rank)."""
    tk = dict(PROD_TOPK(), **(override or {}))
    out = {}
    for r, d in retr.items():
        d = d[d["rank"].values < tk[r]]
        if r == "dense_rev" and len(d):
            d = d.assign(order_rank=within_group_rank(d["s1_pos"].values, d["score"].values).astype(np.int32))
        out[r] = d
    return out


def group_positions(group, rank_key, score):
    """0-based position within each group when sorted by (rank_key asc, score desc)."""
    group = np.asarray(group)
    if len(group) == 0:
        return np.zeros(0, np.int64)
    order = np.lexsort((-np.asarray(score, dtype=np.float64), np.asarray(rank_key), group))
    g = group[order]
    starts = np.r_[0, np.flatnonzero(np.diff(g)) + 1]
    pos_sorted = np.arange(len(g)) - np.repeat(starts, np.diff(np.r_[starts, len(g)]))
    pos = np.empty(len(g), dtype=np.int64)
    pos[order] = pos_sorted
    return pos


def build_union(retr, n_pool, cap=None):
    """Dedupe on (s1, pool); keep every retriever's score/rank/flag; cap per S1 by best rank (round-robin)."""
    names = [r for r in RETRIEVERS if r in retr and len(retr[r])]
    n_pool = np.int64(n_pool)
    keys = [retr[r]["s1_pos"].values.astype(np.int64) * n_pool + retr[r]["pool_pos"].values.astype(np.int64) for r in names]
    uniq, inv = (np.unique(np.concatenate(keys), return_inverse=True) if names else (np.zeros(0, np.int64), np.zeros(0, np.int64)))
    n = len(uniq)
    out = {"s1_pos": (uniq // n_pool).astype(np.int64), "pool_pos": (uniq % n_pool).astype(np.int64)}
    best = np.full(n, RANK_SENTINEL, np.int32)
    nret = np.zeros(n, np.int8)
    off = 0
    for r, k in zip(names, keys):
        d = retr[r]
        idx = inv[off:off + len(k)]
        off += len(k)
        o = np.lexsort((d["rank"].values, idx))
        first = np.r_[True, idx[o][1:] != idx[o][:-1]] if len(o) else np.zeros(0, bool)
        sel = o[first]
        sc = np.zeros(n, np.float32)
        rk = np.full(n, RANK_SENTINEL, np.int32)
        fl = np.zeros(n, np.int8)
        sc[idx[sel]] = d["score"].values[sel]
        rk[idx[sel]] = d["rank"].values[sel]
        fl[idx[sel]] = 1
        ordr = (d["order_rank"].values if "order_rank" in d.columns else d["rank"].values)[sel]
        best[idx[sel]] = np.minimum(best[idx[sel]], ordr)
        out[f"{r}_score"], out[f"{r}_rank"], out[f"by_{r}"] = sc, rk, fl
        nret += fl
    for r in RETRIEVERS:
        if r not in names:
            out[f"{r}_score"] = np.zeros(n, np.float32)
            out[f"{r}_rank"] = np.full(n, RANK_SENTINEL, np.int32)
            out[f"by_{r}"] = np.zeros(n, np.int8)
    out["n_retrievers"], out["best_rank"] = nret, best
    tie = np.where(out["by_dense"] == 1, out["dense_score"],
                   np.where(out["by_dense_global"] == 1, out["dense_global_score"], -1.0))
    order = np.lexsort((-tie, -nret.astype(np.int16), best, out["s1_pos"]))
    g = out["s1_pos"][order]
    starts = np.r_[0, np.flatnonzero(np.diff(g)) + 1] if n else np.zeros(0, np.int64)
    pos = np.empty(n, np.int64)
    pos[order] = np.arange(n) - np.repeat(starts, np.diff(np.r_[starts, n])) if n else pos
    wide = pd.DataFrame(out)
    wide["pos_in_entity"] = pos
    if cap is not None:
        wide = wide[wide["pos_in_entity"].values < cap].reset_index(drop=True)
    return wide

## 9. Candidate recall, the gating diagnostic
For known matches: *is the true match in the candidate set at all?* The reranker cannot recover a
match that blocking missed. The report covers:

* **Per retriever:** pair Recall@25/50/100/200/500, using its own ordering (per-source lists
  interleaved by rank).
* **Union:** the diagnostic-depth union, then the **configured union**. The latter uses production
  top-Ks, is capped at `MAX_CANDIDATES`, and is what the reranker actually sees.
* **Per-query candidate counts:** mean, P50 and P95.
* **Summary measures:** the entity "all matches found" rate and the oracle macro F0.5 (copied
  `blocking_ceiling`).
* **Leave-one-out:** the recall lost by dropping each retriever. Close to zero means it can be
  switched off at test time to save compute.
* **Candidate caps:** recall and candidates per query (mean, P50, P95) at every cap in `RECALL_KS`
  (50 / 100 / 200 / 500). This is shown for the configured top-Ks and for the diagnostic-depth union,
  which is what raising the top-Ks would give.
* **Reverse K:** the share of true pairs whose S1 is in the pool record's reverse top-k, and the
  configured-union recall if `REV_TOP_K` were k. If it still grows at the configured value, raise
  `REV_TOP_K`.
* **Country buckets:** the cross-country candidate count per retriever, which must be 0 (it raises
  otherwise), and the share of true pairs inside the same country bucket (1.0 under the invariant).
* **Breakdowns:** by country, by S2/S3, and by Latin vs non-Latin candidate name.

In [ ]:
def _dedupe_best_rank(frames, n_pool):
    """Concatenate retriever frames and keep each (s1, pool) pair once, at its best rank."""
    E = pd.concat(frames, ignore_index=True)
    E["key"] = E["s1_pos"].values.astype(np.int64) * n_pool + E["pool_pos"].values
    E = E.sort_values(["key", "rank"], kind="stable").drop_duplicates("key")
    return E


def _count_stats(counts):
    return dict(mean=round(float(counts.mean()), 2) if len(counts) else 0.0,
                p50=float(np.percentile(counts, 50)) if len(counts) else 0.0,
                p95=float(np.percentile(counts, 95)) if len(counts) else 0.0)


RECALL_BLOCK_PAIRS = 5_000_000   # retriever rows per S1 block when the diagnostic-depth union is evaluated


def _deep_union_stats(retr, n_pool, true_keys, eval_mask, Ks):
    """Everything recall_report needs from the diagnostic-depth union, accumulated over contiguous S1 blocks,
    so the full union (45M rows x 32 columns, ~17 GB while being built) is never materialized. Union rows and
    their pos_in_entity are per-S1 quantities, so each block's rows are exactly the full union's rows for it."""
    n_s1 = len(eval_mask)
    hits = {K: 0 for K in Ks}
    hits_all = xc = 0
    found = {K: np.zeros(n_s1, np.int32) for K in Ks}   # true in-cap candidates per S1 (eval entities only)
    cnt = np.zeros(n_s1, np.int64)                     # union candidates per S1
    q = np.flatnonzero(eval_mask)
    n_rows = sum(len(d) for d in retr.values())
    n_blocks = max(1, int(np.ceil(n_rows / RECALL_BLOCK_PAIRS)))
    edges = np.unique(np.r_[q[np.linspace(0, len(q), n_blocks, endpoint=False).astype(np.int64)], n_s1]) if len(q) else [0, n_s1]
    edges[0] = 0
    for bi, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        part = {}
        for r, d in retr.items():
            s = d["s1_pos"].values
            part[r] = d[(s >= lo) & (s < hi)]
        W = build_union(part, n_pool)
        del part
        w_s1, w_pos = W["s1_pos"].values, W["pos_in_entity"].values
        t = np.isin(w_s1 * n_pool + W["pool_pos"].values, true_keys) & eval_mask[w_s1]
        for K in Ks:
            m = t & (w_pos < K)
            hits[K] += int(m.sum())
            found[K] += np.bincount(w_s1[m], minlength=n_s1).astype(np.int32)
        hits_all += int(t.sum())
        xc += cross_country_count(w_s1, W["pool_pos"].values)
        cnt += np.bincount(w_s1, minlength=n_s1)
        if bi == 0:
            mem_mark(f"recall_report: diagnostic-depth union block 1/{len(edges) - 1}", W)
        del W, w_s1, w_pos, t
    gc.collect()
    return dict(hits=hits, hits_all=hits_all, xc=xc, found=found, cnt=cnt, n_rows=int(cnt.sum()))


def recall_report(retr, n_pool, true_keys, n_true, eval_mask, label):
    Ks = cfg.RECALL_KS
    total = int(n_true[eval_mask].sum())
    n_pool = np.int64(n_pool)
    rows = []
    mem_mark("recall_report: retriever lists", retr)

    def _row(name, s1, keys, pos, pool):
        t = np.isin(keys, true_keys) & eval_mask[s1]
        row = dict(retriever=name)
        for K in Ks:
            row[f"R@{K}"] = round(float(t[pos < K].sum()) / max(total, 1), 4)
        row["R@all"] = round(float(t.sum()) / max(total, 1), 4)
        row["cross_country"] = cross_country_count(s1, pool)
        return row

    for r in RETRIEVERS:
        d = retr.get(r)
        if d is None or not len(d):
            continue
        s1 = d["s1_pos"].values
        keys = s1.astype(np.int64) * n_pool + d["pool_pos"].values
        ordr = d["order_rank"].values if "order_rank" in d.columns else d["rank"].values
        rows.append(_row(r, s1, keys, group_positions(s1, ordr, d["score"].values), d["pool_pos"].values))
    ex = [retr[k] for k in EXACT_KEYS if k in retr and len(retr[k])]
    if ex:
        E = _dedupe_best_rank(ex, n_pool)
        rows.append(_row("exact (any key)", E["s1_pos"].values, E["key"].values,
                         group_positions(E["s1_pos"].values, E["rank"].values, E["score"].values), E["pool_pos"].values))
        del E
    deep = _deep_union_stats(retr, n_pool, true_keys, eval_mask, Ks)
    rows.append(dict(retriever="UNION (diagnostic depth)",
                     **{f"R@{K}": round(float(deep["hits"][K]) / max(total, 1), 4) for K in Ks},
                     **{"R@all": round(float(deep["hits_all"]) / max(total, 1), 4), "cross_country": deep["xc"]}))
    prod = truncate_retr(retr)
    P = build_union(prod, n_pool)
    mem_mark("recall_report: configured-top-K union", P)
    pkeys = P["s1_pos"].values * n_pool + P["pool_pos"].values
    rows.append(_row("UNION (configured top-Ks)", P["s1_pos"].values, pkeys, P["pos_in_entity"].values, P["pool_pos"].values))
    table = pd.DataFrame(rows)
    n_cross = int(table.loc[table["retriever"].isin(RETRIEVERS), "cross_country"].sum())
    if n_cross and not cfg.ENABLE_GLOBAL_FALLBACK:
        raise RuntimeError(f"country partition violated: {n_cross:,} retrieved candidates are from another country "
                           f"({table[['retriever', 'cross_country']].to_dict('records')})")
    # ---- candidate caps: recall + candidates/query at every cap, configured top-Ks and diagnostic depth ----
    cap_rows = []
    w_s1 = P["s1_pos"].values
    tw = np.isin(pkeys, true_keys)
    cnt = np.bincount(w_s1, minlength=len(n_true))[eval_mask]
    for K in Ks:
        m = P["pos_in_entity"].values < K
        _, orc = blocking_ceiling(w_s1[m], tw[m], n_true, eval_mask)
        cap_rows.append(dict(union="configured top-Ks", cap=K, recall=round(float((tw[m] & eval_mask[w_s1[m]]).sum()) / max(total, 1), 4),
                             oracle_macro_f05=round(orc, 4), **{f"cands_{k}": v for k, v in _count_stats(np.minimum(cnt, K)).items()}))
    cnt = deep["cnt"][eval_mask]
    for K in Ks:   # oracle = every true in-cap candidate predicted: macro_f05 needs only the per-S1 counts
        s1_found = np.repeat(np.arange(len(n_true)), deep["found"][K])
        orc = macro_f05(s1_found, np.ones(len(s1_found), bool), n_true, eval_mask)
        cap_rows.append(dict(union="diagnostic depth", cap=K, recall=round(float(deep["hits"][K]) / max(total, 1), 4),
                             oracle_macro_f05=round(orc, 4), **{f"cands_{k}": v for k, v in _count_stats(np.minimum(cnt, K)).items()}))
    del deep, w_s1, tw
    gc.collect()
    # ---- reverse K: is REV_TOP_K deep enough? ----
    rev_rows = []
    d = retr.get("dense_rev")
    if d is not None and len(d):
        dr = d["rank"].values
        t_rev = np.isin(d["s1_pos"].values.astype(np.int64) * n_pool + d["pool_pos"].values, true_keys) & eval_mask[d["s1_pos"].values]
        searched = int(dr.max()) + 1
        for k in sorted({1, 3, 5, 10, 20, 50, 100, cfg.REV_TOP_K}):
            if k > searched:
                continue
            Pk = build_union(truncate_retr(retr, {"dense_rev": k}), n_pool, cap=cfg.MAX_CANDIDATES)
            yk = np.isin(Pk["s1_pos"].values * n_pool + Pk["pool_pos"].values, true_keys) & eval_mask[Pk["s1_pos"].values]
            rev_rows.append(dict(rev_k=k, reverse_recall=round(float(t_rev[dr < k].sum()) / max(total, 1), 4),
                                 union_recall_at_cap=round(float(yk.sum()) / max(total, 1), 4),
                                 configured=k == cfg.REV_TOP_K))
            del Pk
    capm = P["pos_in_entity"].values < cfg.MAX_CANDIDATES
    Pc = P[capm]
    yc = np.isin(pkeys[capm], true_keys)
    rec_cap = float((yc & eval_mask[Pc["s1_pos"].values]).sum()) / max(total, 1)
    pair_rec, oracle = blocking_ceiling(Pc["s1_pos"].values, yc, n_true, eval_mask)
    counts = np.bincount(Pc["s1_pos"].values, minlength=len(n_true))[eval_mask]
    found = np.bincount(Pc["s1_pos"].values[yc], minlength=len(n_true))
    has = eval_mask & (n_true > 0)
    loo = {}
    for r in list(prod):
        Pr = build_union({k: v for k, v in prod.items() if k != r}, n_pool, cap=cfg.MAX_CANDIDATES)
        yr = np.isin(Pr["s1_pos"].values * n_pool + Pr["pool_pos"].values, true_keys) & eval_mask[Pr["s1_pos"].values]
        loo[r] = round(rec_cap - float(yr.sum()) / max(total, 1), 5)
    # breakdowns over TRUE pairs (denominators include matches blocking missed)
    t_s1, t_pool = true_keys // n_pool, true_keys % n_pool
    t_eval = eval_mask[t_s1]
    t_found = np.isin(true_keys, pkeys[capm])
    same_country = float((FR.s1_cc[t_s1] == FR.pool_cc[t_pool])[t_eval].mean()) if t_eval.any() else float("nan")
    brk = {}
    for name, grp in [("country", FR.s1["country_norm"].values[t_s1]), ("source", FR.pool["pool_source"].values[t_pool]),
                      ("cand_script", np.where(FR.pool["non_latin"].values[t_pool] == 1, "non-latin", "latin"))]:
        brk[name] = {str(g): dict(true_pairs=int((t_eval & (grp == g)).sum()),
                                  recall=round(float((t_found & t_eval & (grp == g)).sum()) / max(int((t_eval & (grp == g)).sum()), 1), 4))
                     for g in sorted(set(grp[t_eval]))}
    res = dict(label=label, eval_entities=int(eval_mask.sum()), true_pairs=total, table=table.to_dict("records"),
               union_recall_at_cap=round(rec_cap, 4), max_candidates=cfg.MAX_CANDIDATES,
               oracle_macro_f05=round(oracle, 4), entity_all_found_rate=round(float((found[has] >= n_true[has]).mean()) if has.any() else float("nan"), 4),
               cands_per_query=dict(_count_stats(counts), max=int(counts.max()) if len(counts) else 0,
                                    zero_frac=round(float((counts == 0).mean()), 4)),
               cap_table=cap_rows, reverse_k_table=rev_rows, cross_country_candidates=n_cross,
               same_country_true_pair_frac=round(same_country, 6),
               leave_one_out_recall_drop=loo, breakdown=brk, retr_cfg_fp=retr_cfg_fp())
    print(f"\n==== candidate recall [{label}]  eval entities={res['eval_entities']:,}  true pairs={total:,} ====")
    print(table.to_string(index=False))
    print(f"configured union @ MAX_CANDIDATES={cfg.MAX_CANDIDATES}: recall={rec_cap:.4f}  oracle macro F0.5={oracle:.4f}  "
          f"entities with all matches found={res['entity_all_found_rate']}")
    print(f"candidates/query: {res['cands_per_query']}")
    print(f"country buckets: cross-country candidates={n_cross:,}   true pairs inside the same country bucket="
          f"{same_country:.4%}")
    print("candidate caps (recall, oracle F0.5 and candidates/query at each cap):\n"
          + pd.DataFrame(cap_rows).to_string(index=False))
    if rev_rows:
        print(f"reverse K (searched depth {searched}; configured REV_TOP_K={cfg.REV_TOP_K}):\n"
              + pd.DataFrame(rev_rows).to_string(index=False))
    print(f"leave-one-out recall drop (~0 -> retriever could be disabled at test time): {loo}")
    for k, v in brk.items():
        print(f"  by {k}: {v}")
    return res

## 10. Pairwise features
The name, address, geography and source features are computed by the copied `_features_chunk`
(rapidfuzz `cpdist`, vectorized, Jaro-Winkler included) on the lower-cased raw name / address. Feature chunks
(entity-aligned) are computed in `NUM_WORKERS` forked processes that share the frames. New in this notebook:

| group | features |
|---|---|
| embedding | `emb_cos` (combined `w·name + (1-w)·addr` cosine) for **every** pair (exact score reused when a dense retriever found it), `emb_name_cos`, `emb_addr_cos` (each precomputed view on its own), `tfidf_cos` |
| provenance | `{r}_score`, `{r}_rank`, `by_{r}` for every retriever; `n_retrievers`, `best_rank`, `mutual_top5` (forward & reverse top-5) |
| address | house-number agree/conflict (0.5 = unknown), presence flags, postcode agreement, address-word Jaccard |
| missingness / script | missing city/state/address flags per side (city is always missing in the raw data), `cand_non_latin` |
| entity context | rank / gap to the entity's best `emb_cos`, gap to best name score, BM25 relative to entity best, candidate count |

Missing is never treated as a match: soft-equality features give 0.5 when either side is missing.
Contact features (phone, email, website) cannot be built, because this dataset has no such columns.

In [ ]:
try:
    from rapidfuzz.process import cpdist as _cpdist
    from rapidfuzz.utils import default_process as _rf_default_process
    from rapidfuzz.distance import JaroWinkler as _JW
    CPDIST_AVAILABLE = True
except ImportError:
    CPDIST_AVAILABLE = False
    from rapidfuzz.utils import default_process as _rf_default_process
    from rapidfuzz.distance import JaroWinkler as _JW

_RF_WORKERS = -1   # rapidfuzz threads per call; set to 1 inside feature worker processes (no oversubscription)


# ------------------------- copied from er_crossencoder_v2 §6 -------------------------
def _pairwise(scorer, a, b, processor=None):
    if CPDIST_AVAILABLE:
        return _cpdist(a, b, scorer=scorer, processor=processor, workers=_RF_WORKERS, dtype=np.float32)
    if processor is not None:
        return np.fromiter((scorer(processor(x), processor(y)) for x, y in zip(a, b)), dtype=np.float32, count=len(a))
    return np.fromiter((scorer(x, y) for x, y in zip(a, b)), dtype=np.float32, count=len(a))


def _soft_equal(a, b):
    """1.0 if both present and equal, 0.0 if both present and different, 0.5 if either missing.
    Works on object arrays directly (no astype(str): that makes a huge fixed-width unicode copy)."""
    present = (a != "") & (b != "")
    return np.where(present, (a == b).astype(np.float32), np.float32(0.5)).astype(np.float32)


def _set_jaccard(sa, sb):
    return np.fromiter(
        ((len(x & y) / len(x | y)) if (x or y) else 0.0 for x, y in zip(sa, sb)),
        dtype=np.float32, count=len(sa))


def _features_chunk(c, s1_df, pool_df):
    i1 = c["s1_pos"].values
    i2 = c["pool_pos"].values
    g1 = lambda col: s1_df[col].values[i1]
    g2 = lambda col: pool_df[col].values[i2]
    f = {}
    n1, n2 = g1("name_clean"), g2("name_clean")
    f["name_ratio"] = _pairwise(rf_fuzz.ratio, n1, n2, _rf_default_process) / 100
    f["name_partial"] = _pairwise(rf_fuzz.partial_ratio, n1, n2, _rf_default_process) / 100
    f["name_token_sort"] = _pairwise(rf_fuzz.token_sort_ratio, n1, n2, _rf_default_process) / 100
    f["name_token_set"] = _pairwise(rf_fuzz.token_set_ratio, n1, n2, _rf_default_process) / 100
    c1, c2 = g1("name_core"), g2("name_core")
    f["core_jw"] = _pairwise(_JW.normalized_similarity, c1, c2)
    f["core_exact"] = ((c1 == c2) & (c1 != "")).astype(np.float32)
    f["sorted_exact"] = ((g1("name_sorted") == g2("name_sorted")) & (c1 != "")).astype(np.float32)
    a1, a2 = g1("name_acronym"), g2("name_acronym")
    c1_compact = np.array([s.replace(" ", "") for s in c1], dtype=object)
    c2_compact = np.array([s.replace(" ", "") for s in c2], dtype=object)
    f["acronym_match"] = (((a1 != "") & (a1 == c2_compact)) | ((a2 != "") & (a2 == c1_compact))).astype(np.float32)
    p1, p2 = g1("name_phonetic"), g2("name_phonetic")
    f["phonetic_match"] = ((p1 == p2) & (p1 != "")).astype(np.float32)
    f["suffix_agree"] = _soft_equal(g1("name_suffix"), g2("name_suffix"))
    f["name_jaccard"] = _set_jaccard(g1("name_tokens"), g2("name_tokens"))
    l1 = np.fromiter((len(s) for s in n1), dtype=np.float32, count=len(n1))
    l2 = np.fromiter((len(s) for s in n2), dtype=np.float32, count=len(n2))
    f["name_len_diff"] = np.abs(l1 - l2) / np.maximum(np.maximum(l1, l2), 1)
    ad1, ad2 = g1("addr_clean"), g2("addr_clean")
    f["addr_ratio"] = _pairwise(rf_fuzz.ratio, ad1, ad2, _rf_default_process) / 100
    f["addr_token_sort"] = _pairwise(rf_fuzz.token_sort_ratio, ad1, ad2, _rf_default_process) / 100
    f["addr_token_set"] = _pairwise(rf_fuzz.token_set_ratio, ad1, ad2, _rf_default_process) / 100
    d1, d2 = g1("addr_digits"), g2("addr_digits")
    f["digits_jaccard"] = _set_jaccard(d1, d2)
    f["digits_conflict"] = np.fromiter(
        (1.0 if (x and y and not (x & y)) else 0.0 for x, y in zip(d1, d2)), dtype=np.float32, count=len(d1))
    f["city_match"] = _soft_equal(g1("city_norm"), g2("city_norm"))
    f["state_match"] = _soft_equal(g1("state_norm"), g2("state_norm"))
    f["country_match"] = _soft_equal(g1("country_norm"), g2("country_norm"))
    f["city_ratio"] = _pairwise(rf_fuzz.ratio, g1("city_norm"), g2("city_norm")) / 100
    f["cand_is_s3"] = (g2("pool_source") == "S3").astype(np.float32)
    return pd.DataFrame({k: v.astype(np.float32) for k, v in f.items()}, index=c.index)


def within_group_rank(group, score):
    """0-based rank of each element within its group, by descending score (numpy, O(n log n))."""
    if len(group) == 0:
        return np.zeros(0, dtype=np.int64)
    order = np.lexsort((-score, group))
    g_sorted = group[order]
    starts = np.r_[0, np.flatnonzero(np.diff(g_sorted)) + 1]
    sizes = np.diff(np.r_[starts, len(g_sorted)])
    rank_sorted = np.arange(len(g_sorted)) - np.repeat(starts, sizes)
    rank = np.empty(len(group), dtype=np.int64)
    rank[order] = rank_sorted
    return rank

# ------------------------- new features -------------------------
BASE_FEATURE_COLS = [
    "name_ratio", "name_partial", "name_token_sort", "name_token_set", "core_jw", "core_exact",
    "sorted_exact", "acronym_match", "phonetic_match", "suffix_agree", "name_jaccard", "name_len_diff",
    "addr_ratio", "addr_token_sort", "addr_token_set", "digits_jaccard", "digits_conflict",
    "city_match", "state_match", "country_match", "city_ratio", "cand_is_s3",
]
NEW_FEATURE_COLS = (
    ["emb_cos", "emb_name_cos", "emb_addr_cos", "tfidf_cos"]
    + [f"{r}_{k}" for r in RETRIEVERS for k in ("score", "rank")] + [f"by_{r}" for r in RETRIEVERS]
    + ["n_retrievers", "best_rank", "mutual_top5",
       "house_agree", "house_present_s1", "house_present_cand", "postcode_agree", "addr_word_jaccard",
       "s1_city_missing", "cand_city_missing", "cand_state_missing", "cand_addr_missing", "cand_non_latin",
       "emb_rank_in_entity", "emb_gap_to_best", "name_gap_to_best", "bm25_rel_to_best", "n_cands_entity"]
)
FEATURE_COLS = BASE_FEATURE_COLS + NEW_FEATURE_COLS
DENSE_LIKE = ("dense", "dense_global", "dense_rev")


def _pair_cos(views, s1_rows, pool_rows):
    out = np.empty(len(s1_rows), np.float32)
    order = np.argsort(pool_rows, kind="stable")   # sequential reads of the pool cache
    for a in range(0, len(order), cfg.PAIR_BATCH):
        o = order[a:a + cfg.PAIR_BATCH]
        out[o] = (views[0].gather(s1_rows[o], np.float32) * views[1].gather(pool_rows[o], np.float32)).sum(1)
    return out


def _chunk_rows(C):
    """The S1 / pool frame rows a candidate chunk needs (sorted unique positions, as pair_features uses them)."""
    return (FR.s1.iloc[np.unique(C["s1_pos"].values)].reset_index(drop=True),
            FR.pool.iloc[np.unique(C["pool_pos"].values)].reset_index(drop=True))


def pair_features(C, ctx, rows=None):
    """Features for candidate pairs C (complete entities only -- entity-context features need them).
    rows = _chunk_rows(C) when computed by the caller (forked workers must not touch FR, see features_in_chunks)."""
    i1, i2 = C["s1_pos"].values, C["pool_pos"].values
    u1, inv1 = np.unique(i1, return_inverse=True)
    u2, inv2 = np.unique(i2, return_inverse=True)
    r1, r2 = rows if rows is not None else _chunk_rows(C)
    sub1 = add_set_columns(r1)
    sub2 = add_set_columns(r2)
    base = _features_chunk(pd.DataFrame({"s1_pos": inv1, "pool_pos": inv2}), sub1, sub2)
    f = {c: base[c].values for c in BASE_FEATURE_COLS}
    n = len(C)
    qcos = np.full(n, np.nan, np.float32)
    for r in DENSE_LIKE:
        m = C[f"by_{r}"].values == 1
        qcos[m] = C[f"{r}_score"].values[m]
    miss = np.isnan(qcos)
    if miss.any():
        qcos[miss] = _pair_cos(ctx.dense, i1[miss], i2[miss])
    f["emb_cos"] = qcos
    f["emb_name_cos"] = _pair_cos(ctx.emb_name, i1, i2)
    f["emb_addr_cos"] = _pair_cos(ctx.emb_addr, i1, i2)
    if ctx.tfidf is not None:
        tcos = np.where(C["by_tfidf"].values == 1, C["tfidf_score"].values, np.nan).astype(np.float32)
        tm = np.isnan(tcos)
        if tm.any():
            tcos[tm] = _pair_cos(ctx.tfidf, i1[tm], i2[tm])
        f["tfidf_cos"] = tcos
    else:
        f["tfidf_cos"] = np.zeros(n, np.float32)
    for r in RETRIEVERS:
        f[f"{r}_score"] = C[f"{r}_score"].values.astype(np.float32)
        f[f"{r}_rank"] = np.minimum(C[f"{r}_rank"].values, RANK_SENTINEL).astype(np.float32)
        f[f"by_{r}"] = C[f"by_{r}"].values.astype(np.float32)
    f["n_retrievers"] = C["n_retrievers"].values.astype(np.float32)
    f["best_rank"] = C["best_rank"].values.astype(np.float32)
    f["mutual_top5"] = ((C["dense_rank"].values < 5) & (C["dense_rev_rank"].values < 5)).astype(np.float32)
    h1, h2 = sub1["house_no"].values[inv1], sub2["house_no"].values[inv2]
    f["house_agree"] = _soft_equal(h1, h2)
    f["house_present_s1"] = (h1 != "").astype(np.float32)
    f["house_present_cand"] = (h2 != "").astype(np.float32)
    f["postcode_agree"] = _soft_equal(sub1["postcode"].values[inv1], sub2["postcode"].values[inv2])
    f["addr_word_jaccard"] = _set_jaccard(sub1["addr_words"].values[inv1], sub2["addr_words"].values[inv2])
    f["s1_city_missing"] = (sub1["city_norm"].values[inv1] == "").astype(np.float32)
    f["cand_city_missing"] = (sub2["city_norm"].values[inv2] == "").astype(np.float32)
    f["cand_state_missing"] = (sub2["state_norm"].values[inv2] == "").astype(np.float32)
    f["cand_addr_missing"] = (sub2["addr_clean"].values[inv2] == "").astype(np.float32)
    f["cand_non_latin"] = sub2["non_latin"].values[inv2].astype(np.float32)
    g = pd.Series(i1)
    f["emb_rank_in_entity"] = within_group_rank(i1, qcos).astype(np.float32)
    f["emb_gap_to_best"] = (pd.Series(qcos).groupby(g).transform("max").values - qcos).astype(np.float32)
    nts = f["name_token_set"]
    f["name_gap_to_best"] = (pd.Series(nts).groupby(g).transform("max").values - nts).astype(np.float32)
    b = f["bm25_score"]
    bmax = pd.Series(b).groupby(g).transform("max").values
    f["bm25_rel_to_best"] = np.where(bmax > 0, b / np.maximum(bmax, 1e-9), 0.0).astype(np.float32)
    f["n_cands_entity"] = pd.Series(i1).groupby(g).transform("size").values.astype(np.float32)
    return pd.DataFrame({k: f[k] for k in FEATURE_COLS})


# ------------------------- parallel feature computation (fork: workers share FR and the memmaps) -------------------------
_FEAT_JOB = None


def _feature_worker(a, b, rows):
    global _RF_WORKERS
    _RF_WORKERS = 1
    C, ctx = _FEAT_JOB
    return a, pair_features(C.iloc[a:b], ctx, rows).to_numpy(np.float32)


def _private_gb(pid):
    try:
        with open(f"/proc/{pid}/smaps_rollup") as f:
            return sum(int(l.split()[1]) for l in f if l.startswith(("Private_Clean", "Private_Dirty", "Swap:"))) / 1e6
    except OSError:
        return 0.0


def feature_workers():
    """Worker count under RAM_BUDGET_GB: parent anonymous RSS + FEATURE_WORKER_GB per worker (measured and
    updated after every parallel run)."""
    anon = rss_now_gb()[1] or 0.0
    room = cfg.RAM_BUDGET_GB - anon - 3.0   # 3 GB headroom for the chunk's feature matrix and results
    return max(1, min(cfg.NUM_WORKERS, int(room // max(_WORKER_GB[0], 0.1))))


_WORKER_GB = [1.0]   # running estimate of one feature worker's private memory
FEATURE_CHUNK_PAIRS = 20_000   # pairs per worker task: fixed, so a worker's memory does not grow when fewer run


def cpu_temp_c():
    """Hottest CPU core (coretemp / k10temp), or None."""
    temps = []
    for t in glob.glob("/sys/class/hwmon/hwmon*/temp*_input"):
        try:
            if open(os.path.join(os.path.dirname(t), "name")).read().strip() in ("coretemp", "k10temp", "zenpower"):
                temps.append(int(open(t).read()) / 1000)
        except (OSError, ValueError):
            pass
    return max(temps) if temps else None


def _entity_cuts(s1, target):
    starts = np.r_[0, np.flatnonzero(np.diff(s1)) + 1]
    cuts, last = [0], 0
    for st_ in starts:
        if st_ - last >= target:
            cuts.append(int(st_))
            last = st_
    cuts.append(len(s1))
    return [(a, b) for a, b in zip(cuts[:-1], cuts[1:]) if b > a]


def features_in_chunks(C, ctx, label=""):
    """Entity-aligned chunks (C is sorted by s1_pos), computed in NUM_WORKERS forked processes. Forked workers
    inherit the frames and memmaps (nothing large is pickled) and run CPU code only (no CUDA in children)."""
    global _FEAT_JOB
    t0 = time.time()
    n_proc = min(feature_workers(), max(1, len(C) // 50_000))
    target = max(min(FEATURE_CHUNK_PAIRS, math.ceil(len(C) / max(4 * n_proc, 1))), 1)
    bounds = _entity_cuts(C["s1_pos"].values, target)
    X = np.empty((len(C), len(FEATURE_COLS)), np.float32)   # results land here directly (no parts + concat copy)
    worker_gb = 0.0
    if n_proc > 1 and len(bounds) > 1:
        # The parent slices each chunk's frame rows and sends them (a few MB); workers never index FR, so they do
        # not touch (and copy-on-write) the pages of its ~10 GB of Python strings. gc.freeze keeps the children's
        # garbage collector off the inherited objects. At most 2 chunks per worker are in flight.
        import multiprocessing as mp
        _FEAT_JOB = (C, ctx)
        gc.collect()
        gc.freeze()
        try:
            with mp.get_context("fork").Pool(n_proc) as pool:
                pending, it = [], iter(bounds)
                for a, b in it:
                    pending.append(pool.apply_async(_feature_worker, (a, b, _chunk_rows(C.iloc[a:b]))))
                    while len(pending) >= 2 * n_proc or (pending and pending[0].ready()):
                        a_, arr = pending.pop(0).get()
                        X[a_:a_ + len(arr)] = arr
                    if len(pending) == 2 * n_proc - 1:
                        worker_gb = max(worker_gb, max(_private_gb(p.pid) for p in pool._pool))
                for r in pending:
                    a_, arr = r.get()
                    X[a_:a_ + len(arr)] = arr
                worker_gb = max(worker_gb, max(_private_gb(p.pid) for p in pool._pool))
        finally:
            _FEAT_JOB = None
            gc.unfreeze()
        _WORKER_GB[0] = max(worker_gb, 0.3)
    else:
        for a, b in bounds:
            X[a:b] = pair_features(C.iloc[a:b], ctx).to_numpy(np.float32)
        _WORKER_GB[0] = min(_WORKER_GB[0], 1.0)   # re-measure next time instead of staying serial
    out = pd.DataFrame(X, columns=FEATURE_COLS, copy=False)
    dt = time.time() - t0
    REPORT.setdefault("feature_seconds", {}).setdefault(ctx.name, 0.0)
    REPORT["feature_seconds"][ctx.name] += dt
    if label:
        log(f"  features {label}: {len(C):,} pairs in {dt:.1f}s ({len(C) / max(dt, 1e-9):,.0f} pairs/s, {n_proc} processes, "
            f"worker private peak {worker_gb:.2f} GB, parent RSS {rss_now_gb()[0]} GB, CPU {cpu_temp_c()} C)")
    return out


## 11. Labels, the competition metric, and decision rules (copied from `er_crossencoder_v2` §7–§8)
The metric is F0.5 per Source-1 entity. An entity with no true matches scores 1 for an empty
prediction and 0 otherwise. The final score is the macro average over **all** evaluated entities,
and recall's denominator counts matches that blocking missed. Decision rules are **searched on
out-of-fold predictions, never hardcoded**:
* **global** threshold;
* **relative** to the entity's best candidate;
* **exclusive**: each S2/S3 record goes to at most one S1. The data confirms this invariant, since
  no pool record is matched to more than one S1.
* **per-source**: separate S2 and S3 thresholds, new here, found by coordinate ascent.

**Selection prefers simplicity.** The complexity order is: global < +exclusive < per-source < relative.
The **simplest** rule whose CV macro F0.5 is within `RULE_SIMPLICITY_TOL` of the best one is chosen, so a
more complex rule is not kept for a gain smaller than the CV noise. The holdout reports the chosen rule
and, for comparison, the plain global threshold.

In [ ]:
def build_truth(gt_df, s1_df, pool_df):   # copied
    s1_index = pd.Index(s1_df["entity_id"])
    pool_index = pd.Index(pool_df["entity_id"])
    n_pool = np.int64(len(pool_df))
    rows = gt_df[gt_df["matched_entity_ids"] != ""]
    sid = np.repeat(rows["source1_entity_id"].values, rows["matched_entity_ids"].str.count(",").values + 1)
    cid = np.array([x.strip() for s in rows["matched_entity_ids"].values for x in s.split(",")], dtype=object)
    s1p = s1_index.get_indexer(sid)
    pp = pool_index.get_indexer(cid)
    ok = (s1p >= 0) & (pp >= 0)
    true_keys = np.unique(s1p[ok].astype(np.int64) * n_pool + pp[ok])
    n_true = np.bincount(s1p[ok], minlength=len(s1_df)).astype(np.int32)
    return true_keys, n_true


def isin_keys(s1_pos, pool_pos, n_pool, true_keys, chunk=20_000_000):
    """np.isin(s1_pos * n_pool + pool_pos, true_keys) in chunks (a single call on ~265M keys needs ~8 GB)."""
    out = np.empty(len(s1_pos), bool)
    tk = np.unique(true_keys)
    for a in range(0, len(s1_pos), chunk):
        k = np.asarray(s1_pos[a:a + chunk], np.int64) * np.int64(n_pool) + np.asarray(pool_pos[a:a + chunk], np.int64)
        out[a:a + chunk] = np.isin(k, tk, assume_unique=False)
    return out


def macro_f05(s1_pos_pred, tp_pred, n_true, eval_mask):   # copied
    """s1_pos_pred: entity positions of PREDICTED pairs; tp_pred: whether each predicted pair is
    true; n_true: true-match count per entity (all entities); eval_mask: entities being scored."""
    n = len(n_true)
    n_pred = np.bincount(s1_pos_pred, minlength=n).astype(np.float64)
    n_tp = np.bincount(s1_pos_pred, weights=tp_pred.astype(np.float64), minlength=n)
    t = n_true.astype(np.float64)
    with np.errstate(divide="ignore", invalid="ignore"):
        prec = np.where(n_pred > 0, n_tp / n_pred, 0.0)
        rec = np.where(t > 0, n_tp / t, 0.0)
        f = np.where(n_tp > 0, 1.25 * prec * rec / (0.25 * prec + rec), 0.0)
    f = np.where(t == 0, (n_pred == 0).astype(np.float64), f)
    sel = eval_mask
    return float(f[sel].mean()) if sel.any() else float("nan")


def blocking_ceiling(cands_s1_pos, y, n_true, eval_mask):   # copied
    """Pair recall of the candidate set, and the macro F_0.5 an oracle matcher would get."""
    in_eval = eval_mask[cands_s1_pos]
    found = y[in_eval].sum()
    total = n_true[eval_mask].sum()
    oracle = macro_f05(cands_s1_pos[in_eval & y], np.ones(int((in_eval & y).sum()), bool), n_true, eval_mask)
    return (found / total if total else 1.0), oracle


GLOBAL_THRESHOLD_GRID = np.round(np.concatenate([np.arange(0.10, 0.90, 0.02), np.arange(0.90, 0.996, 0.005)]), 3)
ADAPTIVE_PROB_FLOOR_GRID = np.round(np.arange(0.20, 0.96, 0.05), 3)
ADAPTIVE_GAP_MULT_GRID = [1.5, 2.0, 3.0]


class _RuleCache(dict):
    """Per-prediction-set quantities reused by every rule in a grid search, computed on first use with O(n)
    scatter-max (no groupby / lexsort over the whole set: those need ~8 GB at 265M pairs). Same values as the
    copied version: ent_max = max p of the pair's S1; pool_is_max = the pair is its pool record's best, ties to
    the lowest index (= rank 0 of within_group_rank's stable lexsort)."""
    def __init__(self, s1_pos, pool_pos, p):
        super().__init__()
        self.s1, self.pool, self.p = s1_pos, pool_pos, p

    def __missing__(self, key):
        if key == "ent_max":
            m = np.full(int(self.s1.max()) + 1 if len(self.s1) else 0, -np.inf, self.p.dtype)
            np.maximum.at(m, self.s1, self.p)
            v = m[self.s1]
        elif key == "pool_is_max":
            m = np.full(int(self.pool.max()) + 1 if len(self.pool) else 0, -np.inf, self.p.dtype)
            np.maximum.at(m, self.pool, self.p)
            idx = np.flatnonzero(self.p == m[self.pool])
            _, first = np.unique(self.pool[idx], return_index=True)
            v = np.zeros(len(self.p), bool)
            v[idx[first]] = True
        else:
            raise KeyError(key)
        self[key] = v
        return v


def rule_cache(s1_pos, pool_pos, p):
    return _RuleCache(s1_pos, pool_pos, p)


def apply_rule(s1_pos, pool_pos, p, rule, cache=None):   # copied
    """Returns a boolean mask over pairs. rule: dict(kind, thr | floor+gap_mult, exclusive)."""
    cache = cache if cache is not None else rule_cache(s1_pos, pool_pos, p)
    if rule["kind"] == "global":
        keep = p >= rule["thr"]
        if rule.get("exclusive"):
            # thresholding is monotone in p, so a pool record's best kept pair is its best pair overall
            return keep & cache["pool_is_max"]
        return keep
    keep = (p >= rule["floor"]) & (p * rule["gap_mult"] >= cache["ent_max"])
    if rule.get("exclusive"):
        idx = np.flatnonzero(keep)
        if len(idx):
            # among kept pairs, each pool record keeps only its highest-probability S1 entity
            order = idx[np.lexsort((-p[idx], pool_pos[idx]))]
            first = np.r_[True, pool_pos[order][1:] != pool_pos[order][:-1]]
            keep = np.zeros_like(keep)
            keep[order[first]] = True
    return keep


def candidate_rules():   # copied
    rules = [dict(kind="global", thr=float(t)) for t in GLOBAL_THRESHOLD_GRID]
    rules += [dict(kind="relative", floor=float(fl), gap_mult=float(g))
              for fl in ADAPTIVE_PROB_FLOOR_GRID for g in ADAPTIVE_GAP_MULT_GRID]
    return rules + [dict(r, exclusive=True) for r in rules]


def search_rule(s1_pos, pool_pos, p, y, n_true, eval_mask, label=""):   # copied
    """Grid-search decision rules on (held-out) predictions; returns (best_rule, best_score, table)."""
    rows = []
    cache = rule_cache(s1_pos, pool_pos, p)
    for r in candidate_rules():
        keep = apply_rule(s1_pos, pool_pos, p, r, cache)
        rows.append((r, macro_f05(s1_pos[keep], y[keep], n_true, eval_mask)))
    rows.sort(key=lambda x: -x[1])
    best_rule, best = rows[0]
    best_global = max((s for r, s in rows if r["kind"] == "global" and not r.get("exclusive")), default=float("nan"))
    print(f"  [{label}] best rule {best_rule} -> F0.5={best:.4f}   (best plain global threshold: {best_global:.4f})")
    return best_rule, best, rows


def apply_rule_v3(s1_pos, pool_pos, is_s3, p, rule, cache=None):
    if rule["kind"] == "per_source":
        keep = p >= np.where(is_s3, rule["thr_s3"], rule["thr_s2"])
        if rule.get("exclusive"):   # a pool record has one source -> still monotone per record
            keep &= (cache if cache is not None else rule_cache(s1_pos, pool_pos, p))["pool_is_max"]
        return keep
    return apply_rule(s1_pos, pool_pos, p, rule, cache)


_RULE_COMPLEXITY = {"global": 0, "per_source": 2, "relative": 4}


def rule_complexity(rule):
    return _RULE_COMPLEXITY[rule["kind"]] + int(bool(rule.get("exclusive")))


def select_rule(rows, tol):
    """The simplest rule whose score is within `tol` of the best; ties broken by score."""
    best = max(s for _, s in rows)
    eligible = [(r, s) for r, s in rows if s >= best - tol]
    return min(eligible, key=lambda t: (rule_complexity(t[0]), -t[1]))


def search_rule_v3(s1_pos, pool_pos, is_s3, p, y, n_true, eval_mask, label=""):
    """Copied rule families plus per-source thresholds; the simplest rule within RULE_SIMPLICITY_TOL wins."""
    best_rule, best, rows = search_rule(s1_pos, pool_pos, p, y, n_true, eval_mask, label=label)
    cache = rule_cache(s1_pos, pool_pos, p)

    def _score(r):
        keep = apply_rule_v3(s1_pos, pool_pos, is_s3, p, r, cache)
        return macro_f05(s1_pos[keep], y[keep], n_true, eval_mask)

    for excl in (False, True):
        g = [(r, s) for r, s in rows if r["kind"] == "global" and bool(r.get("exclusive")) == excl]
        r0, s0 = max(g, key=lambda t: t[1])
        cur, cur_s = dict(kind="per_source", thr_s2=r0["thr"], thr_s3=r0["thr"], exclusive=excl), s0
        for _ in range(2):
            for key in ("thr_s2", "thr_s3"):
                for t in GLOBAL_THRESHOLD_GRID:
                    cand = dict(cur, **{key: float(t)})
                    sc = _score(cand)
                    if sc > cur_s + 1e-9:
                        cur, cur_s = cand, sc
        rows.append((cur, cur_s))
    rows.sort(key=lambda x: -x[1])
    chosen, chosen_s = select_rule(rows, cfg.RULE_SIMPLICITY_TOL)
    print(f"  [{label}] best-scoring rule {rows[0][0]} -> CV macro F0.5={rows[0][1]:.4f}\n"
          f"  [{label}] SELECTED (simplest within {cfg.RULE_SIMPLICITY_TOL}) {chosen} -> CV macro F0.5={chosen_s:.4f}")
    return chosen, chosen_s, rows


def eval_rule(s1_pos, pool_pos, is_s3, p, y, rule, n_true, eval_mask, label=""):
    """Competition macro F0.5 plus micro P/R/F1, TP/FP/FN and breakdowns (P/R/F1 reported, F0.5 optimized)."""
    keep = apply_rule_v3(s1_pos, pool_pos, is_s3, p, rule)
    in_eval = eval_mask[s1_pos]
    tp = int((keep & y & in_eval).sum())
    fp = int((keep & ~y & in_eval).sum())
    fn = int(n_true[eval_mask].sum()) - tp
    P = tp / max(tp + fp, 1)
    R = tp / max(tp + fn, 1)
    res = dict(label=label, rule=rule, entities=int(eval_mask.sum()),
               macro_f05=round(macro_f05(s1_pos[keep], y[keep], n_true, eval_mask), 5),
               micro_precision=round(P, 5), micro_recall=round(R, 5),
               micro_f1=round(2 * P * R / max(P + R, 1e-12), 5), micro_f05=round(1.25 * P * R / max(0.25 * P + R, 1e-12), 5),
               tp=tp, fp=fp, fn=fn, pred_matches_per_entity=round(float(keep[in_eval].sum()) / max(int(eval_mask.sum()), 1), 3),
               true_matches_per_entity=round(float(n_true[eval_mask].mean()), 3))
    country = FR.s1["country_norm"].values
    res["by_country_macro_f05"] = {str(c): round(macro_f05(s1_pos[keep], y[keep], n_true, eval_mask & (country == c)), 4)
                                   for c in sorted(set(country[eval_mask]))}
    nl = FR.pool["non_latin"].values[pool_pos] == 1
    for name, groups in [("by_source", (("S2", ~is_s3), ("S3", is_s3))),
                         ("by_cand_script", (("latin", ~nl), ("non-latin", nl)))]:   # boolean masks, not string arrays
        d = {}
        for gv, gm in groups:
            if not gm.any():
                continue
            m = in_eval & gm
            tpg, fpg = int((keep & y & m).sum()), int((keep & ~y & m).sum())
            pos_g = int((y & m).sum())
            d[gv] = dict(precision=round(tpg / max(tpg + fpg, 1), 4), recall_of_candidates=round(tpg / max(pos_g, 1), 4))
        res[name] = d
    print(f"  [{label}] macro F0.5={res['macro_f05']:.4f}  micro P={P:.4f} R={R:.4f} F1={res['micro_f1']:.4f}  "
          f"TP={tp:,} FP={fp:,} FN={fn:,}  pred/entity={res['pred_matches_per_entity']} (true {res['true_matches_per_entity']})")
    return res

## 12. Reranker: XGBoost (GPU) with leakage-safe entity folds (copied trainer from `er_crossencoder_v2` §9–§10)
**Which entities.** A keyed hash of the S1 id puts `GBDT_TRAIN_ENTITY_FRACTION` (20%, ~440k entities,
~35M candidate pairs) of train S1 into the **training set**. Every other train S1 (~80%) is an
**evaluation entity**: never trained on, never used to pick the rule, scored once by `train_eval`.

**Folds.** Inside the training set, folds are assigned per **Source-1 entity**, so all candidates of an
entity land in the same fold and each pool record's single true S1 is in one fold only.
`NESTED_HOLDOUT_FRAC` of the training entities are a nested holdout (scored once with the CV model).

**Training.** The quantized matrix is built once on the GPU. `GBDT_PARAM_GRID` is compared on a
`GBDT_GRID_SELECT_FRACTION` subset of the training entities (5-fold OOF macro F0.5); the winner is then run
with full 5-fold CV on all training entities. The decision rule is searched on those out-of-fold
predictions. The final model is refit on all training entities. OOF predictions are checkpointed per
parameter set, so a crash does not redo finished CV.

**F0.5 checkpoints** recorded here: grid-subset OOF, full OOF (with the chosen rule), nested holdout.


In [ ]:
import xgboost as xgb

_XGB_V2 = _vt(xgb.__version__) >= (2, 0, 0)


def xgb_params(params):   # copied (device handling extended for xgboost<2)
    p = dict(objective="binary:logistic", eval_metric="logloss", max_depth=params["max_depth"],
             learning_rate=params["learning_rate"], subsample=0.8, colsample_bytree=0.8,
             min_child_weight=5, reg_lambda=1.0, max_bin=256, seed=cfg.RANDOM_SEED, nthread=cfg.NUM_WORKERS)
    if _XGB_V2:
        p.update(tree_method="hist", device=("cuda" if DEVICE == "cuda" else "cpu"))
    else:
        p.update(tree_method="gpu_hist" if DEVICE == "cuda" else "hist")
    return p


def fit_weights(row_mask, y, seed):   # copied (globals -> args)
    """1.0 for rows the model trains on (all positives + sampled negatives), 0.0 otherwise."""
    w = row_mask.astype(np.float32)
    if cfg.GBDT_TRAIN_NEG_FRAC < 1.0:
        neg = np.flatnonzero(row_mask & ~y)
        drop = np.random.default_rng(seed).choice(neg, size=int(len(neg) * (1 - cfg.GBDT_TRAIN_NEG_FRAC)), replace=False)
        w[drop] = 0.0
    return w


def fit_gbdt(dmat, params, row_mask, y, seed):   # copied
    dmat.set_weight(fit_weights(row_mask, y, seed))
    return xgb.train(xgb_params(params), dmat, num_boost_round=params["n_estimators"])


def gbdt_predict(booster, X, chunk=5_000_000):   # copied
    out = np.empty(len(X), dtype=np.float32)
    for s in range(0, len(X), chunk):
        part = X.iloc[s:s + chunk].values if hasattr(X, "iloc") else X[s:s + chunk]
        out[s:s + chunk] = booster.inplace_predict(np.ascontiguousarray(part, dtype=np.float32))
    return out


def entity_folds():
    """Per S1 row: 0..N_FOLDS-1 = CV fold, -1 = nested holdout (both = TRAINING entities, a
    GBDT_TRAIN_ENTITY_FRACTION hash sample), -3 = train EVALUATION entity (never trained on), -2 = test row."""
    ids = ids_to_int(FR.s1["entity_id"].values)
    u = hash_u01(ids, SALT["fold"])
    h = cfg.NESTED_HOLDOUT_FRAC
    fold = np.where(u < h, -1, np.minimum(((u - h) / (1 - h) * cfg.N_FOLDS).astype(np.int64), cfg.N_FOLDS - 1))
    fold[hash_u01(ids, SALT["gbdt"]) >= cfg.GBDT_TRAIN_ENTITY_FRACTION] = -3
    fold[FR.n_tr_s1:] = -2
    return fold


MODEL_KEYS = ["NESTED_HOLDOUT_FRAC", "N_FOLDS", "GBDT_TRAIN_NEG_FRAC", "GBDT_PARAM_GRID", "RANDOM_SEED", "MAX_CANDIDATES",
              "RULE_SIMPLICITY_TOL", "GBDT_TRAIN_ENTITY_FRACTION", "GBDT_GRID_SELECT_FRACTION"]


def model_fp(ctx):
    return cfg_fp(MODEL_KEYS, extra=[ctx.fp, FEATURE_COLS, MODEL_CODE_VERSION])


def leakage_checks(s1p, pp, y, fold):
    """Label-leakage audit of a training set (entity-level split, exclusivity, evaluation entities, test rows)."""
    tr = np.arange(FR.n_tr_s1)
    txt = pd.Series(FR.s1["name_clean"].values[tr] + "|" + FR.s1["addr_clean"].values[tr])
    f_ = fold[tr]
    dup_groups = pd.DataFrame({"t": txt.values[f_ >= -1], "f": f_[f_ >= -1]}).groupby("t")["f"].nunique()
    pos_per_pool = np.bincount(pp[y], minlength=len(FR.pool))
    res = dict(
        s1_entities_split_across_folds=0,   # by construction: every row of an S1 entity gets that entity's fold
        s1_exact_text_duplicates_straddling_folds=int((dup_groups > 1).sum()),
        pool_records_positive_for_more_than_one_s1=int((pos_per_pool > 1).sum()),
        evaluation_entity_rows_in_training_set=int((fold[s1p] == -3).sum()),
        test_s1_rows_in_training_set=int((s1p >= FR.n_tr_s1).sum()))
    assert res["test_s1_rows_in_training_set"] == 0, "test Source-1 rows in a training set"
    assert res["evaluation_entity_rows_in_training_set"] == 0, "evaluation entities in the training set"
    assert res["pool_records_positive_for_more_than_one_s1"] == 0, "a pool record is labelled positive for two S1"
    log(f"  leakage checks: {res}")
    return res


def _cv_oof(dmat, params, row_fold, rows, y, rel):
    """Out-of-fold predictions over `rows` (a boolean row mask), checkpointed at `rel` (one .npy per fold)."""
    oof = np.full(len(y), np.nan, dtype=np.float32)
    p_ = locate(rel)
    if p_:
        return np.load(p_)
    for f in range(cfg.N_FOLDS):
        frel = rel.replace(".npy", f"_fold{f}.npy")
        va = rows & (row_fold == f)
        fp_ = locate(frel)
        if fp_:
            oof[va] = np.load(fp_)[va]
            continue
        tr = rows & (row_fold >= 0) & (row_fold != f)
        if not tr.any() or not va.any():
            continue
        t0 = time.time()
        booster = fit_gbdt(dmat, params, tr, y, cfg.RANDOM_SEED + f)
        oof[va] = booster.predict(dmat)[va]
        save_npy(frel, oof)   # checkpoint per fold
        log(f"    fold {f + 1}/{cfg.N_FOLDS}: {int(tr.sum()):,} train rows, {int(va.sum()):,} scored in {time.time() - t0:.0f}s")
        del booster
        gc.collect()
    save_npy(rel, oof)
    return oof


def _best_global(s1p, y, p, n_true, mask_ent, rows):
    return max(macro_f05(s1p[rows][p[rows] >= t], y[rows][p[rows] >= t], n_true, mask_ent) for t in GLOBAL_THRESHOLD_GRID)


class TrainBlocks:
    """The training feature matrix as its checkpointed parquet blocks (rows in block order = the order of y)."""
    def __init__(self, paths, n):
        self.paths, self.n = list(paths), int(n)

    def __len__(self):
        return self.n


class _BlockIter(xgb.DataIter):
    """Feeds TrainBlocks to xgb.QuantileDMatrix one block at a time (optionally only the rows in `mask`)."""
    def __init__(self, blocks, y, mask=None):
        self.blocks, self.y, self.mask, self.i, self.a = blocks, y, mask, 0, 0
        super().__init__(release_data=True)

    def next(self, input_data):
        if self.i >= len(self.blocks.paths):
            return 0
        t = pq.read_table(self.blocks.paths[self.i], columns=FEATURE_COLS)
        b = self.a + t.num_rows
        Xb = np.empty((t.num_rows, len(FEATURE_COLS)), np.float32)
        for j, c in enumerate(FEATURE_COLS):
            Xb[:, j] = t.column(c).to_numpy()
        yb = self.y[self.a:b].astype(np.float32)
        if self.mask is not None:
            m = self.mask[self.a:b]
            Xb, yb = Xb[m], yb[m]
        self.i, self.a = self.i + 1, b
        input_data(data=Xb, label=yb)
        return 1

    def reset(self):
        self.i, self.a = 0, 0


def _qdm(X, y, mask=None, ref=None):
    """QuantileDMatrix from an in-memory matrix or, for TrainBlocks, streamed block by block."""
    if isinstance(X, TrainBlocks):
        return xgb.QuantileDMatrix(_BlockIter(X, y, mask), max_bin=256, ref=ref)
    Xm, ym = (X, y) if mask is None else (X[mask], y[mask])
    return xgb.QuantileDMatrix(Xm, label=ym.astype(np.float32), max_bin=256, ref=ref)


def train_and_calibrate(name, ctx, C, X, y, n_true):
    """C: (s1_pos, pool_pos) of the training entities' candidates; X: float32 feature matrix (FEATURE_COLS)."""
    fold = entity_folds()
    s1p, pp = C["s1_pos"].values.astype(np.int64), C["pool_pos"].values.astype(np.int64)
    is_s3 = FR.pool_is_s3[pp]
    row_fold = fold[s1p]
    cv_rows, hold_rows = row_fold >= 0, row_fold == -1
    cv_mask, hold_mask = fold >= 0, fold == -1
    mfp = model_fp(ctx)
    mrel = f"models/{name}_{mfp}"
    leak = leakage_checks(s1p, pp, y, fold)
    rec_cv = blocking_ceiling(s1p, y, n_true, cv_mask)
    rec_ho = blocking_ceiling(s1p, y, n_true, hold_mask)
    log(f"[{name}] rows={len(y):,} positives={int(y.sum()):,}  CV entities={int(cv_mask.sum()):,} "
        f"(pair recall {rec_cv[0]:.4f}, oracle F0.5 {rec_cv[1]:.4f})  holdout entities={int(hold_mask.sum()):,} "
        f"(pair recall {rec_ho[0]:.4f}, oracle F0.5 {rec_ho[1]:.4f})")
    t_train = time.time()
    dmat = _qdm(X, y)
    mem_mark("XGBoost QuantileDMatrix built")
    # ---- parameter selection on a subset of the training entities ----
    grid = cfg.GBDT_PARAM_GRID
    grid_scores = []
    if len(grid) > 1:
        g_ent = hash_u01(ids_to_int(FR.s1["entity_id"].values), SALT["grid"]) < cfg.GBDT_GRID_SELECT_FRACTION
        g_rows = cv_rows & g_ent[s1p]
        g_mask = cv_mask & g_ent
        dsub = _qdm(X, y, g_rows)
        for gi, params in enumerate(grid):
            log(f"  grid {gi} {params} on {int(g_mask.sum()):,} entities / {int(g_rows.sum()):,} rows")
            oof_s = _cv_oof(dsub, params, row_fold[g_rows], np.ones(int(g_rows.sum()), bool), y[g_rows],
                            f"{mrel}/oof_grid_{gi}.npy")
            oof_full = np.full(len(y), np.nan, np.float32)
            oof_full[g_rows] = oof_s
            grid_scores.append(_best_global(s1p, y, oof_full, n_true, g_mask, g_rows))
            log(f"  grid {gi}: OOF macro F0.5 @ best global thr = {grid_scores[-1]:.4f}")
        del dsub
        gc.collect()
        gi = int(np.argmax(grid_scores))
        track_f05("2a train: grid-select OOF (subset, best global thr)", grid_scores[gi],
                  entities=int(g_mask.sum()), params=str(grid[gi]))
    else:
        gi = 0
    params = grid[gi]
    # ---- full 5-fold CV with the chosen parameters ----
    log(f"  CV with {params} on {int(cv_mask.sum()):,} entities / {int(cv_rows.sum()):,} rows")
    oof = _cv_oof(dmat, params, row_fold, cv_rows, y, f"{mrel}/oof_{gi}.npy")
    rule, cv_score, rule_rows = search_rule_v3(s1p[cv_rows], pp[cv_rows], is_s3[cv_rows], oof[cv_rows], y[cv_rows],
                                               n_true, cv_mask, label=f"{name} OOF")
    plain = max(((r, s) for r, s in rule_rows if r["kind"] == "global" and not r.get("exclusive")), key=lambda t: t[1])
    track_f05("2b train: OOF with the chosen rule (CV)", cv_score, entities=int(cv_mask.sum()), rule=str(rule))
    cpath = cache_path(f"{mrel}/cv_model.json")
    if os.path.exists(cpath):
        cv_model = xgb.Booster()
        cv_model.load_model(cpath)
    else:
        cv_model = fit_gbdt(dmat, params, cv_rows, y, cfg.RANDOM_SEED)
        cv_model.save_model(cpath + ".tmp.json")
        os.replace(cpath + ".tmp.json", cpath)
    t_pred = time.time()
    p_hold = cv_model.predict(dmat)[hold_rows]
    rerank_s = time.time() - t_pred
    hold = eval_rule(s1p[hold_rows], pp[hold_rows], is_s3[hold_rows], p_hold, y[hold_rows], rule, n_true, hold_mask,
                     label=f"{name} NESTED HOLDOUT")
    hold_plain = hold if plain[0] == rule else eval_rule(
        s1p[hold_rows], pp[hold_rows], is_s3[hold_rows], p_hold, y[hold_rows], plain[0], n_true, hold_mask,
        label=f"{name} holdout, plain global threshold (comparison only)")
    track_f05("2c train: nested holdout (CV model + rule)", hold["macro_f05"], entities=int(hold_mask.sum()),
              precision=hold["micro_precision"], recall=hold["micro_recall"])
    gain = cv_model.get_score(importance_type="gain")
    imp = pd.Series({FEATURE_COLS[int(k[1:])] if k.startswith("f") and k[1:].isdigit() else k: v for k, v in gain.items()})
    imp = (imp / imp.sum()).sort_values(ascending=False)
    print("  top feature importances (gain):\n" + imp.head(15).round(4).to_string())
    mpath = cache_path(f"{mrel}/final.json")
    if os.path.exists(mpath):
        final = xgb.Booster()
        final.load_model(mpath)
    else:
        final = fit_gbdt(dmat, params, cv_rows | hold_rows, y, cfg.RANDOM_SEED)
        final.save_model(mpath + ".tmp.json")
        os.replace(mpath + ".tmp.json", mpath)
    # out-of-sample predictions of every training row (OOF for CV rows, CV model for the holdout): train_eval
    # combines them with the evaluation entities' scores so exclusivity sees every competing S1
    oos = oof.copy()
    oos[hold_rows] = p_hold
    ctx_save_parquet(ctx, pd.DataFrame({"s1_pos": s1p.astype(np.int32), "pool_pos": pp.astype(np.int32),
                                        "p": oos.astype(np.float32)}), f"model_{mfp}", "train_entities_oos.parquet")
    meta = dict(name=name, params=params, rule=rule, cv_macro_f05=cv_score, holdout=hold, feature_cols=FEATURE_COLS,
                grid_scores=grid_scores, plain_global_rule=plain[0], plain_global_cv_macro_f05=plain[1],
                plain_global_holdout_macro_f05=hold_plain["macro_f05"], leakage_checks=leak,
                ctx_fp=ctx.fp, model_fp=mfp, blocking_cv=rec_cv, blocking_holdout=rec_ho,
                train_seconds=round(time.time() - t_train, 1), holdout_rerank_seconds=round(rerank_s, 2),
                holdout_pairs=int(hold_rows.sum()), training_pairs=int(len(y)), top_features=imp.head(25).round(5).to_dict())
    save_json(f"{mrel}/meta.json", meta)
    save_json(f"models/latest_{name}.json", dict(rel=mrel))
    save_json(f"reports/model_{name}.json", meta)
    del dmat
    gc.collect()
    return SimpleNamespace(booster=final, meta=meta)


def load_model(name, which="final"):
    """which='final' = refit on all train entities (used for test); 'cv' = CV folds only (scores the holdout)."""
    p = locate(f"models/latest_{name}.json")
    if p is None:
        return None
    with open(p) as f:
        mrel = json.load(f)["rel"]
    mp, jp = locate(f"{mrel}/{'final' if which == 'final' else 'cv_model'}.json"), locate(f"{mrel}/meta.json")
    if mp is None or jp is None:
        return None
    b = xgb.Booster()
    b.load_model(mp)
    with open(jp) as f:
        meta = json.load(f)
    if meta["feature_cols"] != FEATURE_COLS:
        raise RuntimeError(f"model {mrel} was trained with a different feature schema")
    return SimpleNamespace(booster=b, meta=meta)

## 13. Stage runners
Every stage is **checkpointed**: retrieval per (retriever, country, source) partition, the training set,
the train evaluation and the test inference per S1 block of `QUERY_BLOCK_ROWS`, CV per fold. A crash, a
kernel restart or a `MAX_SESSION_HOURS` stop loses at most the block in progress; running the notebook
again resumes from the last checkpoint. Progress lines (with ETAs) go to `output/progress.log`, stage
timings to `output/run_log.jsonl`, and every **macro F0.5 checkpoint** to `reports/f05_progress.json`:

| step | what the F0.5 means |
|---|---|
| `1a` retrieval ceiling (diag) | oracle F0.5 of the configured union on the 1% diagnostic queries vs the full pool (perfect reranker) |
| `1b` retrieval ceiling (all train) | the same over all 2.2M train S1 |
| `2a` grid-select OOF | out-of-fold F0.5 of the best parameter set on a subset of the training entities |
| `2b` OOF with the chosen rule | 5-fold CV F0.5 on the training entities (this picks the decision rule) |
| `2c` nested holdout | training entities held out of CV, scored once |
| `3` train evaluation | final model + rule on the ~80% of train entities never trained on (overall and per country) |

Stages:
* **`sample`**: country invariant on the full ground truth; builds the train frame (100% of train by default).
* **`embed_check`**: true pairs vs random pairs for the name, address and combined vectors.
* **`weight_sweep`**: dense Recall@K and dense-only oracle F0.5 per `w` on the diagnostic queries vs the full
  pool; `EMB_NAME_WEIGHT="auto"` takes the best `w`.
* **`diag_recall`**: all retrievers at diagnostic depth on the diagnostic queries vs the full train pool: the
  recall / cap / reverse-K / leave-one-out tables used to choose the top-Ks and `MAX_CANDIDATES`.
* **`train_model`**: production retrieval for all train S1; full-train union recall and oracle F0.5; the
  training set (training entities only, block by block); XGBoost CV, rule search, nested holdout, final model.
* **`train_eval`**: the final model + rule on every other train entity, block by block, with a running
  per-block F0.5; then the rule is applied to all train S1 at once (out-of-sample scores everywhere) and the
  F0.5 is reported on the evaluation entities, per country. Ends with the **GATE** (`reports/gate1.json`).
* **`test_inference`**: refuses to start unless the gate passed. 100% test in S1 blocks: union → features →
  score, each block checkpointed; the rule is then applied globally. Per-country summary and the France
  audit files.
* **`submit`**: writes and strictly validates `matching_results.tsv` / `candidate_pairs.tsv`.


In [ ]:
def sample_truth():
    gt = load_parquet(locate(f"{SAMPLE_REL}/gt.parquet"))
    return build_truth(gt, FR.s1, FR.pool)


def skip(stage_name, reason):
    print(f"\n[{stage_name}] SKIPPED: {reason}")
    REPORT.setdefault("skipped", {})[stage_name] = reason


def _read_report(name):
    p = locate(f"reports/{name}.json")
    if p is None:
        return None
    with open(p) as f:
        return json.load(f)


# ------------------------- weight sweep: w * cos(name) + (1 - w) * cos(addr) -------------------------
def _weight_sweep_rel():
    return f"reports/weight_sweep_{FP_SAMPLE}_{cfg_fp(['EMB_WEIGHT_SWEEP', 'DIAG_QUERY_FRACTION', 'RECALL_KS', 'DENSE_TOP_K'], extra=sorted(EMB_INFO.items()))}.json"


def resolve_emb_weight():
    """EMB_NAME_WEIGHT, or with "auto" the best w of the weight sweep (run once, then cached)."""
    if cfg.EMB_NAME_WEIGHT != "auto":
        return float(cfg.EMB_NAME_WEIGHT)
    p = locate(_weight_sweep_rel())
    if p is None:
        stage_weight_sweep()
        p = locate(_weight_sweep_rel())
    with open(p) as f:
        w = float(json.load(f)["best_w"])
    cfg.EMB_NAME_WEIGHT = w
    CFG["EMB_NAME_WEIGHT"] = w
    log(f"EMB_NAME_WEIGHT auto -> {w} (from the weight sweep)")
    return w


def stage_weight_sweep():
    """Dense forward Recall@K and dense-only oracle F0.5 on the diagnostic queries vs the FULL train pool, per w."""
    rel = _weight_sweep_rel()
    if locate(rel):
        with open(locate(rel)) as f:
            REPORT["weight_sweep"] = json.load(f)
        print("weight sweep (cached):\n" + pd.DataFrame(REPORT["weight_sweep"]["table"]).to_string(index=False)
              + f"\n  best w = {REPORT['weight_sweep']['best_w']}")
        return REPORT["weight_sweep"]
    with stage("weight_sweep") as st:
        ensure_frames("train")
        q = np.flatnonzero(diag_query_mask())
        ctx_like = SimpleNamespace(q=q, pool=np.arange(len(FR.pool)), rev=np.arange(len(FR.s1)))
        true_keys, n_true = sample_truth()
        n_pool = np.int64(len(FR.pool))
        eval_mask = np.zeros(len(FR.s1), bool)
        eval_mask[q] = True
        total = int(n_true[eval_mask].sum())
        kmax = max(cfg.RECALL_KS)
        rows = []
        for w in cfg.EMB_WEIGHT_SWEEP:
            t0 = time.time()
            v1 = EmbView([CombinedStore("train", "s1", w)], EMB_DIMS["name"] + EMB_DIMS["addr"])
            v2 = EmbView([CombinedStore("train", "pool", w)], v1.dim)
            parts = [_dense_fwd((v1, v2), qr, pr, kmax) for _, _, qr, pr, _ in partitions(ctx_like) if len(qr) and len(pr)]
            d_ = pd.concat(parts, ignore_index=True)
            s1 = d_["s1_pos"].values
            t = np.isin(s1 * n_pool + d_["pool_pos"].values, true_keys)
            pos = group_positions(s1, d_["rank"].values, d_["score"].values)
            row = dict(w=w, **{f"R@{K}": round(float(t[pos < K].sum()) / max(total, 1), 4) for K in cfg.RECALL_KS})
            m = d_["rank"].values < cfg.DENSE_TOP_K
            row[f"oracle_F05@top{cfg.DENSE_TOP_K}/src"] = round(blocking_ceiling(s1[m], t[m], n_true, eval_mask)[1], 4)
            row["seconds"] = round(time.time() - t0, 1)
            rows.append(row)
            log(f"  w={w:<5} " + "  ".join(f"R@{K}={row[f'R@{K}']:.4f}" for K in cfg.RECALL_KS)
                + f"  oracle F0.5={row[f'oracle_F05@top{cfg.DENSE_TOP_K}/src']:.4f}  ({row['seconds']}s)")
            del d_, parts
            gc.collect()
        tab = pd.DataFrame(rows)
        best = tab.sort_values(["R@100", "R@50"], ascending=False).iloc[0]
        res = dict(table=rows, best_w=float(best["w"]), queries=int(len(q)), true_pairs=total)
        print(f"\nweight sweep on {len(q):,} diagnostic queries ({total:,} true pairs) vs the full train pool:\n"
              + tab.to_string(index=False) + f"\n  best w (max dense R@100) = {res['best_w']}")
        REPORT["weight_sweep"] = res
        save_json(rel, res)
        st.add(best_w=res["best_w"])
    return res


# ------------------------- diagnostic recall: choose the top-Ks and MAX_CANDIDATES -------------------------
def stage_diag_recall():
    resolve_emb_weight()
    ctx = make_ctx("diag")
    rel = f"reports/recall_diag_{ctx.fp}_{cfg.MAX_CANDIDATES}.json"
    if locate(rel):
        with open(locate(rel)) as f:
            rec = json.load(f)
        log(f"diagnostic recall (cached): union recall@{cfg.MAX_CANDIDATES}={rec['union_recall_at_cap']}  "
            f"oracle F0.5={rec['oracle_macro_f05']}")
    else:
        with stage("diag_recall"):
            run_retrievers(ctx)
            retr = load_retr(ctx)
            true_keys, n_true = sample_truth()
            eval_mask = np.zeros(len(FR.s1), bool)
            eval_mask[ctx.q] = True
            rec = recall_report(retr, len(FR.pool), true_keys, n_true, eval_mask,
                                f"DIAGNOSTIC: {len(ctx.q):,} train S1 vs the FULL train pool")
            del retr
            gc.collect()
            save_json(rel, rec)
    REPORT["recall_diag"] = rec
    track_f05("1a retrieval ceiling: oracle @ configured union (diag queries, full pool)", rec["oracle_macro_f05"],
              union_recall=rec["union_recall_at_cap"], max_candidates=cfg.MAX_CANDIDATES)
    caps = pd.DataFrame(rec["cap_table"])
    deep = caps[caps["union"] == "diagnostic depth"]
    ok = deep[(deep["recall"] >= 0.995) & (deep["oracle_macro_f05"] >= 0.995)]
    print(f"\nchoosing K: configured union @ MAX_CANDIDATES={cfg.MAX_CANDIDATES}: recall={rec['union_recall_at_cap']:.4f} "
          f"oracle F0.5={rec['oracle_macro_f05']:.4f}")
    print("  smallest cap on the diagnostic-depth union with recall >= 0.995 and oracle F0.5 >= 0.995: "
          + (f"{int(ok['cap'].min())}" if len(ok) else "none within the measured caps -- retrieval is the limit"))
    print(f"  leave-one-out recall drop per retriever: {rec['leave_one_out_recall_drop']}")
    return rec


# ------------------------- blocked union over a production context -------------------------
def _union_blocks(ctx):
    B = cfg.QUERY_BLOCK_ROWS
    q = ctx.q
    return [(int(q[a]), int(q[min(a + B, len(q)) - 1]) + 1) for a in range(0, len(q), B)]


def _block_union(ctx, lo, hi):
    return build_union(truncate_retr(load_retr(ctx, lo, hi)), len(FR.pool), cap=cfg.MAX_CANDIDATES)


def full_train_recall(ctx, true_keys, n_true):
    """Configured-union recall and oracle F0.5 over ALL train S1 (production depth, full pool), per country.
    Per-block counts are checkpointed."""
    rel = f"reports/recall_train_full_{ctx.fp}_{cfg.MAX_CANDIDATES}.json"
    if locate(rel):
        with open(locate(rel)) as f:
            return json.load(f)
    with stage("recall_train_full"):
        n_pool = np.int64(len(FR.pool))
        found = np.zeros(len(FR.s1), np.int32)
        cands = np.zeros(len(FR.s1), np.int32)
        xc = 0
        blocks = _union_blocks(ctx)
        for bi, (lo, hi) in enumerate(blocks):
            brel = ctx_rel(ctx, "recall_blocks", f"b{bi:05d}_{cfg.MAX_CANDIDATES}.npy")
            if locate(brel):
                arr = np.load(locate(brel))
            else:
                C = _block_union(ctx, lo, hi)
                s1 = C["s1_pos"].values
                y = np.isin(s1 * n_pool + C["pool_pos"].values, true_keys)
                xc_b = cross_country_count(s1, C["pool_pos"].values)
                arr = np.stack([np.bincount(s1[y] - lo, minlength=hi - lo), np.bincount(s1 - lo, minlength=hi - lo),
                                np.full(hi - lo, xc_b)]).astype(np.int32)
                save_npy(brel, arr)
            found[lo:hi], cands[lo:hi] = arr[0], arr[1]
            xc += int(arr[2][0]) if hi > lo else 0
            if (bi + 1) % 5 == 0 or bi + 1 == len(blocks):
                log(f"  full-train recall: block {bi + 1}/{len(blocks)}  running recall="
                    f"{found[:hi].sum() / max(n_true[:hi].sum(), 1):.4f}")
        if xc and not cfg.ENABLE_GLOBAL_FALLBACK:
            raise RuntimeError(f"country partition violated: {xc} cross-country candidates")
        s1_pos_found = np.repeat(np.arange(len(found)), found)
        allm = np.ones(len(FR.s1), bool)
        oracle = macro_f05(s1_pos_found, np.ones(len(s1_pos_found), bool), n_true, allm)
        country = FR.s1["country_norm"].values
        by_c = {}
        for c in sorted(set(country)):
            m = country == c
            by_c[str(c)] = dict(entities=int(m.sum()), recall=round(float(found[m].sum() / max(n_true[m].sum(), 1)), 4),
                                oracle_macro_f05=round(macro_f05(s1_pos_found, np.ones(len(s1_pos_found), bool), n_true, m), 4))
        rec = dict(entities=int(len(found)), true_pairs=int(n_true.sum()), union_recall_at_cap=round(float(found.sum() / max(n_true.sum(), 1)), 4),
                   oracle_macro_f05=round(oracle, 4), max_candidates=cfg.MAX_CANDIDATES,
                   cands_per_query=dict(mean=round(float(cands.mean()), 2), p50=float(np.percentile(cands, 50)),
                                        p95=float(np.percentile(cands, 95)), zero_frac=round(float((cands == 0).mean()), 4)),
                   entity_all_found_rate=round(float((found[n_true > 0] >= n_true[n_true > 0]).mean()), 4),
                   cross_country_candidates=xc, by_country=by_c, retr_cfg_fp=retr_cfg_fp())
        save_json(rel, rec)
    print(f"\n==== configured union over ALL {rec['entities']:,} train S1 (full pool) ====\n"
          f"  recall@{cfg.MAX_CANDIDATES}={rec['union_recall_at_cap']:.4f}  oracle F0.5={rec['oracle_macro_f05']:.4f}  "
          f"candidates/query={rec['cands_per_query']}\n  by country: {rec['by_country']}")
    return rec


def _trainset_dir():
    return f"trainset_{cfg_fp(['MAX_CANDIDATES', 'GBDT_TRAIN_ENTITY_FRACTION', 'NESTED_HOLDOUT_FRAC', 'N_FOLDS'], extra=[FEATURE_COLS])}"


def build_training_set(ctx, true_keys, fold):
    """Candidates + features + labels of the TRAINING entities, block by block (each block checkpointed),
    then assembled into one float32 matrix."""
    tdir = _trainset_dir()
    n_pool = np.int64(len(FR.pool))
    train_ent = fold >= -1
    blocks = _union_blocks(ctx)
    t0, n_new = time.time(), 0
    for bi, (lo, hi) in enumerate(blocks):
        if is_done(ctx_rel(ctx, tdir, f"block_{bi:05d}.parquet")):
            continue
        if hours_left() <= 0:
            raise RuntimeError("MAX_SESSION_HOURS reached while building the training set -- finished blocks are saved; run again")
        C = _block_union(ctx, lo, hi)
        C = C[train_ent[C["s1_pos"].values]].reset_index(drop=True)
        feats = features_in_chunks(C, ctx, label=f"training block {bi + 1}/{len(blocks)}")
        y = np.isin(C["s1_pos"].values * n_pool + C["pool_pos"].values, true_keys)
        out = pd.concat([C[["s1_pos", "pool_pos"]].astype(np.int32).reset_index(drop=True), feats], axis=1)
        out["label"] = y.astype(np.int8)
        ctx_save_parquet(ctx, out, tdir, f"block_{bi:05d}.parquet")
        n_new += 1
        el = time.time() - t0
        left = sum(1 for b in range(bi + 1, len(blocks)) if not is_done(ctx_rel(ctx, tdir, f"block_{b:05d}.parquet")))
        log(f"  training set block {bi + 1}/{len(blocks)}: {len(C):,} pairs ({int(y.sum()):,} positives)  "
            f"ETA {_fmt_duration(el / n_new * left)}")
        del C, feats, out
        gc.collect()
    paths = [locate(ctx_rel(ctx, tdir, f"block_{bi:05d}.parquet")) for bi in range(len(blocks))]
    n = sum(int(read_done(ctx_rel(ctx, tdir, f"block_{bi:05d}.parquet"))["rows"]) for bi in range(len(blocks)))
    y = np.empty(n, bool)
    s1p = np.empty(n, np.int64)
    pp = np.empty(n, np.int64)
    a = 0
    for p_ in paths:   # ids and labels only; the feature matrix is streamed from these blocks into XGBoost
        t = pq.read_table(p_, columns=["s1_pos", "pool_pos", "label"])
        b = a + t.num_rows
        y[a:b] = t.column("label").to_numpy().astype(bool)
        s1p[a:b] = t.column("s1_pos").to_numpy()
        pp[a:b] = t.column("pool_pos").to_numpy()
        a = b
        del t
    X = TrainBlocks(paths, n)
    log(f"training set: {n:,} pairs, {int(y.sum()):,} positives, {len(np.unique(s1p)):,} entities, "
        f"features {n * len(FEATURE_COLS) * 4 / 1e9:.1f} GB as float32 (streamed from {len(paths)} blocks, never assembled)")
    mem_mark("training set ids + labels", [y, s1p, pp], rows=n)
    return pd.DataFrame({"s1_pos": s1p, "pool_pos": pp}), X, y


def _existing_model_for(name, ctx):
    m = load_model(name)
    if m is not None and m.meta.get("ctx_fp") == ctx.fp and m.meta.get("model_fp") == model_fp(ctx):
        return m
    return None


def stage_train_model():
    resolve_emb_weight()
    ctx = make_ctx("train")
    done = _existing_model_for("train", ctx)
    if done is not None:
        log("[train_model] model already trained for this configuration -- reusing "
            "(delete models/latest_train.json to retrain)")
        rec = _read_report("recall_train_full") or {}
        return rec, done
    with stage("train_model"):
        run_retrievers(ctx)
        true_keys, n_true = sample_truth()
        rec = full_train_recall(ctx, true_keys, n_true)
        save_json("reports/recall_train_full.json", rec)
        REPORT["recall_train_full"] = rec
        track_f05("1b retrieval ceiling: oracle @ configured union (ALL train S1)", rec["oracle_macro_f05"],
                  union_recall=rec["union_recall_at_cap"])
        C, X, y = build_training_set(ctx, true_keys, entity_folds())
        res = train_and_calibrate("train", ctx, C, X, y, n_true)
        del C, X, y
        gc.collect()
    REPORT["model_train"] = res.meta
    return rec, res


# ------------------------- train evaluation: the other train entities, never trained on -------------------------
def stage_train_eval():
    resolve_emb_weight()
    ctx = make_ctx("train")
    model = _existing_model_for("train", ctx)
    if model is None:
        return skip("train_eval", "no trained model for the current configuration -- run train_model first")
    mfp = model.meta["model_fp"]
    rule = model.meta["rule"]
    rel = f"reports/train_eval_{mfp}.json"
    if locate(rel):
        with open(locate(rel)) as f:
            res = json.load(f)
        log(f"train evaluation (cached): macro F0.5={res['macro_f05']}")
        REPORT["train_eval"] = res
        evaluate_gate1(res)
        return res
    with stage("train_eval") as st:
        true_keys, n_true = sample_truth()
        n_pool = np.int64(len(FR.pool))
        fold = entity_folds()
        ev = fold == -3
        edir = f"eval_{mfp}"
        blocks = _union_blocks(ctx)
        t0, n_new = time.time(), 0
        for bi, (lo, hi) in enumerate(blocks):
            if is_done(ctx_rel(ctx, edir, f"block_{bi:05d}.parquet")):
                continue
            if hours_left() <= 0:
                raise RuntimeError("MAX_SESSION_HOURS reached during the train evaluation -- finished blocks are saved; run again")
            C = _block_union(ctx, lo, hi)
            C = C[ev[C["s1_pos"].values]].reset_index(drop=True)
            feats = features_in_chunks(C, ctx, label=f"eval block {bi + 1}/{len(blocks)}")
            p = gbdt_predict(model.booster, feats[FEATURE_COLS])
            s1p, pp = C["s1_pos"].values.astype(np.int64), C["pool_pos"].values.astype(np.int64)
            y = isin_keys(s1p, pp, n_pool, true_keys)
            out = pd.DataFrame({"s1_pos": s1p.astype(np.int32), "pool_pos": pp.astype(np.int32), "p": p.astype(np.float32),
                                "y": y.astype(np.int8)})
            ctx_save_parquet(ctx, out, edir, f"block_{bi:05d}.parquet")   # checkpoint per block
            n_new += 1
            # running estimate: rule applied inside this block only (exclusivity across blocks comes at the end)
            m_blk = np.zeros(len(FR.s1), bool)
            m_blk[lo:hi] = True
            m_blk &= ev
            keep = apply_rule_v3(s1p, pp, FR.pool_is_s3[pp], p, rule)
            f_blk = macro_f05(s1p[keep], y[keep], n_true, m_blk)
            left = sum(1 for b in range(bi + 1, len(blocks)) if not is_done(ctx_rel(ctx, edir, f"block_{b:05d}.parquet")))
            log(f"  train-eval block {bi + 1}/{len(blocks)}: {len(C):,} pairs  block macro F0.5={f_blk:.4f}  "
                f"ETA {_fmt_duration((time.time() - t0) / n_new * left)}")
            del C, feats, out
            gc.collect()
        ev_preds = pd.concat([ctx_read_parquet(ctx, edir, f"block_{bi:05d}.parquet") for bi in range(len(blocks))],
                             ignore_index=True)
        oos = ctx_read_parquet(ctx, f"model_{mfp}", "train_entities_oos.parquet")
        s1p = np.r_[ev_preds["s1_pos"].values, oos["s1_pos"].values].astype(np.int64)
        pp = np.r_[ev_preds["pool_pos"].values, oos["pool_pos"].values].astype(np.int64)
        p = np.r_[ev_preds["p"].values, oos["p"].values].astype(np.float32)
        y = isin_keys(s1p, pp, n_pool, true_keys)
        del ev_preds, oos
        gc.collect()
        # the rule is applied to ALL train S1 at once (out-of-sample scores everywhere), so exclusivity sees every
        # competing S1; the score is reported on the evaluation entities (never trained on / used for the rule)
        res = eval_rule(s1p, pp, FR.pool_is_s3[pp], p, y, rule, n_true, ev,
                        label=f"TRAIN EVALUATION ({int(ev.sum()):,} entities never trained on, full pool)")
        keep = apply_rule_v3(s1p, pp, FR.pool_is_s3[pp], p, rule)
        res["entities_all_train_macro_f05"] = round(macro_f05(s1p[keep], y[keep], n_true, np.ones(len(FR.s1), bool)), 5)
        save_json(rel, res)
        st.add(macro_f05=res["macro_f05"])
    REPORT["train_eval"] = res
    track_f05("3 train evaluation: final model + rule (80% of train, never trained on)", res["macro_f05"],
              precision=res["micro_precision"], recall=res["micro_recall"], entities=res["entities"])
    for c, v in res["by_country_macro_f05"].items():
        track_f05(f"3 train evaluation: {c}", v)
    evaluate_gate1(res)
    return res


def evaluate_gate1(eval_res):
    """GATE: may the test phase start? Written to reports/gate1.json and checked by test_inference."""
    reasons = []
    rec = _read_report("recall_train_full")
    if rec is None:
        reasons.append("full-train recall not computed (run train_model)")
    else:
        if rec["union_recall_at_cap"] < cfg.GATE1_MIN_UNION_RECALL:
            reasons.append(f"configured-union recall {rec['union_recall_at_cap']:.4f} < GATE1_MIN_UNION_RECALL "
                           f"{cfg.GATE1_MIN_UNION_RECALL}")
        if rec.get("cross_country_candidates", 0) and not cfg.ENABLE_GLOBAL_FALLBACK:
            reasons.append(f"{rec['cross_country_candidates']} cross-country candidates")
    inv = REPORT.get("country_invariant") or _read_report("country_invariant")
    if inv is None:
        reasons.append("country invariant not checked (run the sample stage)")
    elif inv["cross_country_pairs"] > cfg.COUNTRY_INVARIANT_MAX_VIOLATIONS:
        reasons.append(f"country invariant violated ({inv['cross_country_pairs']} pairs)")
    if not np.isfinite(eval_res["macro_f05"]):
        reasons.append("train-evaluation macro F0.5 is not finite")
    g = dict(passed=not reasons, reasons=reasons, retr_cfg_fp=retr_cfg_fp(),
             union_recall_at_cap=rec["union_recall_at_cap"] if rec else None,
             oracle_macro_f05=rec["oracle_macro_f05"] if rec else None,
             train_eval_macro_f05=eval_res["macro_f05"], by_country=eval_res.get("by_country_macro_f05"), t=time.time())
    save_json("reports/gate1.json", g)
    log(f"==== GATE: {'PASSED' if g['passed'] else 'FAILED ' + str(reasons)} ====  union recall={g['union_recall_at_cap']}  "
        f"oracle F0.5={g['oracle_macro_f05']}  train-eval F0.5={g['train_eval_macro_f05']}  by country={g['by_country']}")
    return g


def gate1_status():
    g = _read_report("gate1")
    if g is None:
        return False, "the gate has not been evaluated (run train_model and train_eval)"
    if g.get("retr_cfg_fp") != retr_cfg_fp():
        return False, "the gate was evaluated with different retrieval settings -- run train_eval again"
    if not g["passed"]:
        return False, f"gate failed: {g['reasons']}"
    return True, "passed"


def scale_phase_allowed(stage_name, quiet=False):
    if cfg.ENABLE_SCALE_PHASE:
        return True
    if not quiet:
        skip(stage_name, "ENABLE_SCALE_PHASE=False")
    return False


def _final_model():
    m = load_model("train")
    if m is None:
        raise RuntimeError("no trained model found -- run train_model first")
    return "train", m


def _train_countries():
    inv = REPORT.get("country_invariant") or _read_report("country_invariant")
    if inv:
        return {c for k, d in inv["records_per_country"].items() if k.startswith("train") for c in d}
    return set(FR.s1["country_norm"].values[:FR.n_tr_s1])


def _test_country_breakdown(s1_local, p, keep, n_test):
    """Per test country: candidates, predicted matches, top-1 probability distribution (no labels needed)."""
    country = FR.s1["country_norm"].values[FR.n_tr_s1:]
    counts = np.bincount(s1_local, minlength=n_test)
    mcounts = np.bincount(s1_local[keep], minlength=n_test)
    pmax = np.zeros(n_test, np.float32)
    np.maximum.at(pmax, s1_local, p.astype(np.float32))
    seen = _train_countries()
    out = {}
    for c in sorted(set(country)):
        m = country == c
        out[str(c)] = dict(s1=int(m.sum()), seen_in_train=c in seen, cands_mean=round(float(counts[m].mean()), 2),
                           zero_cand_frac=round(float((counts[m] == 0).mean()), 4),
                           matches_per_entity=round(float(mcounts[m].mean()), 3),
                           entities_with_match=round(float((mcounts[m] > 0).mean()), 4),
                           top1_p_q10=round(float(np.quantile(pmax[m], 0.1)), 4),
                           top1_p_q50=round(float(np.quantile(pmax[m], 0.5)), 4),
                           top1_p_q90=round(float(np.quantile(pmax[m], 0.9)), 4))
    print("  per test country (no labels -- compare unseen countries with the seen ones):")
    print(pd.DataFrame(out).T.to_string())
    return out


def _raw_by_id(split, s, ids, cols):
    """Raw columns for the given entity ids of one source file (reads only those rows)."""
    _, ids_int = read_ids(split, s)
    df = read_source(split, s, ["entity_id"] + cols, np.isin(ids_int, ids_to_int(np.asarray(ids, dtype=object))))
    return df.set_index("entity_id")


def export_country_audit(preds, keep):
    """Hand-labelling files for test countries unseen in training (France): a hash sample of AUDIT_N_ENTITIES
    entities with their top AUDIT_TOP_N candidates (+ any predicted match), raw text side by side, and an empty
    is_match column. Fill it with 1/0, attach the file and set AUDIT_LABELS_PATH to get a precision estimate."""
    te_country = FR.s1["country_norm"].values[FR.n_tr_s1:]
    countries = cfg.AUDIT_COUNTRIES if cfg.AUDIT_COUNTRIES is not None else sorted(set(te_country) - _train_countries())
    if not countries or cfg.AUDIT_N_ENTITIES <= 0:
        return {}
    s1p = preds["s1_pos"].values.astype(np.int64)
    out_dir = os.path.join(OUTPUT_DIR, "audit")
    os.makedirs(out_dir, exist_ok=True)
    written = {}
    raw_cols = ["business_name", "business_address"]
    for c in countries:
        rows_c = FR.n_tr_s1 + np.flatnonzero(te_country == c)
        if not len(rows_c):
            continue
        ids_c = FR.s1["entity_id"].values[rows_c]
        pick = rows_c[np.argsort(hash_u01(ids_to_int(ids_c), SALT["audit"]), kind="stable")[:cfg.AUDIT_N_ENTITIES]]
        m = np.isin(s1p, pick)
        sub = pd.DataFrame({"s1_pos": s1p[m], "pool_pos": preds["pool_pos"].values[m].astype(np.int64),
                            "p": preds["p"].values[m], "predicted_match": keep[m].astype(np.int8)})
        sub["cand_rank"] = within_group_rank(sub["s1_pos"].values, sub["p"].values).astype(np.int32)
        sub = sub[(sub["cand_rank"] < cfg.AUDIT_TOP_N) | (sub["predicted_match"] == 1)].copy()
        sub["s1_entity_id"] = FR.s1["entity_id"].values[sub["s1_pos"].values]
        sub["cand_entity_id"] = FR.pool["entity_id"].values[sub["pool_pos"].values]
        sub["cand_source"] = FR.pool["pool_source"].values[sub["pool_pos"].values]
        r1 = _raw_by_id("test", 1, sub["s1_entity_id"].unique(), raw_cols)
        rp = pd.concat([_raw_by_id("test", s, sub.loc[sub["cand_source"] == f"S{s}", "cand_entity_id"].unique(), raw_cols)
                        for s in (2, 3)])
        for col in raw_cols:
            sub[f"s1_{col}"] = r1[col].reindex(sub["s1_entity_id"].values).values
            sub[f"cand_{col}"] = rp[col].reindex(sub["cand_entity_id"].values).values
        sub["is_match"] = ""
        cols = (["s1_entity_id"] + [f"s1_{c_}" for c_ in raw_cols] + ["cand_rank", "cand_entity_id", "cand_source"]
                + [f"cand_{c_}" for c_ in raw_cols] + ["p", "predicted_match", "is_match"])
        sub = sub.sort_values(["s1_pos", "cand_rank"])[cols]
        path = os.path.join(out_dir, f"audit_{_slug(c)}.tsv")
        sub.to_csv(path, sep="\t", index=False)
        no_cand = int(len(pick) - len(np.unique(s1p[m])))
        written[str(c)] = dict(path=path, entities=int(len(pick)), rows=int(len(sub)), entities_without_candidates=no_cand)
        print(f"  audit file for {c}: {path} ({len(pick)} entities, {len(sub)} rows; {no_cand} entities had no candidate)")
    return written


def score_audit(path):
    """Precision of predicted matches (and matches the rule missed) from a hand-labelled audit file."""
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    lab = df[df["is_match"].str.strip().isin(["0", "1"])]
    if not len(lab):
        return None
    y, pm = (lab["is_match"].str.strip() == "1").values, (lab["predicted_match"] == "1").values
    res = dict(path=path, labelled_rows=int(len(lab)), labelled_entities=int(lab["s1_entity_id"].nunique()),
               predicted_matches=int(pm.sum()), precision=round(float(y[pm].mean()), 4) if pm.any() else None,
               true_matches_listed=int(y.sum()), true_matches_missed_by_rule=int((y & ~pm).sum()),
               recall_within_listed_candidates=round(float((y & pm).sum() / max(int(y.sum()), 1)), 4))
    print(f"  audit {os.path.basename(path)}: {res}")
    return res


def stage_test_inference():
    if not scale_phase_allowed("test_inference"):
        return
    resolve_emb_weight()
    if load_model("train") is None:
        return skip("test_inference", "no trained model yet")
    if cfg.TEST_FRACTION >= 1.0 and not cfg.FORCE_PAST_GATE:
        ok, why = gate1_status()
        if not ok:
            raise RuntimeError(f"test inference refused: {why}. Fix it, or set FORCE_PAST_GATE=True")
    with stage("test_inference") as st:
        mname, model = _final_model()
        rule = model.meta["rule"]
        print(f"final model: {mname}  rule={rule}  holdout macro F0.5={model.meta['holdout']['macro_f05']}")
        ctx = make_ctx("test")
        sdir = f"scores_{model.meta['model_fp']}"
        q = ctx.q
        B = cfg.QUERY_BLOCK_ROWS
        blocks = [(int(q[a]), int(q[min(a + B, len(q)) - 1]) + 1) for a in range(0, len(q), B)]
        t_feat = t_pred = 0.0
        if is_done(ctx_rel(ctx, sdir, "final_preds.parquet")):
            log("test predictions for this model already exist -- reusing (summary and audit files are rebuilt)")
        else:
            run_retrievers(ctx)
            t0, n_new = time.time(), 0
            n_cached = sum(1 for bi in range(len(blocks)) if is_done(ctx_rel(ctx, sdir, f"block_{bi:05d}.parquet")))
            log(f"test inference: {n_cached}/{len(blocks)} blocks already checkpointed")
            for bi, (lo, hi) in enumerate(blocks):
                if is_done(ctx_rel(ctx, sdir, f"block_{bi:05d}.parquet")):
                    continue
                if hours_left() <= 0:
                    raise RuntimeError("MAX_SESSION_HOURS reached during test inference -- completed blocks are saved; "
                                       "attach this output and run again to continue")
                C = build_union(truncate_retr(load_retr(ctx, lo, hi)), len(FR.pool), cap=cfg.MAX_CANDIDATES)
                xc = cross_country_count(C["s1_pos"].values, C["pool_pos"].values)
                if xc and not cfg.ENABLE_GLOBAL_FALLBACK:
                    raise RuntimeError(f"country partition violated in test block {bi}: {xc} cross-country candidates")
                ta = time.time()
                feats = features_in_chunks(C, ctx, label=f"test block {bi + 1}/{len(blocks)}")
                tb = time.time()
                p = gbdt_predict(model.booster, feats[FEATURE_COLS])
                tc = time.time()
                t_feat += tb - ta
                t_pred += tc - tb
                out = pd.DataFrame({"s1_pos": C["s1_pos"].values.astype(np.int32), "pool_pos": C["pool_pos"].values.astype(np.int32),
                                    "p": p.astype(np.float32), "pos_in_entity": C["pos_in_entity"].values.astype(np.int16)})
                ctx_save_parquet(ctx, out, sdir, f"block_{bi:05d}.parquet")
                n_new += 1
                eta = (time.time() - t0) / n_new * (len(blocks) - n_cached - n_new)
                log(f"  test block {bi + 1}/{len(blocks)}: {len(C):,} pairs  features {tb - ta:.0f}s  score {tc - tb:.1f}s  "
                    f"ETA {_fmt_duration(eta)}")
            preds = pd.concat([ctx_read_parquet(ctx, sdir, f"block_{bi:05d}.parquet") for bi in range(len(blocks))],
                              ignore_index=True)
            mem_mark("test predictions (all blocks)", preds)
            s1p, pp = preds["s1_pos"].values.astype(np.int64), preds["pool_pos"].values.astype(np.int64)
            preds["match"] = apply_rule_v3(s1p, pp, FR.pool_is_s3[pp], preds["p"].values, rule)
            ctx_save_parquet(ctx, preds, sdir, "final_preds.parquet")
            save_json(f"reports/test_timing_{model.meta['model_fp']}.json",
                      dict(feature_seconds=round(t_feat, 1), rerank_seconds=round(t_pred, 1)))
        preds = ctx_read_parquet(ctx, sdir, "final_preds.parquet")
        s1p, pp = preds["s1_pos"].values.astype(np.int64), preds["pool_pos"].values.astype(np.int64)
        keep = preds["match"].values.astype(bool)
        xc = cross_country_count(s1p, pp)
        if xc and not cfg.ENABLE_GLOBAL_FALLBACK:
            raise RuntimeError(f"country partition violated: {xc} cross-country test candidates")
        n_test = len(FR.s1) - FR.n_tr_s1
        s1_local = s1p - FR.n_tr_s1
        counts = np.bincount(s1_local, minlength=n_test)
        mcounts = np.bincount(s1_local[keep], minlength=n_test)
        timing = _read_report(f"test_timing_{model.meta['model_fp']}") or {}
        info = dict(model=mname, rule=rule, test_s1=n_test, test_pool=len(ctx.pool), candidate_pairs=len(preds),
                    predicted_matches=int(keep.sum()), cross_country_candidates=xc,
                    cands_per_query=dict(mean=round(float(counts.mean()), 2), p50=float(np.percentile(counts, 50)),
                                         p95=float(np.percentile(counts, 95)), zero_frac=round(float((counts == 0).mean()), 4)),
                    matches_per_entity=round(float(mcounts.mean()), 3), entities_with_match=round(float((mcounts > 0).mean()), 4),
                    feature_seconds=timing.get("feature_seconds"), rerank_seconds=timing.get("rerank_seconds"),
                    preds_path=ctx_rel(ctx, sdir, "final_preds.parquet"))
        print(f"test: {info}")
        print("  (train ground truth averages ~3.46 matches/entity with ~5.6% singletons -- a large deviation is worth "
              "investigating before submitting)")
        info["by_country"] = _test_country_breakdown(s1_local, preds["p"].values, keep, n_test)
        info["audit_files"] = export_country_audit(preds, keep)
        REPORT["test_inference"] = info
        save_json("reports/test_inference.json", info)
        st.add(**{k: v for k, v in info.items() if isinstance(v, (int, float, str))})

## 14. Submission: `matching_results.tsv` + `candidate_pairs.tsv`, with strict validation that fails loudly
There is exactly one row per test Source-1 entity, **in the original `test_source1` order**. Id lists
are ordered by descending model probability. The validator applies the rules of the copied
`validate_output`:
* exact header, 2 fields per row;
* no duplicate, missing or extra S1 ids;
* only existing S2-/S3- ids, with no duplicates inside a list.

It is re-implemented as a streaming check, because keeping ~170M candidate ids in Python sets would
not fit in RAM. It also checks:
* the exact row count, compared with the parquet metadata;
* row order;
* no `nan` / `None` tokens;
* that listed ids are **test** ids, never train ids;
* matches ⊆ candidates;
* each pool id matched at most once when the exclusive rule is used.

The official `validate_submission.py` also runs if it is found. Any failure **raises**.
SHA-256 hashes go into `run_report.json`. When `TEST_FRACTION < 1` the files go to
`dryrun_NOT_SUBMITTABLE/`.

In [ ]:
class SubmissionValidationError(RuntimeError):
    pass


def build_id_lists(s1_pos, ids, score, n_s1):   # copied from er_crossencoder_v2 §15
    order = np.lexsort((-score, s1_pos))
    s_sorted, ids_sorted = s1_pos[order], ids[order]
    bounds = np.searchsorted(s_sorted, np.arange(n_s1 + 1))
    return [",".join(ids_sorted[bounds[i]:bounds[i + 1]]) for i in range(n_s1)]


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()


def validate_submission_strict(match_path, cand_path, expected_ids, valid_pool_ids, exclusive, expected_rows):
    errors = []

    def err(msg):
        if len(errors) < 50:
            errors.append(msg)

    seen_matched = set()
    with open(match_path, newline="", encoding="utf-8") as fm, open(cand_path, newline="", encoding="utf-8") as fc:
        rm, rc = csv.reader(fm, delimiter="\t"), csv.reader(fc, delimiter="\t")
        hm, hc = next(rm, None), next(rc, None)
        if hm != ["source1_entity_id", "matched_entity_ids"]:
            err(f"matching_results: bad header {hm}")
        if hc != ["source1_entity_id", "candidate_entity_ids"]:
            err(f"candidate_pairs: bad header {hc}")
        n = 0
        seen_s1 = set()
        for n, (row_m, row_c) in enumerate(itertools.zip_longest(rm, rc), start=1):
            ln = n + 1
            if row_m is None or row_c is None:
                err(f"line {ln}: the two files have different row counts")
                break
            if len(row_m) != 2 or len(row_c) != 2:
                err(f"line {ln}: expected 2 tab-separated fields (got {len(row_m)} / {len(row_c)})")
                continue
            (sid, mlist), (sid_c, clist) = row_m, row_c
            if n > len(expected_ids) or sid != expected_ids[n - 1] or sid_c != expected_ids[n - 1]:
                err(f"line {ln}: source1 id order mismatch ({sid} / {sid_c} vs expected "
                    f"{expected_ids[n - 1] if n <= len(expected_ids) else 'EOF'})")
            if sid in seen_s1:
                err(f"line {ln}: duplicate source1_entity_id {sid}")
            seen_s1.add(sid)
            m_ids = mlist.split(",") if mlist else []
            c_ids = clist.split(",") if clist else []
            c_set = set(c_ids)
            for name, lst, st_ in (("matched", m_ids, set(m_ids)), ("candidate", c_ids, c_set)):
                if len(lst) != len(st_):
                    err(f"line {ln}: duplicate ids within {name} list of {sid}")
                bad = [x for x in lst if x in ("nan", "None", "NaN", "") or not (x.startswith("S2-") or x.startswith("S3-"))
                       or x not in valid_pool_ids]
                if bad:
                    err(f"line {ln}: {len(bad)} invalid {name} ids for {sid}, e.g. {bad[:3]}")
            if not set(m_ids) <= c_set:
                err(f"line {ln}: matches not a subset of candidates for {sid}")
            if exclusive:
                for x in m_ids:
                    if x in seen_matched:
                        err(f"line {ln}: pool id {x} matched to more than one source1 entity under an exclusive rule")
                    seen_matched.add(x)
    if n != expected_rows or n != len(expected_ids):
        err(f"row count {n:,} != expected {expected_rows:,}")
    missing = set(expected_ids) - seen_s1
    if missing:
        err(f"{len(missing):,} test source1 ids missing, e.g. {sorted(missing)[:3]}")
    if errors:
        raise SubmissionValidationError("submission validation FAILED:\n  - " + "\n  - ".join(errors))
    return dict(rows=n, matching_sha256=_sha256(match_path), candidate_sha256=_sha256(cand_path))


def _run_official_validator(match_path, cand_path):
    cands = [p for p in ["utils/validate_submission.py", "../utils/validate_submission.py"] if os.path.exists(p)]
    cands += [os.path.join(LOCAL_ROOT, p_) for p_ in ("utils/validate_submission.py", "validate_submission.py")
              if os.path.exists(os.path.join(LOCAL_ROOT, p_))]
    test_dirs = [d for d in ["dataset/test", "../dataset/test"] if os.path.isdir(d)]
    test_dirs += [os.path.join(DATA_DIR, "test")] if os.path.isdir(os.path.join(DATA_DIR, "test")) else []
    if not cands or not test_dirs or cfg.TEST_FRACTION < 1.0:
        print("official validator not found (or dry run) -- the strict in-notebook checks above apply the same rules")
        return None
    r = subprocess.run([sys.executable, cands[0], "--matching", match_path, "--candidate", cand_path,
                        "--test-dir", test_dirs[0]], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:])
    if r.returncode != 0:
        raise SubmissionValidationError(f"official validator failed (exit {r.returncode})")
    return r.returncode


def stage_submit():
    if not scale_phase_allowed("submit"):
        return
    resolve_emb_weight()
    if load_model("train") is None:
        return skip("submit", "no trained model yet")
    with stage("submit") as st:
        mname, model = _final_model()
        ctx = make_ctx("test")
        sdir = f"scores_{model.meta['model_fp']}"
        if not is_done(ctx_rel(ctx, sdir, "final_preds.parquet")):
            return skip("submit", "no test predictions for the final model yet -- run test_inference")
        preds = ctx_read_parquet(ctx, sdir, "final_preds.parquet")
        n_test = len(FR.s1) - FR.n_tr_s1
        s1_local = preds["s1_pos"].values - FR.n_tr_s1
        assert s1_local.min() >= 0 and s1_local.max() < n_test, "prediction rows outside the test S1 range"
        assert (preds["pool_pos"].values >= FR.n_tr_pool).all(), "a train record leaked into test predictions"
        pool_ids = FR.pool["entity_id"].values
        pool_pos = preds["pool_pos"].values
        p, keep = preds["p"].values, preds["match"].values.astype(bool)
        mem_mark("submit: final test predictions", preds)
        test_ids = FR.s1["entity_id"].values[FR.n_tr_s1:]
        expected_rows = pq.ParquetFile(SRC_FILE[("test", 1)]).metadata.num_rows if cfg.TEST_FRACTION >= 1.0 else n_test
        if cfg.TEST_FRACTION >= 1.0:
            file_order = pq.read_table(SRC_FILE[("test", 1)], columns=["entity_id"]).column(0).to_pylist()
            if file_order != list(test_ids):
                raise SubmissionValidationError("test S1 frame order differs from test_source1_final.parquet order")
        out_dir = os.path.join(OUTPUT_DIR, "submission_1" if cfg.TEST_FRACTION >= 1.0 else "dryrun_NOT_SUBMITTABLE")
        os.makedirs(out_dir, exist_ok=True)
        match_path = os.path.join(out_dir, "matching_results.tsv")
        cand_path = os.path.join(out_dir, "candidate_pairs.tsv")
        # written in S1 chunks (same bytes as one to_csv call): only one chunk's id lists are in memory at a time
        if not (np.diff(s1_local) >= 0).all():
            o = np.argsort(s1_local, kind="stable")
            s1_local, pool_pos, p, keep = s1_local[o], pool_pos[o], p[o], keep[o]
        B = cfg.QUERY_BLOCK_ROWS
        bounds = np.searchsorted(s1_local, np.arange(0, n_test + B, B).clip(max=n_test))
        for bi, a in enumerate(range(0, n_test, B)):
            b = min(a + B, n_test)
            r0, r1 = bounds[bi], bounds[bi + 1]
            s_, ids_, p_, k_ = s1_local[r0:r1] - a, pool_ids[pool_pos[r0:r1]], p[r0:r1], keep[r0:r1]
            mode, header = ("w", True) if a == 0 else ("a", False)
            pd.DataFrame({"source1_entity_id": test_ids[a:b], "matched_entity_ids": build_id_lists(s_[k_], ids_[k_], p_[k_], b - a)}
                         ).to_csv(match_path + ".tmp", sep="\t", index=False, quoting=csv.QUOTE_NONE, mode=mode, header=header)
            pd.DataFrame({"source1_entity_id": test_ids[a:b], "candidate_entity_ids": build_id_lists(s_, ids_, p_, b - a)}
                         ).to_csv(cand_path + ".tmp", sep="\t", index=False, quoting=csv.QUOTE_NONE, mode=mode, header=header)
            if bi == 0:
                mem_mark(f"submit: first TSV chunk written ({b - a:,} S1)")
        os.replace(match_path + ".tmp", match_path)
        os.replace(cand_path + ".tmp", cand_path)
        gc.collect()
        valid_pool = set(FR.pool["entity_id"].values[FR.n_tr_pool:])
        res = validate_submission_strict(match_path, cand_path, list(test_ids), valid_pool,
                                         bool(model.meta["rule"].get("exclusive")), expected_rows)
        _run_official_validator(match_path, cand_path)
        res.update(matching_path=match_path, candidate_path=cand_path, submittable=cfg.TEST_FRACTION >= 1.0,
                   config_fp=cfg_fp(sorted(CFG)), model=mname)
        log(f"VALIDATION PASSED: {res}")
        if cfg.TEST_FRACTION < 1.0:
            print("NOTE: TEST_FRACTION < 1 -- these files are a dry run and are NOT a valid submission.")
        REPORT["submission"] = res
        save_json("reports/submission.json", res)
        st.add(rows=res["rows"])

## 15. Run report
The report gathers every metric requested for Submission 1 from this session and from earlier
sessions (via their saved `reports/*.json`). It prints the metrics and writes
`OUTPUT_DIR/run_report.json`. Values that no session has produced yet are shown as `None`.

In [ ]:
def _load_reports():
    out = {}
    for base in CACHE_SEARCH_PATHS[::-1] + [CACHE_DIR]:   # later sessions override earlier ones
        for p in glob.glob(os.path.join(base, "reports", "*.json")):
            try:
                with open(p) as f:
                    out[os.path.basename(p)[:-5]] = json.load(f)
            except (OSError, ValueError):
                pass
    return out


def _all_stage_logs():
    recs = []
    paths = [RUN_LOG_PATH]
    for p in dict.fromkeys(paths):
        if os.path.exists(p):
            with open(p) as f:
                recs += [json.loads(l) for l in f if l.strip()]
    return recs


def stage_report():
    reps = _load_reports()
    logs = _all_stage_logs()
    samp = reps.get(f"sample_{FP_SAMPLE}", {})
    rd, rf = reps.get("recall_diag") or next((v for k, v in reps.items() if k.startswith("recall_diag_")), {}), \
        reps.get("recall_train_full", {})
    mt, te = reps.get("model_train", {}), next((v for k, v in reps.items() if k.startswith("train_eval_")), {})
    ti, sub = reps.get("test_inference", {}), reps.get("submission", {})
    audits = {}
    if cfg.AUDIT_LABELS_PATH:
        for pth in sorted(glob.glob(cfg.AUDIT_LABELS_PATH, recursive=True)):
            audits[pth] = score_audit(pth)

    def _retr_secs(ctx_name, keys):
        tot = 0.0
        for k, v in reps.items():
            if k.startswith(f"retrieval_seconds_{ctx_name}_"):
                tot += sum(float(v.get(x, 0)) for x in keys)
        return round(tot, 1)

    def _recall_row(rep, name):
        for row in rep.get("table", []):
            if row["retriever"] == name:
                return {k: v for k, v in row.items() if k.startswith("R@")}
        return None

    stage_secs = defaultdict(float)
    for r in logs:
        stage_secs[r["stage"]] += float(r.get("seconds", 0))
    report = {
        "profile": PROFILE,
        "f05_progress": reps.get("f05_progress"),
        "country_invariant": {k: (reps.get("country_invariant") or {}).get(k) for k in
                              ("positive_pairs_checked", "cross_country_pairs", "pairs_with_empty_country",
                               "test_countries_unseen_in_train")},
        "gate": reps.get("gate1"),
        "records": dict(train_s1=samp.get("n_s1"), train_pool=samp.get("n_pool"), train_true_pairs=samp.get("n_true_pairs"),
                        test_s1=ti.get("test_s1"), test_pool=ti.get("test_pool")),
        "embeddings": dict(dims=EMB_DIMS, name_weight=cfg.EMB_NAME_WEIGHT, weight_sweep=reps.get("weight_sweep") or next(
            (v for k, v in reps.items() if k.startswith("weight_sweep_")), None), sanity=next(
            (v for k, v in reps.items() if k.startswith("embedding_sanity_")), None)),
        "diagnostic_recall": dict(per_retriever={rn: _recall_row(rd, rn) for rn in RETRIEVERS + ["exact (any key)",
                                  "UNION (diagnostic depth)", "UNION (configured top-Ks)"] if _recall_row(rd, rn)},
                                  union_recall_at_cap=rd.get("union_recall_at_cap"), oracle_macro_f05=rd.get("oracle_macro_f05"),
                                  cap_table=rd.get("cap_table"), reverse_k_table=rd.get("reverse_k_table"),
                                  leave_one_out_recall_drop=rd.get("leave_one_out_recall_drop")),
        "full_train_recall": rf or None,
        "model": dict(params=mt.get("params"), rule=mt.get("rule"), grid_scores=mt.get("grid_scores"),
                      cv_macro_f05=mt.get("cv_macro_f05"), training_pairs=mt.get("training_pairs"),
                      holdout={kk: (mt.get("holdout") or {}).get(kk) for kk in
                               ("macro_f05", "micro_precision", "micro_recall", "micro_f1", "tp", "fp", "fn")},
                      plain_global_threshold_holdout_f05=mt.get("plain_global_holdout_macro_f05"),
                      leakage_checks=mt.get("leakage_checks"), top_features=mt.get("top_features")),
        "train_evaluation": {kk: te.get(kk) for kk in ("entities", "macro_f05", "micro_precision", "micro_recall", "micro_f1",
                                                       "tp", "fp", "fn", "by_country_macro_f05", "by_source",
                                                       "entities_all_train_macro_f05")} if te else None,
        "test": {kk: ti.get(kk) for kk in ("rule", "candidate_pairs", "predicted_matches", "cands_per_query",
                                          "matches_per_entity", "entities_with_match", "by_country", "audit_files",
                                          "cross_country_candidates")} if ti else None,
        "audit_scores": audits or None,
        "seconds": dict(dense_retrieval=_retr_secs("train", ["dense", "dense_rev", "dense_global"]) + _retr_secs("test", ["dense", "dense_rev", "dense_global"]),
                        bm25=_retr_secs("train", ["bm25"]) + _retr_secs("test", ["bm25"]),
                        tfidf=_retr_secs("train", ["tfidf"]) + _retr_secs("test", ["tfidf"]),
                        reranker_train=mt.get("train_seconds"), test_features=ti.get("feature_seconds"),
                        test_rerank=ti.get("rerank_seconds"), per_stage={k: round(v, 1) for k, v in stage_secs.items()}),
        "total_runtime_seconds_all_sessions": round(sum(
            max(r.get("t_session", 0) for r in logs if r.get("session") == sid) for sid in {r.get("session") for r in logs}), 1),
        "sessions": len({r.get("session") for r in logs}),
        "this_session_seconds": round(time.time() - T0, 1),
        "peak_gpu_memory_gb": max([r["gpu_peak_gb"] for r in logs if r.get("gpu_peak_gb") is not None], default=None),
        "peak_ram_gb": max([r["peak_rss_gb"] for r in logs if r.get("peak_rss_gb") is not None], default=None),
        "disk_usage_gb": dict(cache=dir_size_gb(CACHE_DIR), scratch=dir_size_gb(SCRATCH_DIR), output=dir_size_gb(OUTPUT_DIR)),
        "submission": sub or None,
    }
    path = os.path.join(OUTPUT_DIR, "run_report.json")
    with open(path, "w") as f:
        json.dump(report, f, indent=1, default=str)
    print(json.dumps(report, indent=1, default=str))
    show_f05_progress()
    log(f"run report written to {path}")
    return report


## 16. Self-tests (seconds; run first)
Each test checks one piece of logic against a known answer:
* `macro_f05` competition edge cases;
* BM25 against a brute-force implementation, and **GPU BM25 == CPU BM25** on a random partition;
* candidate-union dedupe, provenance and cap;
* sharded search matches unsharded search, and **multi-GPU sharded search** matches brute force;
* the TF-IDF vector store's resume after a simulated crash, and the int8 round trip;
* nested hash sampling;
* the strict validator rejecting crafted bad files;
* reverse retrieval orientation, and `order_rank` after truncation;
* the candidate cap;
* country buckets never mixing countries, and the cross-country counter;
* embedding-dimension consistency (over-reading and mismatched caches are refused);
* scale-phase gating;
* decision-rule simplicity;
* the combined vector's inner product = `w·cos(name) + (1-w)·cos(addr)`;
* the raw-field views (`name_clean` / `addr_clean` = lower-cased raw text, city/state empty).


In [ ]:
def run_self_tests():
    with stage("selftest"):
        tmp = os.path.join(SCRATCH_DIR, "selftest")
        shutil.rmtree(tmp, ignore_errors=True)
        os.makedirs(tmp)
        # 1. metric edge cases
        n_true = np.array([0, 0, 2, 1])
        s1p = np.array([1, 2])          # entity 0: empty/no-truth -> 1; 1: FP with no truth -> 0;
        tp = np.array([False, True])    # entity 2: 1 of 2 correct -> P=1,R=.5 -> 0.8333; entity 3: missed -> 0
        got = macro_f05(s1p, tp, n_true, np.ones(4, bool))
        assert abs(got - (1 + 0 + 1.25 * 0.5 / (0.25 + 0.5) + 0) / 4) < 1e-9, got
        # 2. BM25 vs brute force
        docs = pd.DataFrame({"name_core": ["alpha beta", "beta gamma", "alpha alpha delta", "omega"],
                             "addr_clean": ["", "", "", ""], "city_norm": ["x", "x", "y", "x"], "state_norm": ["", "", "", ""]})
        qs = pd.DataFrame({"name_core": ["alpha delta"], "addr_clean": [""], "city_norm": ["y"], "state_norm": [""]})
        old = (cfg.BM25_MIN_DF_CAP, cfg.BM25_MAX_DF_FRAC)
        cfg.BM25_MIN_DF_CAP, cfg.BM25_MAX_DF_FRAC = 10, 1.0
        qi, di, sc, rk = bm25_search(docs, np.arange(4), qs, np.arange(1), 4, fields=("n", "c"))
        toks = _bm25_tokens(docs, range(4), ("n", "c"))
        qt = _bm25_tokens(qs, range(1), ("n",))[0]
        qtc = _bm25_tokens(qs, range(1), ("c",))[0]
        N, avgdl = 4, np.mean([len(t) for t in toks])
        brute = []
        for d in toks:
            s_ = 0.0
            for t, w in [(t, cfg.BM25_FIELD_WEIGHTS["n"]) for t in set(qt)] + [(t, cfg.BM25_FIELD_WEIGHTS["c"]) for t in set(qtc)]:
                df_ = sum(t in x for x in toks)
                tf = d.count(t)
                if tf:
                    idf = math.log1p((N - df_ + 0.5) / (df_ + 0.5))
                    s_ += w * idf * tf * (cfg.BM25_K1 + 1) / (tf + cfg.BM25_K1 * (1 - cfg.BM25_B + cfg.BM25_B * len(d) / avgdl))
            brute.append(s_)
        cfg.BM25_MIN_DF_CAP, cfg.BM25_MAX_DF_FRAC = old
        got = dict(zip(di.tolist(), sc.tolist()))
        for j, b in enumerate(brute):
            assert abs(got.get(j, 0.0) - b) < 1e-4, (j, got, brute)
        assert di[np.argmax(sc)] == 2
        # 3. union dedupe / provenance / cap
        ra = pd.DataFrame({"s1_pos": [0, 0, 0, 1], "pool_pos": [5, 6, 7, 5], "score": [.9, .8, .7, .6], "rank": [0, 1, 2, 0]})
        rb = pd.DataFrame({"s1_pos": [0, 0], "pool_pos": [7, 8], "score": [3.0, 2.0], "rank": [0, 1]})
        U = build_union({"dense": ra, "bm25": rb}, 10, cap=3)
        u0 = U[U.s1_pos == 0].set_index("pool_pos")
        # best ranks: 5->0, 7->0 (bm25 rank 0, found twice), 6->1, 8->1; ties broken by n_retrievers then dense
        # score, so the cap of 3 keeps {7, 5, 6} and drops 8
        assert set(u0.index) == {5, 6, 7}, set(u0.index)
        assert u0.loc[7, "pos_in_entity"] == 0 and u0.loc[5, "pos_in_entity"] == 1
        assert u0.loc[7, "by_dense"] == 1 and u0.loc[7, "by_bm25"] == 1 and u0.loc[7, "n_retrievers"] == 2
        assert u0.loc[7, "dense_rank"] == 2 and u0.loc[7, "bm25_rank"] == 0 and u0.loc[5, "bm25_rank"] == RANK_SENTINEL
        assert len(U[U.s1_pos == 1]) == 1
        # 4. sharded == unsharded dense search
        rng = np.random.default_rng(0)
        A = rng.standard_normal((50, 16)).astype(np.float32)
        Bm = rng.standard_normal((300, 16)).astype(np.float32)
        A /= np.linalg.norm(A, axis=1, keepdims=True)
        Bm /= np.linalg.norm(Bm, axis=1, keepdims=True)

        class _Arr:
            def __init__(self, M):
                self.M, self.dim = M, M.shape[1]

            def gather(self, rows, out_dtype=np.float16):
                return self.M[rows].astype(out_dtype)

        old_shard = cfg.SEARCH_SHARD_ROWS
        cfg.SEARCH_SHARD_ROWS = 70
        s_sh, i_sh = dense_search(_Arr(A), np.arange(50), _Arr(Bm), np.arange(300), 10)
        cfg.SEARCH_SHARD_ROWS = old_shard
        ref = np.argsort(-(A @ Bm.T), axis=1)[:, :10]
        assert (np.sort(i_sh, 1) == np.sort(ref, 1)).mean() > 0.97, "sharded search disagrees with brute force"
        # 5. embedding store resume + int8 round trip
        ids = np.arange(10, 35, dtype=np.int64)
        V = rng.standard_normal((25, 8)).astype(np.float32)
        V /= np.linalg.norm(V, axis=1, keepdims=True)
        tag = f"selftest_{int(time.time() * 1000)}"
        stx = EmbStore(tag, ids, 8, "int8", 10)
        stx.write(0, V[0:10])
        stx.write(1, V[10:20])                       # "crash" before chunk 2
        assert stx.missing() == [2]
        stx2 = EmbStore(tag, ids, 8, "int8", 10)     # restart: only chunk 2 is missing
        assert stx2.missing() == [2]
        stx2.write(2, V[20:25])
        G = stx2.gather(np.array([24, 3, 11]), 8, np.float32)
        assert np.min((G * V[[24, 3, 11]]).sum(1)) > 0.999, "int8 round trip too lossy"
        G4 = stx2.gather(np.array([5]), 4, np.float32)
        ref4 = V[5, :4] / np.linalg.norm(V[5, :4])
        assert np.allclose(G4[0], ref4, atol=2e-2), "MRL slice + renormalize wrong"
        stx._mm.clear()
        stx2._mm.clear()   # release memmaps first (Windows cannot delete mapped files)
        del stx, stx2, G, G4
        gc.collect()
        shutil.rmtree(os.path.join(CACHE_DIR, "embeddings", tag), ignore_errors=True)
        # 6. nested sampling
        idsn = np.arange(1, 200_001, dtype=np.int64) + 2 * ID_BASE
        a1, a2 = hash_u01(idsn, 1) < 0.01, hash_u01(idsn, 1) < 0.02
        assert (a1 <= a2).all() and 0.008 < a1.mean() < 0.012
        # 7. strict validator rejects bad files
        exp = ["S1-1", "S1-2"]
        pool_ok = {"S2-10", "S3-11", "S2-12"}

        def _w(path, rows, col):
            with open(path, "w", newline="", encoding="utf-8") as f:
                f.write(f"source1_entity_id\t{col}\n" + "".join(f"{a}\t{b}\n" for a, b in rows))

        mp, cp = os.path.join(tmp, "m.tsv"), os.path.join(tmp, "c.tsv")
        _w(mp, [("S1-1", "S2-10"), ("S1-2", "")], "matched_entity_ids")
        _w(cp, [("S1-1", "S2-10,S3-11"), ("S1-2", "S2-12")], "candidate_entity_ids")
        validate_submission_strict(mp, cp, exp, pool_ok, True, 2)
        bad_cases = {
            "dup s1": ([("S1-1", "S2-10"), ("S1-1", "")], [("S1-1", "S2-10"), ("S1-1", "")]),
            "missing row": ([("S1-1", "S2-10")], [("S1-1", "S2-10")]),
            "invalid id": ([("S1-1", "S2-99"), ("S1-2", "")], [("S1-1", "S2-99"), ("S1-2", "")]),
            "not subset": ([("S1-1", "S3-11"), ("S1-2", "")], [("S1-1", "S2-10"), ("S1-2", "")]),
            "order": ([("S1-2", ""), ("S1-1", "S2-10")], [("S1-2", ""), ("S1-1", "S2-10")]),
            "exclusive": ([("S1-1", "S2-10"), ("S1-2", "S2-10")], [("S1-1", "S2-10"), ("S1-2", "S2-10")]),
            "nan": ([("S1-1", "nan"), ("S1-2", "")], [("S1-1", "nan"), ("S1-2", "")]),
        }
        for name, (mr_, cr_) in bad_cases.items():
            _w(mp, mr_, "matched_entity_ids")
            _w(cp, cr_, "candidate_entity_ids")
            try:
                validate_submission_strict(mp, cp, exp, pool_ok, True, 2)
            except SubmissionValidationError:
                continue
            raise AssertionError(f"validator accepted a bad file: {name}")
        # 8. reverse retrieval: each pool record lists its nearest S1 in rank order, and truncating the list
        #    recomputes the per-S1 order_rank the union uses (same as production)
        S1v = rng.standard_normal((6, 8)).astype(np.float32)
        S1v /= np.linalg.norm(S1v, axis=1, keepdims=True)
        PLv = rng.standard_normal((9, 8)).astype(np.float32)
        PLv /= np.linalg.norm(PLv, axis=1, keepdims=True)
        s_r, i_r = dense_search(_Arr(PLv), np.arange(9), _Arr(S1v), np.arange(6), 3)
        Rv = _topk_frame(np.arange(9), np.arange(6), s_r, i_r, reverse=True)
        sims = PLv @ S1v.T
        assert len(Rv) == 27 and all(r.s1_pos in np.argsort(-sims[r.pool_pos])[:3] for r in Rv.itertuples())
        assert all(abs(r.score - sims[r.pool_pos, r.s1_pos]) < 1e-2 for r in Rv.itertuples())
        Rv["order_rank"] = within_group_rank(Rv["s1_pos"].values, Rv["score"].values).astype(np.int32)
        Tv = truncate_retr({"dense_rev": Rv}, {"dense_rev": 1})["dense_rev"]
        assert (Tv["rank"] == 0).all() and len(Tv) == 9
        assert all(sorted(g) == list(range(len(g))) for g in Tv.groupby("s1_pos")["order_rank"].apply(list))
        # 9. candidate cap: never more than `cap` per S1, and the survivors are the best-ranked ones
        rr_ = pd.DataFrame({"s1_pos": np.repeat(np.arange(20), 30), "pool_pos": rng.integers(0, 500, 600),
                            "score": rng.random(600).astype(np.float32)}).drop_duplicates(["s1_pos", "pool_pos"])
        rr_["rank"] = within_group_rank(rr_["s1_pos"].values, rr_["score"].values).astype(np.int32)
        Uc = build_union({"dense": rr_}, 500, cap=7)
        assert Uc.groupby("s1_pos").size().max() == 7 and (Uc["dense_rank"] < 7).all() and Uc["s1_pos"].nunique() == 20
        # 10. country buckets: partitions never mix countries or sources; the cross-country counter is exact
        saved = dict(FR.__dict__)
        try:
            FR.s1 = pd.DataFrame({"country_norm": np.array(["US", "India", "US", "France"], dtype=object)})
            FR.pool = pd.DataFrame({"country_norm": np.array(["India", "US", "US", "France", "France"], dtype=object),
                                    "pool_source": np.array(["S2", "S2", "S3", "S2", "S3"], dtype=object)})
            FR.s1_cc, FR.pool_cc = country_codes(FR.s1["country_norm"].values, FR.pool["country_norm"].values)
            for c_, s_, qr_, pr_, rv_ in partitions(SimpleNamespace(q=np.arange(4), pool=np.arange(5), rev=np.arange(4))):
                assert set(FR.s1["country_norm"].values[qr_]) <= {c_} and set(FR.pool["country_norm"].values[pr_]) <= {c_}
                assert set(FR.s1["country_norm"].values[rv_]) <= {c_} and (FR.pool["pool_source"].values[pr_] == s_).all()
            assert cross_country_count(np.array([0, 1, 3, 3]), np.array([1, 0, 3, 4])) == 0
            assert cross_country_count(np.array([0, 1]), np.array([0, 1])) == 2
        finally:
            FR.__dict__.clear()
            FR.__dict__.update(saved)
        # 11. embedding dimension consistency: a view never reads more dims than stored, and a cache written with
        #     other settings is refused instead of silently reused
        tag2 = f"selftest_dim_{int(time.time() * 1000)}"
        st3 = EmbStore(tag2, ids, 8, "float16", 10)
        for bad_fn, exc in ((lambda: EmbView([st3], 16), AssertionError), (lambda: EmbStore(tag2, ids, 16, "float16", 10), RuntimeError)):
            try:
                bad_fn()
            except exc:
                continue
            raise AssertionError("embedding dimension mismatch was not detected")
        shutil.rmtree(os.path.join(CACHE_DIR, "embeddings", tag2), ignore_errors=True)
        # 12. scale-phase gating: with ENABLE_SCALE_PHASE=False no scale stage may start
        old_sp = cfg.ENABLE_SCALE_PHASE
        cfg.ENABLE_SCALE_PHASE = False
        try:
            assert not any(scale_phase_allowed(s_, quiet=True) for s_ in SCALE_STAGES)
        finally:
            cfg.ENABLE_SCALE_PHASE = old_sp
        # 13. decision rule: the simplest rule wins unless a complex one is better by more than the tolerance
        rows_t = [(dict(kind="relative", floor=.5, gap_mult=2., exclusive=True), 0.9005), (dict(kind="global", thr=.5), 0.9000),
                  (dict(kind="global", thr=.6, exclusive=True), 0.8990)]
        assert select_rule(rows_t, 0.001)[0] == dict(kind="global", thr=.5)
        assert select_rule(rows_t, 0.0)[0]["kind"] == "relative"
        # 14. GPU BM25 == CPU BM25 (same pairs and scores) on a random partition
        if DEVICE == "cuda":
            words = np.array([f"w{i}" for i in range(400)], dtype=object)
            mk = lambda n: pd.DataFrame({"name_core": [" ".join(rng.choice(words, 3)) for _ in range(n)],
                                         "addr_clean": [" ".join(rng.choice(words, 5)) for _ in range(n)],
                                         "city_norm": [""] * n, "state_norm": [""] * n})
            dd, qq = mk(3000), mk(700)
            old = (cfg.BM25_MIN_DF_CAP, cfg.BM25_MAX_DF_FRAC)
            cfg.BM25_MIN_DF_CAP, cfg.BM25_MAX_DF_FRAC = 50, 0.02
            rc = bm25_search(dd, np.arange(3000), qq, np.arange(700), 20, use_gpu=False)
            rg = bm25_search(dd, np.arange(3000), qq, np.arange(700), 20, use_gpu=True)
            cfg.BM25_MIN_DF_CAP, cfg.BM25_MAX_DF_FRAC = old
            kc = pd.Series(rc[2], index=rc[0] * 10_000 + rc[1])
            kg = pd.Series(rg[2], index=rg[0] * 10_000 + rg[1])
            assert len(kc) == len(kg), (len(kc), len(kg))
            # identical per-query score lists (ties at the k-th place may pick different documents)
            sc_c = pd.DataFrame({"q": rc[0], "s": rc[2]}).sort_values(["q", "s"]).reset_index(drop=True)
            sc_g = pd.DataFrame({"q": rg[0], "s": rg[2]}).sort_values(["q", "s"]).reset_index(drop=True)
            assert (sc_c["q"] == sc_g["q"]).all() and np.allclose(sc_c["s"], sc_g["s"], atol=1e-4), "GPU BM25 != CPU BM25"
            # documents scoring strictly above a query's k-th score must be identical (ties at the k-th may differ)
            kth = sc_c.groupby("q")["s"].min()
            above_c = {int(k) for k, s in kc.items() if s > kth[int(k) // 10_000] + 1e-4}
            above_g = {int(k) for k, s in kg.items() if s > kth[int(k) // 10_000] + 1e-4}
            assert above_c == above_g, "GPU BM25 returns different documents than CPU BM25"
            common = kc.index.intersection(kg.index)
            assert np.allclose(kc[common], kg[common], atol=1e-4)
        # 15. multi-GPU sharded dense search == brute force (pool large enough to be split over the GPUs)
        A2 = rng.standard_normal((300, 32)).astype(np.float32)
        B2 = rng.standard_normal((MULTI_GPU_MIN_POOL + 7000, 32)).astype(np.float32)
        A2 /= np.linalg.norm(A2, axis=1, keepdims=True)
        B2 /= np.linalg.norm(B2, axis=1, keepdims=True)
        old_shard = cfg.SEARCH_SHARD_ROWS
        cfg.SEARCH_SHARD_ROWS = 9000
        s_m, i_m = dense_search(_Arr(A2), np.arange(300), _Arr(B2), np.arange(len(B2)), 10)
        cfg.SEARCH_SHARD_ROWS = old_shard
        ref2 = np.argsort(-(A2 @ B2.T), axis=1)[:, :10]
        assert (np.sort(i_m, 1) == np.sort(ref2, 1)).mean() > 0.97, "multi-GPU dense search disagrees with brute force"
        # 16. combined vector: <u, v> = w cos(name) + (1 - w) cos(addr)
        class _FakeView:
            def __init__(self, M):
                self.M, self.dim, self.n = M, M.shape[1], len(M)

            def gather(self, rows, use_dim=None, out_dtype=np.float16):
                X = self.M[rows]
                return (X / np.linalg.norm(X, axis=1, keepdims=True)).astype(out_dtype)
        Nm, Ad = rng.standard_normal((5, 6)), rng.standard_normal((5, 10))
        cs = CombinedStore.__new__(CombinedStore)
        cs.name, cs.addr, cs.w = _FakeView(Nm), _FakeView(Ad), 0.3
        cs.dim, cs.n, cs._a, cs._b = 16, 5, math.sqrt(0.3), math.sqrt(0.7)
        G = cs.gather(np.arange(5), None, np.float32)
        nn_, aa_ = _FakeView(Nm).gather(np.arange(5), None, np.float32), _FakeView(Ad).gather(np.arange(5), None, np.float32)
        assert np.allclose(G @ G.T, 0.3 * nn_ @ nn_.T + 0.7 * aa_ @ aa_.T, atol=1e-5) and np.allclose(np.linalg.norm(G, axis=1), 1)
        # 17. raw-field views: name_clean / addr_clean = lower-cased raw text, city / state empty
        tv = arrow_to_frame(read_raw_table(SRC_FILE[("train", 1)], ["business_name", "name_clean", "addr_clean",
                                                                     "business_address", "extracted_city"]).slice(0, 50))
        assert (tv["name_clean"] == tv["business_name"].str.lower()).all()
        assert (tv["addr_clean"] == tv["business_address"].str.lower()).all() and (tv["extracted_city"] == "").all()
        shutil.rmtree(tmp, ignore_errors=True)
        log("self-tests passed (17)")

# Run
Each cell below runs one stage when it is listed in `STAGES_TO_RUN` (default: all). Every stage is resumable:
re-running it skips checkpointed work. Order:
selftest → sample → embed_check → weight_sweep → diag_recall → train_model → train_eval (**GATE**) →
test_inference → submit → report.
Follow progress in `pipeline_run/output/progress.log` (e.g. `tail -f`).


### Stage `selftest`


In [ ]:
if want("selftest"):
    run_self_tests()


### Stage `sample` — country invariant (full train ground truth), then the train frame (100% of train)


In [ ]:
if want("sample"):
    check_country_invariant()
    build_train_sample()


### Stage `embed_check` — precomputed name / address vectors: true pairs vs random pairs


In [ ]:
if want("embed_check"):
    embedding_sanity_train()


### Stage `weight_sweep` — w for w·cos(name) + (1-w)·cos(address)


In [ ]:
if want("weight_sweep"):
    stage_weight_sweep()
    print("EMB_NAME_WEIGHT used:", resolve_emb_weight())


### Stage `diag_recall` — recall curves / oracle F0.5 on 1% of train vs the FULL pool (choose K)


In [ ]:
if want("diag_recall"):
    stage_diag_recall()
    show_f05_progress()


### Stage `train_model` — full-train retrieval, training set, XGBoost CV + decision rule


In [ ]:
if want("train_model"):
    stage_train_model()
    show_f05_progress()


### Stage `train_eval` — the other ~80% of train entities (never trained on), per country → GATE


In [ ]:
if want("train_eval"):
    stage_train_eval()
    show_f05_progress()


### Stage `test_inference` — 100% test: retrieval, union, features, scoring, decision rule, audit files


In [ ]:
if want("test_inference"):
    stage_test_inference()


### Stage `submit` — write and strictly validate `matching_results.tsv` / `candidate_pairs.tsv`


In [ ]:
if want("submit"):
    stage_submit()


### Stage `report` — run report + the F0.5 progress table


In [ ]:
if want("report"):
    stage_report()
